In [5]:
# ============================================================
# BLOCK 1 — ENVIRONMENT, GPU, SEEDS, AND EXPERIMENT FOLDERS
# Official UPA Reproduction — 3 Seeds
# ============================================================

import os
import sys
import json
import random
import platform
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import torch

# ------------------------------------------------------------
# 1. Experiment configuration
# ------------------------------------------------------------

SEEDS = [42, 123, 2024]

NUM_CLASSES = 3

CLASS_NAMES = [
    "Early Blight",
    "Late Blight",
    "Healthy"
]

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# ------------------------------------------------------------
# 2. Kaggle paths
# ------------------------------------------------------------

KAGGLE_INPUT = Path("/kaggle/input")
KAGGLE_WORKING = Path("/kaggle/working")

OUTPUT_ROOT = (
    KAGGLE_WORKING
    / "official_upa_reproduction"
)

SOURCE_OUTPUT = OUTPUT_ROOT / "source_models"
TARGET_OUTPUT = OUTPUT_ROOT / "target_adaptation"
PRED_OUTPUT = OUTPUT_ROOT / "predictions"
METRIC_OUTPUT = OUTPUT_ROOT / "metrics"
LOG_OUTPUT = OUTPUT_ROOT / "logs"
CONFIG_OUTPUT = OUTPUT_ROOT / "config"
SPLIT_OUTPUT = OUTPUT_ROOT / "locked_splits"
FIGURE_OUTPUT = OUTPUT_ROOT / "figures"

for folder in [
    OUTPUT_ROOT,
    SOURCE_OUTPUT,
    TARGET_OUTPUT,
    PRED_OUTPUT,
    METRIC_OUTPUT,
    LOG_OUTPUT,
    CONFIG_OUTPUT,
    SPLIT_OUTPUT,
    FIGURE_OUTPUT
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )

# ------------------------------------------------------------
# 3. Reproducibility helper
# ------------------------------------------------------------

def set_seed(seed: int):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    # Reproducibility-oriented settings
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(SEEDS[0])

# ------------------------------------------------------------
# 4. Environment information
# ------------------------------------------------------------

print("=" * 80)
print("OFFICIAL UPA REPRODUCTION")
print("=" * 80)

print(f"Python       : {sys.version.split()[0]}")
print(f"Platform     : {platform.platform()}")
print(f"PyTorch      : {torch.__version__}")
print(f"CUDA usable  : {torch.cuda.is_available()}")
print(f"Device       : {DEVICE}")

if torch.cuda.is_available():

    print(
        f"GPU          : "
        f"{torch.cuda.get_device_name(0)}"
    )

    print(
        f"CUDA version : "
        f"{torch.version.cuda}"
    )

    total_mem = (
        torch.cuda.get_device_properties(0)
        .total_memory
        / (1024 ** 3)
    )

    print(
        f"GPU memory   : "
        f"{total_mem:.2f} GB"
    )

print(f"\nSeeds        : {SEEDS}")
print(f"Classes      : {CLASS_NAMES}")
print(f"Num classes  : {NUM_CLASSES}")

# ------------------------------------------------------------
# 5. Show attached Kaggle datasets
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("ATTACHED KAGGLE INPUTS")
print("=" * 80)

input_dirs = sorted([
    p for p in KAGGLE_INPUT.iterdir()
    if p.is_dir()
])

if len(input_dirs) == 0:
    raise RuntimeError(
        "No Kaggle datasets are attached."
    )

for i, path in enumerate(
    input_dirs,
    start=1
):
    print(
        f"{i:02d}. {path.name}"
    )

# ------------------------------------------------------------
# 6. Save base experiment configuration
# ------------------------------------------------------------

base_config = {
    "experiment":
        "Official UPA reproduction",

    "seeds":
        SEEDS,

    "num_classes":
        NUM_CLASSES,

    "class_names":
        CLASS_NAMES,

    "device":
        str(DEVICE),

    "python_version":
        sys.version,

    "pytorch_version":
        torch.__version__,

    "cuda_available":
        torch.cuda.is_available(),

    "cuda_version":
        torch.version.cuda
        if torch.cuda.is_available()
        else None,

    "output_root":
        str(OUTPUT_ROOT)
}

with open(
    CONFIG_OUTPUT / "base_config.json",
    "w"
) as f:

    json.dump(
        base_config,
        f,
        indent=2
    )

print("\n" + "=" * 80)
print("BLOCK 1 COMPLETE")
print("=" * 80)

print(
    "Output folder:",
    OUTPUT_ROOT
)

print(
    "Config saved :",
    CONFIG_OUTPUT / "base_config.json"
)

OFFICIAL UPA REPRODUCTION
Python       : 3.12.13
Platform     : Linux-6.12.90+-x86_64-with-glibc2.35
PyTorch      : 2.10.0+cu128
CUDA usable  : True
Device       : cuda
GPU          : Tesla T4
CUDA version : 12.8
GPU memory   : 14.56 GB

Seeds        : [42, 123, 2024]
Classes      : ['Early Blight', 'Late Blight', 'Healthy']
Num classes  : 3

ATTACHED KAGGLE INPUTS
01. datasets

BLOCK 1 COMPLETE
Output folder: /kaggle/working/official_upa_reproduction
Config saved : /kaggle/working/official_upa_reproduction/config/base_config.json


In [6]:
# ============================================================
# BLOCK 2 — FAST FINAL-PACKAGE + MANIFEST DISCOVERY
# ============================================================

from pathlib import Path
import subprocess
import pandas as pd
import json
import os

INPUT_ROOT = Path("/kaggle/input/datasets")

PACKAGE_NAME = "TWD_paper_final_package_FIXED_20260905_234741"

print("=" * 80)
print("LOCATING FINAL TWD PACKAGE")
print("=" * 80)

# ------------------------------------------------------------
# 1. Use Linux find instead of recursively scanning all images
# ------------------------------------------------------------

cmd = [
    "find",
    str(INPUT_ROOT),
    "-type", "d",
    "-name", PACKAGE_NAME,
    "-print",
    "-quit"
]

result = subprocess.run(
    cmd,
    capture_output=True,
    text=True,
    check=False
)

package_path_text = result.stdout.strip()

if package_path_text:
    FINAL_PACKAGE = Path(package_path_text)
else:
    # Also search for a partial name in case Kaggle renamed the folder
    cmd = [
        "find",
        str(INPUT_ROOT),
        "-type", "d",
        "-iname", "*TWD*paper*final*package*",
        "-print",
        "-quit"
    ]

    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
        check=False
    )

    package_path_text = result.stdout.strip()

    if not package_path_text:
        raise FileNotFoundError(
            "Could not locate the final TWD package under:\n"
            f"{INPUT_ROOT}"
        )

    FINAL_PACKAGE = Path(package_path_text)

print("Final package:")
print(FINAL_PACKAGE)

# ------------------------------------------------------------
# 2. Search ONLY inside final package for CSV / JSON files
# ------------------------------------------------------------

csv_cmd = [
    "find",
    str(FINAL_PACKAGE),
    "-type", "f",
    "-name", "*.csv"
]

csv_result = subprocess.run(
    csv_cmd,
    capture_output=True,
    text=True,
    check=True
)

csv_files = [
    Path(x)
    for x in csv_result.stdout.splitlines()
    if x.strip()
]

json_cmd = [
    "find",
    str(FINAL_PACKAGE),
    "-type", "f",
    "-name", "*.json"
]

json_result = subprocess.run(
    json_cmd,
    capture_output=True,
    text=True,
    check=True
)

json_files = [
    Path(x)
    for x in json_result.stdout.splitlines()
    if x.strip()
]

print("\nCSV files :", len(csv_files))
print("JSON files:", len(json_files))

# ------------------------------------------------------------
# 3. Print manifest filenames and dimensions
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("CSV MANIFESTS IN FINAL PACKAGE")
print("=" * 80)

manifest_rows = []

for i, path in enumerate(sorted(csv_files), start=1):

    try:
        df = pd.read_csv(path)

        manifest_rows.append({
            "filename": path.name,
            "path": str(path),
            "rows": len(df),
            "columns": ", ".join(map(str, df.columns))
        })

        print(
            f"\n[{i:02d}] {path.name}"
        )

        print(
            f"     rows    : {len(df)}"
        )

        print(
            f"     columns : {list(df.columns)}"
        )

    except Exception as e:

        print(
            f"\n[{i:02d}] {path.name}"
        )
        print(
            f"     ERROR: {e}"
        )

# ------------------------------------------------------------
# 4. Highlight files relevant to our 3-seed experiment
# ------------------------------------------------------------

SEARCH_TERMS = [
    "42",
    "123",
    "2024",
    "source",
    "target_hard",
    "train",
    "val",
    "test",
    "ood"
]

print("\n" + "=" * 80)
print("LIKELY REQUIRED MANIFESTS")
print("=" * 80)

for path in sorted(csv_files):

    name = path.name.lower()

    if any(
        term.lower() in name
        for term in SEARCH_TERMS
    ):
        print(path.name)

# ------------------------------------------------------------
# 5. Explicitly locate known hard-target files
# ------------------------------------------------------------

hard_train = [
    p for p in csv_files
    if p.name.lower() == "target_hard_train_unlabeled.csv"
]

hard_test = [
    p for p in csv_files
    if "target_hard_test" in p.name.lower()
]

print("\n" + "=" * 80)
print("KNOWN HARD-TARGET FILES")
print("=" * 80)

print("\nHard target train:")
for p in hard_train:
    print(" ", p)

print("\nHard target test:")
for p in hard_test:
    print(" ", p)

# ------------------------------------------------------------
# 6. Save discovery table
# ------------------------------------------------------------

manifest_df = pd.DataFrame(manifest_rows)

manifest_df.to_csv(
    CONFIG_OUTPUT / "final_package_manifest_inventory.csv",
    index=False
)

# Preserve discovered package path for later blocks
with open(
    CONFIG_OUTPUT / "resolved_paths.json",
    "w"
) as f:
    json.dump(
        {
            "final_package": str(FINAL_PACKAGE)
        },
        f,
        indent=2
    )

print("\n" + "=" * 80)
print("BLOCK 2 COMPLETE")
print("=" * 80)

print(
    "Inventory saved:",
    CONFIG_OUTPUT / "final_package_manifest_inventory.csv"
)

LOCATING FINAL TWD PACKAGE
Final package:
/kaggle/input/datasets/dmmehedihasanabid/twd-kaggle/TWD_paper_final_package_FIXED_20260905_234741

CSV files : 162
JSON files: 0

CSV MANIFESTS IN FINAL PACKAGE

[01] ood_extra_test.csv
     rows    : 2000
     columns : ['file_path', 'dataset_role', 'raw_class', 'canonical_class']

[02] ood_pld_extra_manifest.csv
     rows    : 2000
     columns : ['file_path', 'dataset_role', 'raw_class', 'canonical_class']

[03] accept_gate_sweep_EfficientNetB0_target_hard_test_labeled.csv
     rows    : 56
     columns : ['model', 'dataset', 'confidence_threshold', 'margin_threshold', 'accept_count', 'accept_coverage', 'accept_accuracy', 'healthy_as_disease_count', 'accepted_correct_healthy_recall']

[04] accept_gate_sweep_EfficientNetB0_target_mild_test_labeled.csv
     rows    : 56
     columns : ['model', 'dataset', 'confidence_threshold', 'margin_threshold', 'accept_count', 'accept_coverage', 'accept_accuracy', 'healthy_as_disease_count', 'accepted_corr

In [7]:
# ============================================================
# BLOCK 3 — RESOLVE LOCKED SPLITS FOR SEEDS 42, 123, 2024
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json
import re

SEEDS = [42, 123, 2024]

# Expected sizes from the completed experiment
EXPECTED = {
    "source_train": 1506,
    "source_val": 323,
    "source_test": 323,
    "target_hard_train": 7200,
    "target_hard_test": 1800,
}

print("=" * 80)
print("RESOLVING LOCKED SPLIT MANIFESTS")
print("=" * 80)

# ------------------------------------------------------------
# 1. Load inventory produced by Block 2
# ------------------------------------------------------------

inventory_path = (
    CONFIG_OUTPUT /
    "final_package_manifest_inventory.csv"
)

assert inventory_path.exists(), (
    f"Missing Block-2 inventory:\n{inventory_path}"
)

inventory = pd.read_csv(inventory_path)

assert len(inventory) > 0, (
    "Block-2 inventory contains no CSV files."
)

# ------------------------------------------------------------
# 2. Helpers
# ------------------------------------------------------------

def norm_text(x):
    return str(x).strip().lower().replace("-", "_").replace(" ", "_")


def detect_seed_values(df):
    """
    Return seed values if a likely seed column exists.
    """
    for c in df.columns:
        cn = norm_text(c)

        if cn in {
            "seed",
            "random_seed",
            "split_seed",
            "random_state"
        }:
            vals = pd.to_numeric(
                df[c],
                errors="coerce"
            ).dropna().unique()

            return sorted([
                int(v) for v in vals
                if int(v) in SEEDS
            ])

    return []


def detect_split_values(df):
    """
    Return normalized values from likely split columns.
    """
    for c in df.columns:
        cn = norm_text(c)

        if cn in {
            "split",
            "subset",
            "partition",
            "set"
        }:
            vals = (
                df[c]
                .astype(str)
                .map(norm_text)
                .unique()
                .tolist()
            )

            return sorted(vals)

    return []


def detect_path_column(df):
    candidates = [
        "file_path",
        "filepath",
        "path",
        "image_path",
        "filename",
        "file"
    ]

    norm_map = {
        norm_text(c): c
        for c in df.columns
    }

    for cand in candidates:
        if cand in norm_map:
            return norm_map[cand]

    return None


def classify_filename(name):
    """
    Filename-based hints only.
    """
    n = norm_text(name)

    tags = []

    if "source" in n:
        tags.append("source")

    if "target" in n and "hard" in n:
        tags.append("target_hard")

    if "ood" in n:
        tags.append("ood")

    if "train" in n:
        tags.append("train")

    if "val" in n or "valid" in n:
        tags.append("val")

    if "test" in n:
        tags.append("test")

    for seed in SEEDS:
        if re.search(
            rf"(^|[^0-9]){seed}([^0-9]|$)",
            n
        ):
            tags.append(f"seed_{seed}")

    return tags


# ------------------------------------------------------------
# 3. Inspect every CSV in final package
# ------------------------------------------------------------

records = []

for _, row in inventory.iterrows():

    path = Path(row["path"])

    try:
        df = pd.read_csv(path)
    except Exception:
        continue

    seeds_found = detect_seed_values(df)
    splits_found = detect_split_values(df)
    path_col = detect_path_column(df)
    tags = classify_filename(path.name)

    records.append({
        "filename": path.name,
        "path": str(path),
        "rows": len(df),
        "path_col": path_col,
        "seed_values": ",".join(
            map(str, seeds_found)
        ),
        "split_values": ",".join(
            splits_found
        ),
        "name_tags": ",".join(tags),
    })

scan_df = pd.DataFrame(records)

scan_path = (
    CONFIG_OUTPUT /
    "split_manifest_scan.csv"
)

scan_df.to_csv(
    scan_path,
    index=False
)

# ------------------------------------------------------------
# 4. Print only potentially relevant files
# ------------------------------------------------------------

interesting = scan_df[
    scan_df["rows"].isin(
        [
            323,
            1506,
            1800,
            2152,
            7200,
            9000,
            1050,
            450
        ]
    )
    |
    scan_df["name_tags"].str.contains(
        "source|target_hard|train|test|val",
        regex=True,
        na=False
    )
    |
    scan_df["seed_values"].astype(str).str.len().gt(0)
].copy()

print("\nPotential split manifests:\n")

if len(interesting) == 0:
    print("No likely split manifests detected.")
else:
    display_cols = [
        "filename",
        "rows",
        "path_col",
        "seed_values",
        "split_values",
        "name_tags"
    ]

    print(
        interesting[display_cols]
        .to_string(index=False)
    )

# ------------------------------------------------------------
# 5. Search for exact known Seed-42 hard-target files
# ------------------------------------------------------------

def find_exact_filename(filename):

    hits = scan_df[
        scan_df["filename"]
        .str.lower()
        .eq(filename.lower())
    ]

    return hits


seed42_train_hits = find_exact_filename(
    "target_hard_train_unlabeled.csv"
)

seed42_test_hits = scan_df[
    scan_df["filename"]
    .str.lower()
    .str.contains(
        "target_hard_test",
        regex=False
    )
]

print("\n" + "=" * 80)
print("KNOWN TARGET-HARD MANIFESTS")
print("=" * 80)

print("\nTrain:")
if len(seed42_train_hits):
    print(
        seed42_train_hits[
            ["filename", "rows", "path"]
        ].to_string(index=False)
    )
else:
    print("Not found by exact filename.")

print("\nTest:")
if len(seed42_test_hits):
    print(
        seed42_test_hits[
            ["filename", "rows", "path"]
        ].to_string(index=False)
    )
else:
    print("No target_hard_test* file found.")

# ------------------------------------------------------------
# 6. Detect combined seed-aware manifests
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("COMBINED SEED-AWARE MANIFESTS")
print("=" * 80)

combined_found = False

for _, row in scan_df.iterrows():

    if not str(row["seed_values"]).strip():
        continue

    p = Path(row["path"])

    try:
        df = pd.read_csv(p)
    except Exception:
        continue

    seeds_here = detect_seed_values(df)

    if len(
        set(seeds_here).intersection(SEEDS)
    ) >= 2:

        combined_found = True

        print(
            f"\n{p.name}"
        )
        print(
            "Rows        :", len(df)
        )
        print(
            "Seeds       :", seeds_here
        )
        print(
            "Split values:",
            detect_split_values(df)
        )
        print(
            "Columns     :", list(df.columns)
        )
        print(
            "Path        :", p
        )

if not combined_found:
    print(
        "No multi-seed combined CSV detected."
    )

# ------------------------------------------------------------
# 7. Search specifically for seed-named CSVs
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("SEED-NAMED CSV FILES")
print("=" * 80)

seed_named = {}

for seed in SEEDS:

    matches = []

    for _, row in scan_df.iterrows():

        filename = norm_text(row["filename"])

        if re.search(
            rf"(^|[^0-9]){seed}([^0-9]|$)",
            filename
        ):
            matches.append(
                row["path"]
            )

    seed_named[str(seed)] = matches

    print(
        f"\nSeed {seed}: {len(matches)} file(s)"
    )

    for p in matches:
        print(" ", p)

# ------------------------------------------------------------
# 8. Save resolution report
# ------------------------------------------------------------

resolution = {
    "expected_sizes": EXPECTED,
    "seeds": SEEDS,
    "known_seed42_target_train": (
        seed42_train_hits["path"].tolist()
    ),
    "known_target_test_candidates": (
        seed42_test_hits["path"].tolist()
    ),
    "seed_named_files": seed_named,
}

with open(
    CONFIG_OUTPUT /
    "split_resolution_report.json",
    "w"
) as f:

    json.dump(
        resolution,
        f,
        indent=2
    )

print("\n" + "=" * 80)
print("BLOCK 3 COMPLETE")
print("=" * 80)

print(
    "Scan saved:",
    scan_path
)

print(
    "Resolution saved:",
    CONFIG_OUTPUT /
    "split_resolution_report.json"
)

RESOLVING LOCKED SPLIT MANIFESTS

Potential split manifests:

                                                      filename  rows  path_col seed_values split_values               name_tags
                                            ood_extra_test.csv  2000 file_path                                         ood,test
 accept_gate_sweep_EfficientNetB0_target_hard_test_labeled.csv    56      None                                 target_hard,test
 accept_gate_sweep_EfficientNetB0_target_mild_test_labeled.csv    56      None                                             test
       accept_gate_sweep_ResNet50_target_hard_test_labeled.csv    56      None                                 target_hard,test
       accept_gate_sweep_ResNet50_target_mild_test_labeled.csv    56      None                                             test
         accepted_pseudo_labels_ResNet50_target_hard_train.csv  2901 file_path                                target_hard,train
             predictions_ResNet50_AcceptOn

In [9]:
# ============================================================
# BLOCK 4 — LOCK EXACT MANIFESTS
# Corrected: parent directory is used to identify seed
# ============================================================

from pathlib import Path
import pandas as pd
import hashlib
import json
import re

SEEDS = [42, 123, 2024]

EXPECTED_ROWS = {
    "source_train": 1506,
    "source_val": 323,
    "source_test": 323,
    "target_hard_train": 7200,
    "target_hard_test": 1800,
}

print("=" * 80)
print("LOCKING EXACT THREE-SEED SPLITS")
print("=" * 80)

# ------------------------------------------------------------
# 1. Load Block-2 inventory
# ------------------------------------------------------------

inventory_path = (
    CONFIG_OUTPUT /
    "final_package_manifest_inventory.csv"
)

inventory = pd.read_csv(inventory_path)

csv_paths = [
    Path(p)
    for p in inventory["path"].tolist()
    if Path(p).exists()
]

assert len(csv_paths) > 0, (
    "No CSV manifests found from Block 2."
)

# ------------------------------------------------------------
# 2. Helpers
# ------------------------------------------------------------

def norm(x):
    return (
        str(x)
        .strip()
        .lower()
        .replace("-", "_")
        .replace(" ", "_")
    )


def sha256_file(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()


def get_path_column(df):

    candidates = [
        "file_path",
        "filepath",
        "image_path",
        "path",
        "filename",
        "file"
    ]

    cmap = {
        norm(c): c
        for c in df.columns
    }

    for candidate in candidates:
        if candidate in cmap:
            return cmap[candidate]

    return None


def get_label_column(df):

    candidates = [
        "label",
        "class",
        "class_name",
        "target",
        "y"
    ]

    cmap = {
        norm(c): c
        for c in df.columns
    }

    for candidate in candidates:
        if candidate in cmap:
            return cmap[candidate]

    return None


def role_from_filename(path):

    n = norm(path.name)

    if n == "source_train.csv":
        return "source_train"

    if n in {
        "source_val.csv",
        "source_valid.csv",
        "source_validation.csv"
    }:
        return "source_val"

    if n == "source_test.csv":
        return "source_test"

    if n in {
        "target_hard_train_unlabeled.csv",
        "target_hard_train.csv"
    }:
        return "target_hard_train"

    if n in {
        "target_hard_test_labeled.csv",
        "target_hard_test.csv"
    }:
        return "target_hard_test"

    return None


def explicit_seed_from_path(path):
    """
    Detect seed only from DIRECTORY/PATH information.

    Examples:
       .../seed_123/source_train.csv -> 123
       .../seed_2024/source_train.csv -> 2024

    The unsuffixed raw_result_files files are treated separately
    as the primary Seed-42 manifests.
    """

    full = norm(str(path))

    patterns = {
        42: [
            r"seed_42(?:/|$)",
            r"seed42(?:/|$)"
        ],
        123: [
            r"seed_123(?:/|$)",
            r"seed123(?:/|$)"
        ],
        2024: [
            r"seed_2024(?:/|$)",
            r"seed2024(?:/|$)"
        ]
    }

    for seed, pats in patterns.items():
        for pat in pats:
            if re.search(pat, full):
                return seed

    return None


def is_primary_raw_manifest(path):
    """
    Seed 42 primary manifests live directly under raw_result_files,
    not inside results_multiseed/seed_*.
    """

    parent = norm(path.parent.name)

    return (
        parent == "raw_result_files"
        and explicit_seed_from_path(path) is None
    )


# ------------------------------------------------------------
# 3. Load only recognized split manifests
# ------------------------------------------------------------

manifest_records = []

for path in csv_paths:

    role = role_from_filename(path)

    if role is None:
        continue

    try:
        df = pd.read_csv(path)
    except Exception:
        continue

    manifest_records.append({
        "path": path,
        "role": role,
        "rows": len(df),
        "explicit_seed": explicit_seed_from_path(path),
        "primary_raw": is_primary_raw_manifest(path),
        "df": df
    })


print("\nRecognized split manifests:")

for r in manifest_records:

    print(
        f"{r['role']:<20} "
        f"rows={r['rows']:<5} "
        f"seed={str(r['explicit_seed']):<5} "
        f"primary={r['primary_raw']} | "
        f"{r['path']}"
    )

# ------------------------------------------------------------
# 4. Resolve exact manifest for each seed/role
# ------------------------------------------------------------

resolved = {}

for seed in SEEDS:

    resolved[seed] = {}

    for role, expected_n in EXPECTED_ROWS.items():

        # --------------------------------------------
        # Seeds 123 and 2024:
        # must come from explicit seed directories
        # --------------------------------------------

        if seed in [123, 2024]:

            candidates = [
                r for r in manifest_records
                if (
                    r["role"] == role
                    and r["rows"] == expected_n
                    and r["explicit_seed"] == seed
                )
            ]

        # --------------------------------------------
        # Seed 42:
        # prefer explicit seed_42 if present.
        # Otherwise use primary raw_result_files copy.
        # --------------------------------------------

        else:

            explicit_candidates = [
                r for r in manifest_records
                if (
                    r["role"] == role
                    and r["rows"] == expected_n
                    and r["explicit_seed"] == 42
                )
            ]

            if explicit_candidates:

                candidates = explicit_candidates

            else:

                candidates = [
                    r for r in manifest_records
                    if (
                        r["role"] == role
                        and r["rows"] == expected_n
                        and r["primary_raw"]
                    )
                ]

        # --------------------------------------------
        # Require exactly one
        # --------------------------------------------

        if len(candidates) == 0:

            raise RuntimeError(
                f"\nMissing exact manifest:\n"
                f"Seed = {seed}\n"
                f"Role = {role}\n"
                f"Expected rows = {expected_n}\n"
            )

        if len(candidates) > 1:

            print(
                f"\nMultiple candidates for "
                f"Seed {seed}, {role}:"
            )

            for x in candidates:
                print(" ", x["path"])

            raise RuntimeError(
                "Multiple exact candidates remain. "
                "Stopped rather than guessing."
            )

        resolved[seed][role] = candidates[0]


# ------------------------------------------------------------
# 5. Create immutable copies under /kaggle/working
# ------------------------------------------------------------

lock_records = []

for seed in SEEDS:

    print("\n" + "-" * 80)
    print(f"SEED {seed}")
    print("-" * 80)

    seed_out = (
        SPLIT_OUTPUT /
        f"seed_{seed}"
    )

    seed_out.mkdir(
        parents=True,
        exist_ok=True
    )

    for role, expected_n in EXPECTED_ROWS.items():

        item = resolved[seed][role]

        df = item["df"].copy()

        if len(df) != expected_n:
            raise RuntimeError(
                f"{seed}/{role}: "
                f"expected {expected_n}, "
                f"found {len(df)}"
            )

        path_col = get_path_column(df)

        if path_col is None:
            raise RuntimeError(
                f"{seed}/{role}: "
                f"no image-path column found.\n"
                f"Columns: {list(df.columns)}"
            )

        duplicate_paths = int(
            df[path_col]
            .astype(str)
            .duplicated()
            .sum()
        )

        locked_path = (
            seed_out /
            f"{role}.csv"
        )

        df.to_csv(
            locked_path,
            index=False
        )

        lock_records.append({
            "seed": seed,
            "role": role,
            "rows": len(df),
            "original_manifest": str(item["path"]),
            "locked_manifest": str(locked_path),
            "path_column": path_col,
            "label_column": get_label_column(df),
            "duplicate_paths": duplicate_paths,
            "sha256": sha256_file(locked_path)
        })

        print(
            f"{role:<20} "
            f"{len(df):>5} rows | "
            f"duplicates={duplicate_paths}"
        )


# ------------------------------------------------------------
# 6. Verify train/test leakage within each seed
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("WITHIN-SEED OVERLAP AUDIT")
print("=" * 80)

overlap_records = []

for seed in SEEDS:

    split_sets = {}

    for role in EXPECTED_ROWS:

        path = (
            SPLIT_OUTPUT /
            f"seed_{seed}" /
            f"{role}.csv"
        )

        df = pd.read_csv(path)

        path_col = get_path_column(df)

        split_sets[role] = set(
            df[path_col]
            .astype(str)
            .tolist()
        )

    checks = [
        (
            "source_train",
            "source_val"
        ),
        (
            "source_train",
            "source_test"
        ),
        (
            "source_val",
            "source_test"
        ),
        (
            "target_hard_train",
            "target_hard_test"
        )
    ]

    for a, b in checks:

        overlap = (
            split_sets[a]
            & split_sets[b]
        )

        n = len(overlap)

        print(
            f"Seed {seed:<4} | "
            f"{a:<18} vs "
            f"{b:<18} -> "
            f"{n}"
        )

        overlap_records.append({
            "seed": seed,
            "split_a": a,
            "split_b": b,
            "overlap": n
        })

        if n != 0:

            raise RuntimeError(
                f"Leakage detected for Seed {seed}: "
                f"{a} vs {b} = {n}"
            )


# ------------------------------------------------------------
# 7. Check that target splits differ across seeds
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("CROSS-SEED TARGET SPLIT CHECK")
print("=" * 80)

target_hashes = {}

for seed in SEEDS:

    p = (
        SPLIT_OUTPUT /
        f"seed_{seed}" /
        "target_hard_test.csv"
    )

    df = pd.read_csv(p)

    path_col = get_path_column(df)

    values = sorted(
        df[path_col]
        .astype(str)
        .tolist()
    )

    digest = hashlib.sha256(
        "\n".join(values).encode()
    ).hexdigest()

    target_hashes[seed] = digest

    print(
        f"Seed {seed}: "
        f"{digest[:16]}..."
    )

if len(set(target_hashes.values())) != len(SEEDS):

    print(
        "\nNOTE: At least two seeds have identical target-test "
        "membership. This is not automatically an error, but "
        "it will be recorded."
    )

else:

    print(
        "\nAll three target-test memberships are distinct."
    )


# ------------------------------------------------------------
# 8. Save reproducibility registry
# ------------------------------------------------------------

lock_df = pd.DataFrame(
    lock_records
)

lock_registry_path = (
    CONFIG_OUTPUT /
    "locked_split_registry.csv"
)

lock_df.to_csv(
    lock_registry_path,
    index=False
)

overlap_df = pd.DataFrame(
    overlap_records
)

overlap_df.to_csv(
    CONFIG_OUTPUT /
    "split_overlap_audit.csv",
    index=False
)

with open(
    CONFIG_OUTPUT /
    "locked_split_registry.json",
    "w"
) as f:

    json.dump(
        lock_records,
        f,
        indent=2
    )


# ------------------------------------------------------------
# 9. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("LOCKED SPLIT SUMMARY")
print("=" * 80)

summary = (
    lock_df[
        ["seed", "role", "rows"]
    ]
    .pivot(
        index="seed",
        columns="role",
        values="rows"
    )
)

print(summary.to_string())

print("\n" + "=" * 80)
print("BLOCK 4 COMPLETE")
print("=" * 80)

print(
    "Locked registry:",
    lock_registry_path
)

print(
    "\nExact manifests locked for all "
    "three seeds without regenerating any split."
)

LOCKING EXACT THREE-SEED SPLITS

Recognized split manifests:
source_test          rows=323   seed=123   primary=False | /kaggle/input/datasets/dmmehedihasanabid/twd-kaggle/TWD_paper_final_package_FIXED_20260905_234741/raw_result_files/results_multiseed/seed_123/source_test.csv
source_train         rows=1506  seed=123   primary=False | /kaggle/input/datasets/dmmehedihasanabid/twd-kaggle/TWD_paper_final_package_FIXED_20260905_234741/raw_result_files/results_multiseed/seed_123/source_train.csv
source_val           rows=323   seed=123   primary=False | /kaggle/input/datasets/dmmehedihasanabid/twd-kaggle/TWD_paper_final_package_FIXED_20260905_234741/raw_result_files/results_multiseed/seed_123/source_val.csv
target_hard_test     rows=1800  seed=123   primary=False | /kaggle/input/datasets/dmmehedihasanabid/twd-kaggle/TWD_paper_final_package_FIXED_20260905_234741/raw_result_files/results_multiseed/seed_123/target_hard_test_labeled.csv
target_hard_train    rows=7200  seed=123   primary=False |

In [10]:
# ============================================================
# BLOCK 5 — OFFICIAL UPA REPOSITORY SETUP + COMMIT PINNING
# ============================================================

from pathlib import Path
import subprocess
import json
import shutil
import os

print("=" * 80)
print("OFFICIAL UPA REPOSITORY SETUP")
print("=" * 80)

UPA_REPO_URL = "https://github.com/chenxi52/UPA.git"

UPA_ROOT = Path("/kaggle/working/UPA_official")

# ------------------------------------------------------------
# 1. Fresh clone
# ------------------------------------------------------------

if UPA_ROOT.exists():
    shutil.rmtree(UPA_ROOT)

clone_cmd = [
    "git",
    "clone",
    "--depth",
    "1",
    UPA_REPO_URL,
    str(UPA_ROOT)
]

result = subprocess.run(
    clone_cmd,
    capture_output=True,
    text=True
)

if result.returncode != 0:

    print(result.stdout)
    print(result.stderr)

    raise RuntimeError(
        "Could not clone the official UPA repository. "
        "Check that Internet access is enabled for this Kaggle notebook."
    )

print("Repository cloned successfully.")

# ------------------------------------------------------------
# 2. Record exact commit SHA
# ------------------------------------------------------------

commit_sha = subprocess.check_output(
    [
        "git",
        "-C",
        str(UPA_ROOT),
        "rev-parse",
        "HEAD"
    ],
    text=True
).strip()

branch_name = subprocess.check_output(
    [
        "git",
        "-C",
        str(UPA_ROOT),
        "rev-parse",
        "--abbrev-ref",
        "HEAD"
    ],
    text=True
).strip()

remote_url = subprocess.check_output(
    [
        "git",
        "-C",
        str(UPA_ROOT),
        "config",
        "--get",
        "remote.origin.url"
    ],
    text=True
).strip()

print(f"Repository : {remote_url}")
print(f"Branch     : {branch_name}")
print(f"Commit SHA : {commit_sha}")

# ------------------------------------------------------------
# 3. Verify expected official files
# ------------------------------------------------------------

required_files = [
    "README.md",
    "train_source.py",
    "upa_v2_amp.py",
    "run_source.sh",
    "run_office-home.sh",
    "model/network.py",
    "trainer/engine.py",
    "utils/tools.py",
]

missing_files = []

print("\n" + "=" * 80)
print("VERIFYING OFFICIAL SOURCE FILES")
print("=" * 80)

for rel in required_files:

    p = UPA_ROOT / rel

    status = "FOUND" if p.exists() else "MISSING"

    print(
        f"{status:<8} {rel}"
    )

    if not p.exists():
        missing_files.append(rel)

if missing_files:

    raise RuntimeError(
        "Official repository clone is incomplete. Missing:\n"
        + "\n".join(missing_files)
    )

# ------------------------------------------------------------
# 4. Record repository status
# ------------------------------------------------------------

status_text = subprocess.check_output(
    [
        "git",
        "-C",
        str(UPA_ROOT),
        "status",
        "--porcelain"
    ],
    text=True
).strip()

if status_text:
    raise RuntimeError(
        "Repository is unexpectedly modified immediately after clone:\n"
        + status_text
    )

# ------------------------------------------------------------
# 5. Record package versions relevant to UPA
# ------------------------------------------------------------

package_versions = {}

for package in [
    "torch",
    "torchvision",
    "numpy",
    "scipy",
    "sklearn",
    "PIL"
]:

    try:

        if package == "torch":
            import torch
            package_versions["torch"] = torch.__version__

        elif package == "torchvision":
            import torchvision
            package_versions["torchvision"] = torchvision.__version__

        elif package == "numpy":
            import numpy
            package_versions["numpy"] = numpy.__version__

        elif package == "scipy":
            import scipy
            package_versions["scipy"] = scipy.__version__

        elif package == "sklearn":
            import sklearn
            package_versions["scikit-learn"] = sklearn.__version__

        elif package == "PIL":
            import PIL
            package_versions["Pillow"] = PIL.__version__

    except Exception as e:

        package_versions[package] = f"ERROR: {e}"

print("\nPackage versions:")

for k, v in package_versions.items():
    print(
        f"{k:<15}: {v}"
    )

# ------------------------------------------------------------
# 6. Save immutable repository metadata
# ------------------------------------------------------------

repo_metadata = {
    "repository": remote_url,
    "branch": branch_name,
    "commit_sha": commit_sha,
    "local_path": str(UPA_ROOT),
    "required_files_verified": required_files,
    "working_tree_clean": True,
    "package_versions": package_versions
}

metadata_path = (
    CONFIG_OUTPUT /
    "official_upa_repository.json"
)

with open(
    metadata_path,
    "w"
) as f:

    json.dump(
        repo_metadata,
        f,
        indent=2
    )

# ------------------------------------------------------------
# 7. Save commit SHA separately for easy manuscript reporting
# ------------------------------------------------------------

with open(
    CONFIG_OUTPUT /
    "official_upa_commit.txt",
    "w"
) as f:

    f.write(commit_sha + "\n")

# ------------------------------------------------------------
# 8. Show repository top-level files
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("OFFICIAL UPA TOP-LEVEL CONTENTS")
print("=" * 80)

for p in sorted(UPA_ROOT.iterdir()):
    print(p.name)

print("\n" + "=" * 80)
print("BLOCK 5 COMPLETE")
print("=" * 80)

print("Official repo :", UPA_ROOT)
print("Pinned commit :", commit_sha)
print("Metadata      :", metadata_path)

OFFICIAL UPA REPOSITORY SETUP
Repository cloned successfully.
Repository : https://github.com/chenxi52/UPA.git
Branch     : master
Commit SHA : 3827ef989fa88e91430160b9c8f3187e1a702b65

VERIFYING OFFICIAL SOURCE FILES
FOUND    README.md
FOUND    train_source.py
FOUND    upa_v2_amp.py
FOUND    run_source.sh
FOUND    run_office-home.sh
FOUND    model/network.py
FOUND    trainer/engine.py
FOUND    utils/tools.py

Package versions:
torch          : 2.10.0+cu128
torchvision    : 0.25.0+cu128
numpy          : 2.0.2
scipy          : 1.16.3
scikit-learn   : 1.6.1
Pillow         : 11.3.0

OFFICIAL UPA TOP-LEVEL CONTENTS
.git
.gitignore
README.md
figs
loaders
model
run_domainnet126.sh
run_office-home.sh
run_office.sh
run_source.sh
run_visda.sh
train_source.py
trainer
upa_v2_amp.py
utils

BLOCK 5 COMPLETE
Official repo : /kaggle/working/UPA_official
Pinned commit : 3827ef989fa88e91430160b9c8f3187e1a702b65
Metadata      : /kaggle/working/official_upa_reproduction/config/official_upa_repository.jso

In [11]:
# ============================================================
# BLOCK 6 — BUILD UPA DATA LISTS FROM LOCKED MANIFESTS
# + RESOLVE IMAGE PATHS + VERIFY LABELS + IMAGE READABILITY
# ============================================================

from pathlib import Path
from PIL import Image
import pandas as pd
import numpy as np
import hashlib
import json
import os
from collections import Counter

print("=" * 80)
print("BUILDING OFFICIAL-UPA DATA LISTS")
print("=" * 80)

SEEDS = [42, 123, 2024]

# ------------------------------------------------------------
# 1. Fixed canonical mapping
# ------------------------------------------------------------

CLASS_TO_ID = {
    "Early Blight": 0,
    "Late Blight": 1,
    "Healthy": 2,
}

ID_TO_CLASS = {
    0: "Early Blight",
    1: "Late Blight",
    2: "Healthy",
}

LIST_ROOT = (
    OUTPUT_ROOT /
    "upa_data_lists"
)

LIST_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# 2. Helpers
# ------------------------------------------------------------

def norm_col(x):
    return (
        str(x)
        .strip()
        .lower()
        .replace("-", "_")
        .replace(" ", "_")
    )


def detect_path_column(df):

    candidates = [
        "file_path",
        "filepath",
        "image_path",
        "path",
        "filename",
        "file",
    ]

    cmap = {
        norm_col(c): c
        for c in df.columns
    }

    for candidate in candidates:
        if candidate in cmap:
            return cmap[candidate]

    raise RuntimeError(
        f"No image path column found. Columns={list(df.columns)}"
    )


def detect_label_column(df):

    candidates = [
        "label",
        "class",
        "class_name",
        "target",
        "y",
    ]

    cmap = {
        norm_col(c): c
        for c in df.columns
    }

    for candidate in candidates:
        if candidate in cmap:
            return cmap[candidate]

    raise RuntimeError(
        f"No label column found. Columns={list(df.columns)}"
    )


def canonical_label(value):

    # Numeric labels
    try:
        if isinstance(value, (int, np.integer)):
            iv = int(value)

            if iv in ID_TO_CLASS:
                return iv

        if isinstance(value, float) and value.is_integer():
            iv = int(value)

            if iv in ID_TO_CLASS:
                return iv
    except Exception:
        pass

    s = str(value).strip().lower()

    # Numeric strings
    if s in {"0", "1", "2"}:
        return int(s)

    # Normalize names
    s2 = (
        s.replace("_", " ")
         .replace("-", " ")
         .strip()
    )

    aliases = {
        "early blight": 0,
        "earlyblight": 0,
        "earlyblt": 0,

        "late blight": 1,
        "lateblight": 1,
        "lateblt": 1,

        "healthy": 2,
        "health": 2,
    }

    compact = s2.replace(" ", "")

    if s2 in aliases:
        return aliases[s2]

    if compact in aliases:
        return aliases[compact]

    raise ValueError(
        f"Unrecognized class label: {value}"
    )


def sha256_text(lines):

    text = "\n".join(lines) + "\n"

    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()


# ------------------------------------------------------------
# 3. Build a filename index ONCE
#
# We do this only because old CSVs may contain paths from the
# previous Kaggle mount. We never use this index to change
# split membership.
# ------------------------------------------------------------

print("\nIndexing attached image files once...")

IMAGE_EXTS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".tif",
    ".tiff",
    ".webp",
}

image_index = {}

for root, dirs, files in os.walk("/kaggle/input/datasets"):

    # Skip our final results package: it should not be used as
    # the underlying raw image dataset.
    if "TWD_paper_final_package_FIXED_20260905_234741" in root:
        continue

    for filename in files:

        if Path(filename).suffix.lower() not in IMAGE_EXTS:
            continue

        p = Path(root) / filename

        image_index.setdefault(
            filename,
            []
        ).append(p)

print(
    f"Indexed {sum(len(v) for v in image_index.values()):,} "
    f"image files."
)


# ------------------------------------------------------------
# 4. Resolve stored manifest path to current Kaggle path
# ------------------------------------------------------------

def resolve_image_path(raw_path):

    raw_path = str(raw_path).strip()

    p = Path(raw_path)

    # A. Stored absolute path still works
    if p.exists():
        return p

    # B. Kaggle path may have changed but tail is preserved
    parts = p.parts

    for keep in range(min(8, len(parts)), 1, -1):

        tail = Path(*parts[-keep:])

        candidate = (
            Path("/kaggle/input/datasets") /
            tail
        )

        if candidate.exists():
            return candidate

    # C. Filename lookup
    filename = p.name

    matches = image_index.get(
        filename,
        []
    )

    if len(matches) == 1:
        return matches[0]

    # D. If filename duplicates exist, use longest
    # suffix agreement with original stored path.
    if len(matches) > 1:

        original_parts = [
            x.lower()
            for x in p.parts
        ]

        scored = []

        for m in matches:

            m_parts = [
                x.lower()
                for x in m.parts
            ]

            score = 0

            for a, b in zip(
                reversed(original_parts),
                reversed(m_parts)
            ):
                if a == b:
                    score += 1
                else:
                    break

            scored.append(
                (score, m)
            )

        scored.sort(
            key=lambda x: x[0],
            reverse=True
        )

        # Require unique best match.
        if (
            len(scored) == 1
            or scored[0][0] > scored[1][0]
        ):
            return scored[0][1]

    return None


# ------------------------------------------------------------
# 5. Convert one locked CSV to UPA text list
# ------------------------------------------------------------

def build_upa_list(
    seed,
    role,
    check_images=True
):

    csv_path = (
        SPLIT_OUTPUT /
        f"seed_{seed}" /
        f"{role}.csv"
    )

    if not csv_path.exists():
        raise RuntimeError(
            f"Missing locked manifest: {csv_path}"
        )

    df = pd.read_csv(
        csv_path
    )

    path_col = detect_path_column(
        df
    )

    label_col = detect_label_column(
        df
    )

    lines = []
    resolved_rows = []
    missing = []
    invalid = []

    for idx, row in df.iterrows():

        raw_path = row[path_col]

        try:
            label_id = canonical_label(
                row[label_col]
            )
        except Exception as e:

            raise RuntimeError(
                f"Seed {seed}, {role}, row {idx}: "
                f"bad label {row[label_col]!r}"
            ) from e

        resolved = resolve_image_path(
            raw_path
        )

        if resolved is None:

            missing.append({
                "row": int(idx),
                "raw_path": str(raw_path),
                "label": int(label_id),
            })

            continue

        resolved = resolved.resolve()

        # Verify PIL can actually decode it.
        if check_images:

            try:

                with Image.open(resolved) as img:
                    img.verify()

            except Exception as e:

                invalid.append({
                    "row": int(idx),
                    "raw_path": str(raw_path),
                    "resolved_path": str(resolved),
                    "label": int(label_id),
                    "error": repr(e),
                })

                continue

        line = (
            f"{resolved} {label_id}"
        )

        lines.append(
            line
        )

        resolved_rows.append({
            "row": int(idx),
            "original_path": str(raw_path),
            "resolved_path": str(resolved),
            "label": int(label_id),
            "class_name": ID_TO_CLASS[label_id],
        })

    return {
        "csv_path": csv_path,
        "path_column": path_col,
        "label_column": label_col,
        "lines": lines,
        "resolved_rows": resolved_rows,
        "missing": missing,
        "invalid": invalid,
        "original_rows": len(df),
    }


# ------------------------------------------------------------
# 6. Process all 15 locked split files
# ------------------------------------------------------------

ROLES = [
    "source_train",
    "source_val",
    "source_test",
    "target_hard_train",
    "target_hard_test",
]

EXPECTED_ROWS = {
    "source_train": 1506,
    "source_val": 323,
    "source_test": 323,
    "target_hard_train": 7200,
    "target_hard_test": 1800,
}

all_results = {}

for seed in SEEDS:

    print(
        "\n" + "-" * 80
    )
    print(
        f"SEED {seed}"
    )
    print(
        "-" * 80
    )

    seed_dir = (
        LIST_ROOT /
        f"seed_{seed}"
    )

    seed_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    all_results[str(seed)] = {}

    for role in ROLES:

        result = build_upa_list(
            seed,
            role,
            check_images=True
        )

        all_results[str(seed)][role] = result

        expected = EXPECTED_ROWS[role]

        good = len(
            result["lines"]
        )

        missing_n = len(
            result["missing"]
        )

        invalid_n = len(
            result["invalid"]
        )

        print(
            f"{role:<20} "
            f"expected={expected:<5} "
            f"resolved={good:<5} "
            f"missing={missing_n:<3} "
            f"invalid={invalid_n:<3}"
        )


# ------------------------------------------------------------
# 7. Report any bad files BEFORE writing final UPA lists
# ------------------------------------------------------------

bad_records = []

for seed in SEEDS:

    for role in ROLES:

        r = all_results[str(seed)][role]

        for x in r["missing"]:

            bad_records.append({
                "seed": seed,
                "role": role,
                "problem": "missing",
                **x,
            })

        for x in r["invalid"]:

            bad_records.append({
                "seed": seed,
                "role": role,
                "problem": "invalid_image",
                **x,
            })


bad_df = pd.DataFrame(
    bad_records
)

bad_path = (
    CONFIG_OUTPUT /
    "image_integrity_problems.csv"
)

bad_df.to_csv(
    bad_path,
    index=False
)


# ------------------------------------------------------------
# 8. Critical handling of invalid files
#
# We DO NOT silently remove anything here.
# ------------------------------------------------------------

if len(bad_df) > 0:

    print(
        "\n" + "=" * 80
    )
    print(
        "IMAGE INTEGRITY ISSUE DETECTED"
    )
    print(
        "=" * 80
    )

    print(
        bad_df[
            [
                c for c in [
                    "seed",
                    "role",
                    "problem",
                    "row",
                    "raw_path",
                    "resolved_path",
                    "error",
                ]
                if c in bad_df.columns
            ]
        ].to_string(
            index=False
        )
    )

    print(
        f"\nSaved full report to:\n{bad_path}"
    )

    raise RuntimeError(
        "Stopped before training because at least one locked "
        "sample is missing or unreadable. "
        "We must match the original handling of these files "
        "rather than silently changing the split."
    )


# ------------------------------------------------------------
# 9. Write official UPA-format lists
#
# Format expected by ImageList:
# /absolute/image/path.jpg LABEL
# ------------------------------------------------------------

registry = []

for seed in SEEDS:

    seed_dir = (
        LIST_ROOT /
        f"seed_{seed}"
    )

    for role in ROLES:

        r = all_results[str(seed)][role]

        lines = r["lines"]

        if len(lines) != EXPECTED_ROWS[role]:

            raise RuntimeError(
                f"Seed {seed} {role}: expected "
                f"{EXPECTED_ROWS[role]}, got {len(lines)}."
            )

        txt_path = (
            seed_dir /
            f"{role}.txt"
        )

        with open(
            txt_path,
            "w"
        ) as f:

            f.write(
                "\n".join(lines)
                + "\n"
            )

        resolved_csv = (
            seed_dir /
            f"{role}_resolved.csv"
        )

        pd.DataFrame(
            r["resolved_rows"]
        ).to_csv(
            resolved_csv,
            index=False
        )

        labels = [
            int(x.rsplit(" ", 1)[1])
            for x in lines
        ]

        counts = Counter(
            labels
        )

        registry.append({
            "seed": seed,
            "role": role,
            "rows": len(lines),
            "class_0_early": counts.get(0, 0),
            "class_1_late": counts.get(1, 0),
            "class_2_healthy": counts.get(2, 0),
            "txt_path": str(txt_path),
            "resolved_csv": str(resolved_csv),
            "sha256": sha256_text(lines),
        })


# ------------------------------------------------------------
# 10. Verify no cross-split leakage after path resolution
# ------------------------------------------------------------

print(
    "\n" + "=" * 80
)
print(
    "RESOLVED-PATH LEAKAGE CHECK"
)
print(
    "=" * 80
)

for seed in SEEDS:

    paths_by_role = {}

    for role in ROLES:

        rows = all_results[
            str(seed)
        ][role]["resolved_rows"]

        paths_by_role[role] = {
            x["resolved_path"]
            for x in rows
        }

    checks = [
        (
            "source_train",
            "source_val"
        ),
        (
            "source_train",
            "source_test"
        ),
        (
            "source_val",
            "source_test"
        ),
        (
            "target_hard_train",
            "target_hard_test"
        ),
    ]

    for a, b in checks:

        overlap = (
            paths_by_role[a]
            & paths_by_role[b]
        )

        print(
            f"Seed {seed:<4} | "
            f"{a:<18} vs "
            f"{b:<18} -> "
            f"{len(overlap)}"
        )

        if overlap:

            raise RuntimeError(
                f"Resolved-path leakage detected: "
                f"Seed {seed}, {a} vs {b}"
            )


# ------------------------------------------------------------
# 11. Save registry
# ------------------------------------------------------------

registry_df = pd.DataFrame(
    registry
)

registry_path = (
    CONFIG_OUTPUT /
    "upa_data_list_registry.csv"
)

registry_df.to_csv(
    registry_path,
    index=False
)

with open(
    CONFIG_OUTPUT /
    "class_mapping.json",
    "w"
) as f:

    json.dump(
        {
            "0": "Early Blight",
            "1": "Late Blight",
            "2": "Healthy",
        },
        f,
        indent=2
    )


# ------------------------------------------------------------
# 12. Final summary
# ------------------------------------------------------------

print(
    "\n" + "=" * 80
)
print(
    "UPA DATA LIST SUMMARY"
)
print(
    "=" * 80
)

print(
    registry_df[
        [
            "seed",
            "role",
            "rows",
            "class_0_early",
            "class_1_late",
            "class_2_healthy",
        ]
    ].to_string(
        index=False
    )
)

print(
    "\n" + "=" * 80
)
print(
    "BLOCK 6 COMPLETE"
)
print(
    "=" * 80
)

print(
    "UPA lists:",
    LIST_ROOT
)

print(
    "Registry :",
    registry_path
)

BUILDING OFFICIAL-UPA DATA LISTS

Indexing attached image files once...
Indexed 64,361 image files.

--------------------------------------------------------------------------------
SEED 42
--------------------------------------------------------------------------------
source_train         expected=1506  resolved=1506  missing=0   invalid=0  
source_val           expected=323   resolved=323   missing=0   invalid=0  
source_test          expected=323   resolved=323   missing=0   invalid=0  
target_hard_train    expected=7200  resolved=7200  missing=0   invalid=0  
target_hard_test     expected=1800  resolved=1800  missing=0   invalid=0  

--------------------------------------------------------------------------------
SEED 123
--------------------------------------------------------------------------------
source_train         expected=1506  resolved=1506  missing=0   invalid=0  
source_val           expected=323   resolved=323   missing=0   invalid=0  
source_test          expected=32

RuntimeError: Stopped before training because at least one locked sample is missing or unreadable. We must match the original handling of these files rather than silently changing the split.

In [12]:
# ============================================================
# BLOCK 6B — DOCUMENTED INVALID-IMAGE HANDLING
# Seed 2024 target_hard_train:
# exclude only earlyblt838.jpg
# ============================================================

from pathlib import Path
from collections import Counter
import pandas as pd
import hashlib
import json

print("=" * 80)
print("APPLYING DOCUMENTED INVALID-IMAGE EXCLUSION")
print("=" * 80)

# ------------------------------------------------------------
# 1. Exact known exclusion from the original experiment
# ------------------------------------------------------------

KNOWN_EXCLUSION = {
    "seed": 2024,
    "role": "target_hard_train",
    "filename": "earlyblt838.jpg",
    "reason": "Unreadable image; PIL UnidentifiedImageError"
}

# Verify Block 6 found exactly what we expect
assert len(bad_df) == 1, (
    f"Expected exactly 1 integrity problem, found {len(bad_df)}."
)

bad_row = bad_df.iloc[0]

assert int(bad_row["seed"]) == 2024
assert str(bad_row["role"]) == "target_hard_train"
assert Path(str(bad_row["raw_path"])).name == "earlyblt838.jpg"

print("Verified expected problematic sample:")
print(" Seed     :", int(bad_row["seed"]))
print(" Role     :", bad_row["role"])
print(" File     :", Path(str(bad_row["raw_path"])).name)
print(" Problem  :", bad_row["problem"])

# ------------------------------------------------------------
# 2. Record exclusion explicitly
# ------------------------------------------------------------

exclusion_path = (
    CONFIG_OUTPUT /
    "documented_image_exclusions.csv"
)

pd.DataFrame([
    {
        "seed": 2024,
        "role": "target_hard_train",
        "original_row": int(bad_row["row"]),
        "raw_path": str(bad_row["raw_path"]),
        "resolved_path": str(bad_row["resolved_path"]),
        "filename": "earlyblt838.jpg",
        "reason": "Unreadable image (PIL UnidentifiedImageError)",
        "action": "Excluded from target adaptation training only",
        "replacement": "None",
        "test_set_changed": False
    }
]).to_csv(
    exclusion_path,
    index=False
)

# ------------------------------------------------------------
# 3. Expected usable sizes after documented exclusion
# ------------------------------------------------------------

EXPECTED_USABLE_ROWS = {
    (42, "source_train"): 1506,
    (42, "source_val"): 323,
    (42, "source_test"): 323,
    (42, "target_hard_train"): 7200,
    (42, "target_hard_test"): 1800,

    (123, "source_train"): 1506,
    (123, "source_val"): 323,
    (123, "source_test"): 323,
    (123, "target_hard_train"): 7200,
    (123, "target_hard_test"): 1800,

    (2024, "source_train"): 1506,
    (2024, "source_val"): 323,
    (2024, "source_test"): 323,
    (2024, "target_hard_train"): 7199,
    (2024, "target_hard_test"): 1800,
}

# ------------------------------------------------------------
# 4. Verify there are NO other problems
# ------------------------------------------------------------

for seed in SEEDS:

    for role in ROLES:

        result = all_results[str(seed)][role]

        missing_n = len(result["missing"])
        invalid_n = len(result["invalid"])

        if seed == 2024 and role == "target_hard_train":

            assert missing_n == 0
            assert invalid_n == 1

            invalid_name = Path(
                result["invalid"][0]["resolved_path"]
            ).name

            assert invalid_name == "earlyblt838.jpg"

        else:

            if missing_n != 0 or invalid_n != 0:

                raise RuntimeError(
                    f"Unexpected image problem in "
                    f"Seed {seed}, {role}: "
                    f"missing={missing_n}, invalid={invalid_n}"
                )

print("\nNo additional image problems detected.")

# ------------------------------------------------------------
# 5. Write final UPA lists
# ------------------------------------------------------------

registry = []

for seed in SEEDS:

    seed_dir = (
        LIST_ROOT /
        f"seed_{seed}"
    )

    seed_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    for role in ROLES:

        result = all_results[str(seed)][role]

        # Block 6 already omitted unreadable images
        # from result["lines"] / result["resolved_rows"].
        lines = result["lines"]
        rows = result["resolved_rows"]

        expected = EXPECTED_USABLE_ROWS[
            (seed, role)
        ]

        if len(lines) != expected:

            raise RuntimeError(
                f"Seed {seed} {role}: "
                f"expected {expected} usable samples, "
                f"found {len(lines)}."
            )

        txt_path = (
            seed_dir /
            f"{role}.txt"
        )

        with open(
            txt_path,
            "w"
        ) as f:
            f.write(
                "\n".join(lines) + "\n"
            )

        resolved_csv = (
            seed_dir /
            f"{role}_resolved.csv"
        )

        pd.DataFrame(
            rows
        ).to_csv(
            resolved_csv,
            index=False
        )

        labels = [
            int(line.rsplit(" ", 1)[1])
            for line in lines
        ]

        counts = Counter(labels)

        digest = hashlib.sha256(
            ("\n".join(lines) + "\n").encode("utf-8")
        ).hexdigest()

        registry.append({
            "seed": seed,
            "role": role,
            "original_locked_rows":
                result["original_rows"],
            "usable_rows": len(lines),
            "excluded_rows":
                result["original_rows"] - len(lines),
            "class_0_early":
                counts.get(0, 0),
            "class_1_late":
                counts.get(1, 0),
            "class_2_healthy":
                counts.get(2, 0),
            "txt_path":
                str(txt_path),
            "resolved_csv":
                str(resolved_csv),
            "sha256":
                digest
        })

# ------------------------------------------------------------
# 6. Verify resolved-path leakage again
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("FINAL RESOLVED-PATH LEAKAGE CHECK")
print("=" * 80)

for seed in SEEDS:

    path_sets = {}

    for role in ROLES:

        df = pd.read_csv(
            LIST_ROOT /
            f"seed_{seed}" /
            f"{role}_resolved.csv"
        )

        path_sets[role] = set(
            df["resolved_path"]
            .astype(str)
            .tolist()
        )

    checks = [
        ("source_train", "source_val"),
        ("source_train", "source_test"),
        ("source_val", "source_test"),
        ("target_hard_train", "target_hard_test")
    ]

    for a, b in checks:

        overlap = (
            path_sets[a] &
            path_sets[b]
        )

        print(
            f"Seed {seed:<4} | "
            f"{a:<18} vs "
            f"{b:<18} -> "
            f"{len(overlap)}"
        )

        if overlap:
            raise RuntimeError(
                f"Leakage detected after exclusion: "
                f"Seed {seed}, {a} vs {b}"
            )

# ------------------------------------------------------------
# 7. Save final UPA data registry
# ------------------------------------------------------------

registry_df = pd.DataFrame(
    registry
)

registry_path = (
    CONFIG_OUTPUT /
    "upa_data_list_registry.csv"
)

registry_df.to_csv(
    registry_path,
    index=False
)

with open(
    CONFIG_OUTPUT /
    "invalid_image_handling.json",
    "w"
) as f:

    json.dump(
        {
            "seed": 2024,
            "role": "target_hard_train",
            "filename": "earlyblt838.jpg",
            "original_size": 7200,
            "usable_size": 7199,
            "action":
                "excluded unreadable image; no replacement",
            "target_test_size": 1800,
            "target_test_modified": False
        },
        f,
        indent=2
    )

# ------------------------------------------------------------
# 8. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("FINAL UPA DATA LIST SUMMARY")
print("=" * 80)

print(
    registry_df[
        [
            "seed",
            "role",
            "original_locked_rows",
            "usable_rows",
            "excluded_rows",
            "class_0_early",
            "class_1_late",
            "class_2_healthy"
        ]
    ].to_string(index=False)
)

print("\n" + "=" * 80)
print("BLOCK 6B COMPLETE")
print("=" * 80)

print(
    "Documented exclusion:",
    exclusion_path
)

print(
    "Final UPA registry   :",
    registry_path
)

APPLYING DOCUMENTED INVALID-IMAGE EXCLUSION
Verified expected problematic sample:
 Seed     : 2024
 Role     : target_hard_train
 File     : earlyblt838.jpg
 Problem  : invalid_image

No additional image problems detected.

FINAL RESOLVED-PATH LEAKAGE CHECK
Seed 42   | source_train       vs source_val         -> 0
Seed 42   | source_train       vs source_test        -> 0
Seed 42   | source_val         vs source_test        -> 0
Seed 42   | target_hard_train  vs target_hard_test   -> 0
Seed 123  | source_train       vs source_val         -> 0
Seed 123  | source_train       vs source_test        -> 0
Seed 123  | source_val         vs source_test        -> 0
Seed 123  | target_hard_train  vs target_hard_test   -> 0
Seed 2024 | source_train       vs source_val         -> 0
Seed 2024 | source_train       vs source_test        -> 0
Seed 2024 | source_val         vs source_test        -> 0
Seed 2024 | target_hard_train  vs target_hard_test   -> 0

FINAL UPA DATA LIST SUMMARY
 seed            

In [13]:
# ============================================================
# BLOCK 7 — VERIFY OFFICIAL UPA SOURCE ARCHITECTURE
# + LOCK CONTROLLED SOURCE-TRAINING PROTOCOL
# ============================================================

from pathlib import Path
import sys
import json
import inspect
import torch
import torch.nn as nn
import torchvision.transforms as transforms

print("=" * 80)
print("VERIFYING OFFICIAL UPA SOURCE ARCHITECTURE")
print("=" * 80)

# ------------------------------------------------------------
# 1. Import the pinned official repository
# ------------------------------------------------------------

if str(UPA_ROOT) not in sys.path:
    sys.path.insert(0, str(UPA_ROOT))

from model import network
from utils.loss import CrossEntropyLabelSmooth

# ------------------------------------------------------------
# 2. Locked UPA source configuration
#
# Architecture and defaults follow official UPA code:
# ResNet50 -> 256-D BN bottleneck -> WN classifier
#
# For our controlled benchmark:
# - class_num = 3
# - use our locked source_train/source_val
# - source test remains untouched
# ------------------------------------------------------------

UPA_SOURCE_CONFIG = {
    "net": "resnet50",
    "class_num": 3,

    "bottleneck": 256,
    "classifier": "bn",
    "layer": "wn",
    "bias": True,

    "criterion": "sce",
    "label_smoothing": 0.1,

    "batch_size": 64,
    "num_workers": 4,

    # Official train_source.py defaults
    "base_lr": 1e-2,
    "backbone_lr_multiplier": 0.1,
    "bottleneck_lr_multiplier": 1.0,
    "classifier_lr_multiplier": 1.0,

    "scheduler_gamma": 10.0,
    "scheduler_power": 0.75,
    "weight_decay": 1e-3,
    "momentum": 0.9,
    "nesterov": True,

    # Use official default source-training duration.
    # We are not using run_source.sh's DomainNet-specific 60 epochs.
    "max_epoch": 20,

    "train_resize": 256,
    "train_crop": 224,
    "test_resize": 256,
    "test_crop": 224,

    "normalization": "imagenet",

    "source_split_policy":
        "locked source_train + locked source_val; no internal 90/10 resplit",

    "checkpoint_selection":
        "highest locked source-validation accuracy",

    "target_labels_used_during_source_training": False
}

# ------------------------------------------------------------
# 3. Official image transforms
#
# Matches train_source.py:
#
# train:
#   Resize((256,256))
#   RandomCrop(224)
#   RandomHorizontalFlip()
#   ToTensor()
#   ImageNet Normalize
#
# test:
#   Resize((256,256))
#   CenterCrop(224)
#   ToTensor()
#   ImageNet Normalize
# ------------------------------------------------------------

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

upa_source_train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])

upa_source_eval_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])

# ------------------------------------------------------------
# 4. Instantiate official architecture
# ------------------------------------------------------------

print("\nInstantiating official UPA architecture...")

netF = network.ResBase(
    res_name="resnet50"
).to(DEVICE)

netB = network.feat_bootleneck(
    type="bn",
    feature_dim=netF.in_features,
    bottleneck_dim=256,
    normalized=False
).to(DEVICE)

netC = network.feat_classifier(
    type="wn",
    class_num=3,
    bottleneck_dim=256,
    bias=True
).to(DEVICE)

# ------------------------------------------------------------
# 5. Structural assertions
# ------------------------------------------------------------

assert netF.in_features == 2048, (
    f"Unexpected ResNet50 feature dimension: {netF.in_features}"
)

assert hasattr(netB, "bottleneck")
assert isinstance(netB.bottleneck, nn.Linear)

assert (
    netB.bottleneck.in_features == 2048
    and netB.bottleneck.out_features == 256
), (
    "Official bottleneck shape is not 2048 -> 256."
)

assert hasattr(netB, "bn")
assert isinstance(netB.bn, nn.BatchNorm1d)

assert netB.bn.num_features == 256

assert hasattr(netC, "fc")

# weight_norm in this repository exposes weight_g + weight_v
classifier_param_names = {
    name
    for name, _ in netC.named_parameters()
}

weight_norm_detected = (
    any("weight_g" in x for x in classifier_param_names)
    and
    any("weight_v" in x for x in classifier_param_names)
)

assert weight_norm_detected, (
    "Weight-normalized classifier was not detected."
)

# ------------------------------------------------------------
# 6. Forward-pass verification
# ------------------------------------------------------------

netF.eval()
netB.eval()
netC.eval()

with torch.no_grad():

    dummy = torch.randn(
        2,
        3,
        224,
        224,
        device=DEVICE
    )

    featF = netF(dummy)
    featB = netB(featF)
    logits = netC(featB)

print("\nForward-pass shapes:")
print(" Input      :", tuple(dummy.shape))
print(" ResNet50   :", tuple(featF.shape))
print(" Bottleneck :", tuple(featB.shape))
print(" Logits     :", tuple(logits.shape))

assert tuple(featF.shape) == (2, 2048)
assert tuple(featB.shape) == (2, 256)
assert tuple(logits.shape) == (2, 3)

# ------------------------------------------------------------
# 7. Parameter counts
# ------------------------------------------------------------

def count_params(model):
    return sum(
        p.numel()
        for p in model.parameters()
    )


def count_trainable(model):
    return sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )


param_summary = {
    "netF_total": count_params(netF),
    "netF_trainable": count_trainable(netF),

    "netB_total": count_params(netB),
    "netB_trainable": count_trainable(netB),

    "netC_total": count_params(netC),
    "netC_trainable": count_trainable(netC)
}

print("\nParameter counts:")

for k, v in param_summary.items():
    print(
        f"{k:<20}: {v:,}"
    )

# ------------------------------------------------------------
# 8. Verify optimizer grouping against official source code
#
# Official train_source.py:
# netF LR = base_lr * 0.1
# netB LR = base_lr
# netC LR = base_lr
# ------------------------------------------------------------

base_lr = UPA_SOURCE_CONFIG["base_lr"]

optimizer = torch.optim.SGD(
    [
        {
            "params": netF.parameters(),
            "lr": base_lr * 0.1
        },
        {
            "params": netB.parameters(),
            "lr": base_lr
        },
        {
            "params": netC.parameters(),
            "lr": base_lr
        }
    ]
)

for group in optimizer.param_groups:
    group["lr0"] = group["lr"]

actual_lrs = [
    group["lr"]
    for group in optimizer.param_groups
]

expected_lrs = [
    1e-3,
    1e-2,
    1e-2
]

assert np.allclose(
    actual_lrs,
    expected_lrs
)

print("\nInitial source-training learning rates:")
print(" netF:", actual_lrs[0])
print(" netB:", actual_lrs[1])
print(" netC:", actual_lrs[2])

# ------------------------------------------------------------
# 9. Verify source criterion
# ------------------------------------------------------------

criterion = CrossEntropyLabelSmooth(
    num_classes=3,
    epsilon=0.1
)

print("\nCriterion:")
print(
    " CrossEntropyLabelSmooth("
    "num_classes=3, epsilon=0.1)"
)

# ------------------------------------------------------------
# 10. Verify all final UPA list files exist
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("VERIFYING THREE-SEED UPA LIST FILES")
print("=" * 80)

required_roles = [
    "source_train",
    "source_val",
    "source_test",
    "target_hard_train",
    "target_hard_test"
]

for seed in SEEDS:

    print(f"\nSeed {seed}")

    for role in required_roles:

        p = (
            LIST_ROOT /
            f"seed_{seed}" /
            f"{role}.txt"
        )

        if not p.exists():
            raise RuntimeError(
                f"Missing UPA list: {p}"
            )

        with open(p, "r") as f:
            n = sum(
                1 for line in f
                if line.strip()
            )

        print(
            f" {role:<20}: {n}"
        )

# ------------------------------------------------------------
# 11. Save protocol BEFORE any training
# ------------------------------------------------------------

protocol = {
    "official_repo_commit":
        commit_sha,

    "architecture": {
        "backbone":
            "official UPA network.ResBase(resnet50)",

        "backbone_output_dim":
            2048,

        "bottleneck":
            "official UPA feat_bootleneck(type='bn')",

        "bottleneck_dim":
            256,

        "classifier":
            "official UPA feat_classifier(type='wn')",

        "num_classes":
            3
    },

    "source_training":
        UPA_SOURCE_CONFIG,

    "transforms": {
        "train":
            [
                "Resize((256,256))",
                "RandomCrop(224)",
                "RandomHorizontalFlip()",
                "ToTensor()",
                "ImageNet normalization"
            ],

        "evaluation":
            [
                "Resize((256,256))",
                "CenterCrop(224)",
                "ToTensor()",
                "ImageNet normalization"
            ]
    },

    "parameter_counts":
        param_summary,

    "initial_learning_rates": {
        "netF":
            actual_lrs[0],

        "netB":
            actual_lrs[1],

        "netC":
            actual_lrs[2]
    },

    "controlled_deviation_from_unmodified_train_source_py":
        (
            "The official script internally creates a random "
            "90/10 split from its source list. For this benchmark "
            "that behavior is replaced by the previously locked "
            "source_train and source_val manifests so all methods "
            "use the same source partition. Architecture, source "
            "loss, optimizer grouping, scheduler, transforms, "
            "bottleneck and classifier design are retained."
        )
}

protocol_path = (
    CONFIG_OUTPUT /
    "upa_source_protocol.json"
)

with open(
    protocol_path,
    "w"
) as f:

    json.dump(
        protocol,
        f,
        indent=2
    )

# ------------------------------------------------------------
# 12. Cleanup GPU before next block
# ------------------------------------------------------------

del dummy
del featF
del featB
del logits
del netF
del netB
del netC
del optimizer
del criterion

if torch.cuda.is_available():
    torch.cuda.empty_cache()

# ------------------------------------------------------------
# 13. Final
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("BLOCK 7 COMPLETE")
print("=" * 80)

print(
    "Architecture : ResNet50 -> 256-D BN bottleneck "
    "-> WN 3-class classifier"
)

print(
    "Source epochs:",
    UPA_SOURCE_CONFIG["max_epoch"]
)

print(
    "Batch size   :",
    UPA_SOURCE_CONFIG["batch_size"]
)

print(
    "Base LR      :",
    UPA_SOURCE_CONFIG["base_lr"]
)

print(
    "Protocol     :",
    protocol_path
)

VERIFYING OFFICIAL UPA SOURCE ARCHITECTURE

Instantiating official UPA architecture...


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 211MB/s]
/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)



Forward-pass shapes:
 Input      : (2, 3, 224, 224)
 ResNet50   : (2, 2048)
 Bottleneck : (2, 256)
 Logits     : (2, 3)

Parameter counts:
netF_total          : 23,508,032
netF_trainable      : 23,508,032
netB_total          : 525,056
netB_trainable      : 525,056
netC_total          : 774
netC_trainable      : 774

Initial source-training learning rates:
 netF: 0.001
 netB: 0.01
 netC: 0.01

Criterion:
 CrossEntropyLabelSmooth(num_classes=3, epsilon=0.1)

VERIFYING THREE-SEED UPA LIST FILES

Seed 42
 source_train        : 1506
 source_val          : 323
 source_test         : 323
 target_hard_train   : 7200
 target_hard_test    : 1800

Seed 123
 source_train        : 1506
 source_val          : 323
 source_test         : 323
 target_hard_train   : 7200
 target_hard_test    : 1800

Seed 2024
 source_train        : 1506
 source_val          : 323
 source_test         : 323
 target_hard_train   : 7199
 target_hard_test    : 1800

BLOCK 7 COMPLETE
Architecture : ResNet50 -> 256-D BN bott

In [15]:
# ============================================================
# BLOCK 8A — SPACE-FREE PATH ALIASES FOR OFFICIAL UPA LOADER
# ============================================================

from pathlib import Path
import pandas as pd
import hashlib
import os
import json

print("=" * 80)
print("CREATING SPACE-FREE UPA IMAGE ALIASES")
print("=" * 80)

ALIAS_ROOT = (
    OUTPUT_ROOT /
    "upa_path_aliases"
)

ALIAS_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

SEEDS = [42, 123, 2024]

ROLES = [
    "source_train",
    "source_val",
    "source_test",
    "target_hard_train",
    "target_hard_test",
]

EXPECTED_USABLE_ROWS = {
    (42, "source_train"): 1506,
    (42, "source_val"): 323,
    (42, "source_test"): 323,
    (42, "target_hard_train"): 7200,
    (42, "target_hard_test"): 1800,

    (123, "source_train"): 1506,
    (123, "source_val"): 323,
    (123, "source_test"): 323,
    (123, "target_hard_train"): 7200,
    (123, "target_hard_test"): 1800,

    (2024, "source_train"): 1506,
    (2024, "source_val"): 323,
    (2024, "source_test"): 323,
    (2024, "target_hard_train"): 7199,
    (2024, "target_hard_test"): 1800,
}

# ------------------------------------------------------------
# 1. Create deterministic alias for one image
# ------------------------------------------------------------

def get_alias(original_path):

    original_path = Path(
        str(original_path)
    ).resolve()

    if not original_path.exists():
        raise FileNotFoundError(
            original_path
        )

    digest = hashlib.sha256(
        str(original_path).encode("utf-8")
    ).hexdigest()[:24]

    suffix = original_path.suffix.lower()

    if not suffix:
        suffix = ".img"

    alias = (
        ALIAS_ROOT /
        f"img_{digest}{suffix}"
    )

    if alias.exists() or alias.is_symlink():

        # Verify existing alias points to same source
        if alias.resolve() != original_path:
            raise RuntimeError(
                f"Alias collision:\n"
                f"{alias}\n"
                f"{original_path}"
            )

    else:

        os.symlink(
            original_path,
            alias
        )

    # Critical: UPA parser requires no spaces
    assert " " not in str(alias)

    return alias


# ------------------------------------------------------------
# 2. Rewrite every UPA list using aliases
# ------------------------------------------------------------

registry = []
mapping_records = []

for seed in SEEDS:

    print("\n" + "-" * 80)
    print(f"SEED {seed}")
    print("-" * 80)

    seed_dir = (
        LIST_ROOT /
        f"seed_{seed}"
    )

    for role in ROLES:

        resolved_csv = (
            seed_dir /
            f"{role}_resolved.csv"
        )

        df = pd.read_csv(
            resolved_csv
        )

        expected = EXPECTED_USABLE_ROWS[
            (seed, role)
        ]

        if len(df) != expected:
            raise RuntimeError(
                f"Seed {seed}, {role}: "
                f"expected {expected}, found {len(df)}"
            )

        lines = []

        for _, row in df.iterrows():

            source_path = Path(
                row["resolved_path"]
            )

            label = int(
                row["label"]
            )

            alias = get_alias(
                source_path
            )

            line = (
                f"{alias} {label}"
            )

            # Official UPA expects exactly:
            # PATH LABEL
            parts = line.split()

            if len(parts) != 2:
                raise RuntimeError(
                    f"UPA line still has parser ambiguity:\n{line}"
                )

            if parts[1] not in {
                "0", "1", "2"
            }:
                raise RuntimeError(
                    f"Bad class label in line:\n{line}"
                )

            lines.append(
                line
            )

            mapping_records.append({
                "seed": seed,
                "role": role,
                "original_path":
                    str(source_path),
                "alias_path":
                    str(alias),
                "label": label
            })

        # --------------------------------------------
        # Replace only the UPA text file.
        # Original locked CSV remains untouched.
        # --------------------------------------------

        txt_path = (
            seed_dir /
            f"{role}.txt"
        )

        with open(
            txt_path,
            "w"
        ) as f:

            f.write(
                "\n".join(lines)
                + "\n"
            )

        digest = hashlib.sha256(
            (
                "\n".join(lines)
                + "\n"
            ).encode("utf-8")
        ).hexdigest()

        registry.append({
            "seed": seed,
            "role": role,
            "rows": len(lines),
            "txt_path": str(txt_path),
            "sha256_after_aliasing":
                digest
        })

        print(
            f"{role:<20} "
            f"{len(lines):>5} rows"
        )


# ------------------------------------------------------------
# 3. Verify every rewritten line parses exactly as UPA expects
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("UPA PARSER FORMAT CHECK")
print("=" * 80)

for seed in SEEDS:

    for role in ROLES:

        txt_path = (
            LIST_ROOT /
            f"seed_{seed}" /
            f"{role}.txt"
        )

        with open(
            txt_path,
            "r"
        ) as f:

            lines = [
                x.strip()
                for x in f
                if x.strip()
            ]

        expected = EXPECTED_USABLE_ROWS[
            (seed, role)
        ]

        assert len(lines) == expected

        for line in lines:

            parts = line.split()

            if len(parts) != 2:
                raise RuntimeError(
                    f"Invalid UPA line:\n{line}"
                )

            path, label = parts

            if not Path(path).exists():
                raise RuntimeError(
                    f"Alias does not resolve:\n{path}"
                )

            if label not in {
                "0", "1", "2"
            }:
                raise RuntimeError(
                    f"Invalid label:\n{line}"
                )

        print(
            f"Seed {seed:<4} | "
            f"{role:<20} -> OK"
        )


# ------------------------------------------------------------
# 4. Verify symlink count and uniqueness
# ------------------------------------------------------------

mapping_df = pd.DataFrame(
    mapping_records
)

unique_original = (
    mapping_df["original_path"]
    .nunique()
)

unique_alias = (
    mapping_df["alias_path"]
    .nunique()
)

if unique_original != unique_alias:
    raise RuntimeError(
        "Original-to-alias mapping is not one-to-one."
    )

print("\nUnique original images :", unique_original)
print("Unique alias images    :", unique_alias)

# ------------------------------------------------------------
# 5. Save documentation
# ------------------------------------------------------------

mapping_path = (
    CONFIG_OUTPUT /
    "upa_path_alias_mapping.csv"
)

mapping_df.to_csv(
    mapping_path,
    index=False
)

registry_df = pd.DataFrame(
    registry
)

registry_path = (
    CONFIG_OUTPUT /
    "upa_alias_list_registry.csv"
)

registry_df.to_csv(
    registry_path,
    index=False
)

with open(
    CONFIG_OUTPUT /
    "upa_path_compatibility_fix.json",
    "w"
) as f:

    json.dump(
        {
            "reason":
                "Official UPA ImageList splits list entries "
                "on whitespace; some Kaggle source paths "
                "contain spaces.",

            "solution":
                "Created deterministic local symlinks with "
                "space-free hashed filenames.",

            "data_membership_changed":
                False,

            "labels_changed":
                False,

            "images_copied_or_modified":
                False,

            "locked_manifests_changed":
                False,

            "only_upa_text_paths_changed":
                True,

            "alias_root":
                str(ALIAS_ROOT),

            "unique_images":
                int(unique_original)
        },
        f,
        indent=2
    )

print("\n" + "=" * 80)
print("BLOCK 8A COMPLETE")
print("=" * 80)

print(
    "Alias root :",
    ALIAS_ROOT
)

print(
    "Mapping    :",
    mapping_path
)

print(
    "\nNo samples, labels, or split memberships were changed."
)

CREATING SPACE-FREE UPA IMAGE ALIASES

--------------------------------------------------------------------------------
SEED 42
--------------------------------------------------------------------------------
source_train          1506 rows
source_val             323 rows
source_test            323 rows
target_hard_train     7200 rows
target_hard_test      1800 rows

--------------------------------------------------------------------------------
SEED 123
--------------------------------------------------------------------------------
source_train          1506 rows
source_val             323 rows
source_test            323 rows
target_hard_train     7200 rows
target_hard_test      1800 rows

--------------------------------------------------------------------------------
SEED 2024
--------------------------------------------------------------------------------
source_train          1506 rows
source_val             323 rows
source_test            323 rows
target_hard_train     7199 row

In [16]:
# ============================================================
# BLOCK 8 — SOURCE TRAINING ENGINE + ONE-BATCH SANITY CHECK
# ============================================================

from pathlib import Path
from copy import deepcopy
import json
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import sys

print("=" * 80)
print("BUILDING UPA SOURCE-TRAINING ENGINE")
print("=" * 80)

# ------------------------------------------------------------
# 1. Official UPA imports
# ------------------------------------------------------------

if str(UPA_ROOT) not in sys.path:
    sys.path.insert(0, str(UPA_ROOT))

from model import network
from loaders.data_list import ImageList
from utils.loss import CrossEntropyLabelSmooth

# ------------------------------------------------------------
# 2. Read UPA text list
# ------------------------------------------------------------

def read_list_file(path):

    with open(path, "r") as f:
        lines = [
            line.strip()
            for line in f
            if line.strip()
        ]

    return lines


# ------------------------------------------------------------
# 3. Data loaders using official ImageList
# ------------------------------------------------------------

def build_source_loaders(
    seed,
    batch_size=64,
    num_workers=4
):

    seed_dir = (
        LIST_ROOT /
        f"seed_{seed}"
    )

    train_lines = read_list_file(
        seed_dir /
        "source_train.txt"
    )

    val_lines = read_list_file(
        seed_dir /
        "source_val.txt"
    )

    test_lines = read_list_file(
        seed_dir /
        "source_test.txt"
    )

    train_ds = ImageList(
        train_lines,
        transform=upa_source_train_transform
    )

    val_ds = ImageList(
        val_lines,
        transform=upa_source_eval_transform
    )

    test_ds = ImageList(
        test_lines,
        transform=upa_source_eval_transform
    )

    # Generator makes loader shuffling seed-specific and reproducible
    g = torch.Generator()
    g.manual_seed(seed)

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        drop_last=False,
        pin_memory=True,
        generator=g
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size * 2,
        shuffle=False,
        num_workers=num_workers,
        drop_last=False,
        pin_memory=True
    )

    test_loader = DataLoader(
        test_ds,
        batch_size=batch_size * 2,
        shuffle=False,
        num_workers=num_workers,
        drop_last=False,
        pin_memory=True
    )

    return (
        train_loader,
        val_loader,
        test_loader
    )


# ------------------------------------------------------------
# 4. Official architecture
# ------------------------------------------------------------

def build_upa_source_model():

    netF = network.ResBase(
        res_name="resnet50"
    ).to(DEVICE)

    netB = network.feat_bootleneck(
        type="bn",
        feature_dim=netF.in_features,
        bottleneck_dim=256,
        normalized=False
    ).to(DEVICE)

    netC = network.feat_classifier(
        type="wn",
        class_num=3,
        bottleneck_dim=256,
        bias=True
    ).to(DEVICE)

    return netF, netB, netC


# ------------------------------------------------------------
# 5. Official source optimizer
# ------------------------------------------------------------

def build_source_optimizer(
    netF,
    netB,
    netC,
    base_lr=1e-2
):

    param_groups = []

    for _, p in netF.named_parameters():

        param_groups.append({
            "params": p,
            "lr": base_lr * 0.1
        })

    for _, p in netB.named_parameters():

        param_groups.append({
            "params": p,
            "lr": base_lr
        })

    for _, p in netC.named_parameters():

        param_groups.append({
            "params": p,
            "lr": base_lr
        })

    optimizer = torch.optim.SGD(
        param_groups
    )

    for group in optimizer.param_groups:
        group["lr0"] = group["lr"]

    return optimizer


# ------------------------------------------------------------
# 6. Official polynomial scheduler
# ------------------------------------------------------------

def upa_source_lr_scheduler(
    optimizer,
    iter_num,
    max_iter,
    gamma=10.0,
    power=0.75
):

    decay = (
        1.0
        + gamma * iter_num / max_iter
    ) ** (-power)

    for group in optimizer.param_groups:

        group["lr"] = (
            group["lr0"] * decay
        )

        group["weight_decay"] = 1e-3
        group["momentum"] = 0.9
        group["nesterov"] = True

    return optimizer


# ------------------------------------------------------------
# 7. Evaluation helper
# ------------------------------------------------------------

@torch.no_grad()
def evaluate_source_accuracy(
    loader,
    netF,
    netB,
    netC
):

    netF.eval()
    netB.eval()
    netC.eval()

    correct = 0
    total = 0

    for images, labels in loader:

        images = images.to(
            DEVICE,
            non_blocking=True
        )

        labels = labels.to(
            DEVICE,
            non_blocking=True
        )

        logits = netC(
            netB(
                netF(images)
            )
        )

        preds = logits.argmax(
            dim=1
        )

        correct += (
            preds == labels
        ).sum().item()

        total += labels.size(0)

    return correct / total


# ------------------------------------------------------------
# 8. Source-training function
# ------------------------------------------------------------

def train_upa_source_seed(
    seed,
    max_epoch=20,
    batch_size=64,
    num_workers=4
):

    set_seed(seed)

    seed_output = (
        SOURCE_OUTPUT /
        f"seed_{seed}"
    )

    seed_output.mkdir(
        parents=True,
        exist_ok=True
    )

    train_loader, val_loader, test_loader = (
        build_source_loaders(
            seed=seed,
            batch_size=batch_size,
            num_workers=num_workers
        )
    )

    netF, netB, netC = (
        build_upa_source_model()
    )

    optimizer = build_source_optimizer(
        netF,
        netB,
        netC,
        base_lr=1e-2
    )

    criterion = CrossEntropyLabelSmooth(
        num_classes=3,
        epsilon=0.1,
        use_gpu=True
    )

    steps_per_epoch = len(
        train_loader
    )

    max_iter = (
        max_epoch *
        steps_per_epoch
    )

    iter_num = 0
    best_val_acc = -1.0
    best_epoch = -1

    history = []

    for epoch in range(
        1,
        max_epoch + 1
    ):

        netF.train()
        netB.train()
        netC.train()

        running_loss = 0.0
        seen = 0

        for images, labels in train_loader:

            if images.size(0) == 1:
                continue

            iter_num += 1

            upa_source_lr_scheduler(
                optimizer,
                iter_num,
                max_iter,
                gamma=10.0,
                power=0.75
            )

            images = images.to(
                DEVICE,
                non_blocking=True
            )

            labels = labels.to(
                DEVICE,
                non_blocking=True
            )

            logits = netC(
                netB(
                    netF(images)
                )
            )

            loss = criterion(
                logits,
                labels
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            loss.backward()

            optimizer.step()

            batch_n = labels.size(0)

            running_loss += (
                loss.item() *
                batch_n
            )

            seen += batch_n

        train_loss = (
            running_loss / seen
        )

        val_acc = (
            evaluate_source_accuracy(
                val_loader,
                netF,
                netB,
                netC
            )
        )

        current_lr_F = (
            optimizer.param_groups[0]["lr"]
        )

        current_lr_B = (
            optimizer.param_groups[
                len(list(netF.parameters()))
            ]["lr"]
        )

        history.append({
            "seed": seed,
            "epoch": epoch,
            "train_loss": train_loss,
            "val_accuracy": val_acc,
            "lr_backbone": current_lr_F,
            "lr_bottleneck": current_lr_B
        })

        print(
            f"Seed {seed} | "
            f"Epoch {epoch:02d}/{max_epoch} | "
            f"loss={train_loss:.6f} | "
            f"val_acc={val_acc:.6f}"
        )

        # Save the actual best validation checkpoint immediately.
        if val_acc >= best_val_acc:

            best_val_acc = val_acc
            best_epoch = epoch

            torch.save(
                netF.state_dict(),
                seed_output /
                "source_F.pt"
            )

            torch.save(
                netB.state_dict(),
                seed_output /
                "source_B.pt"
            )

            torch.save(
                netC.state_dict(),
                seed_output /
                "source_C.pt"
            )

            with open(
                seed_output /
                "best_checkpoint.json",
                "w"
            ) as f:

                json.dump(
                    {
                        "seed": seed,
                        "best_epoch": best_epoch,
                        "best_val_accuracy":
                            best_val_acc
                    },
                    f,
                    indent=2
                )

        pd.DataFrame(
            history
        ).to_csv(
            seed_output /
            "source_training_history.csv",
            index=False
        )

    # --------------------------------------------------------
    # Reload the selected source checkpoint
    # --------------------------------------------------------

    netF.load_state_dict(
        torch.load(
            seed_output /
            "source_F.pt",
            map_location=DEVICE
        )
    )

    netB.load_state_dict(
        torch.load(
            seed_output /
            "source_B.pt",
            map_location=DEVICE
        )
    )

    netC.load_state_dict(
        torch.load(
            seed_output /
            "source_C.pt",
            map_location=DEVICE
        )
    )

    source_test_acc = (
        evaluate_source_accuracy(
            test_loader,
            netF,
            netB,
            netC
        )
    )

    summary = {
        "seed": seed,
        "best_epoch": best_epoch,
        "best_val_accuracy":
            float(best_val_acc),
        "source_test_accuracy":
            float(source_test_acc),
        "max_epoch": max_epoch,
        "batch_size": batch_size,
        "optimizer_steps":
            int(iter_num)
    }

    with open(
        seed_output /
        "source_training_summary.json",
        "w"
    ) as f:

        json.dump(
            summary,
            f,
            indent=2
        )

    return summary


# ------------------------------------------------------------
# 9. One-batch SANITY CHECK only
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("ONE-BATCH SANITY CHECK — SEED 42")
print("=" * 80)

set_seed(42)

train_loader, val_loader, test_loader = (
    build_source_loaders(
        seed=42,
        batch_size=64,
        num_workers=4
    )
)

netF, netB, netC = (
    build_upa_source_model()
)

criterion = CrossEntropyLabelSmooth(
    num_classes=3,
    epsilon=0.1,
    use_gpu=True
)

images, labels = next(
    iter(train_loader)
)

images = images.to(
    DEVICE
)

labels = labels.to(
    DEVICE
)

netF.train()
netB.train()
netC.train()

logits = netC(
    netB(
        netF(images)
    )
)

loss = criterion(
    logits,
    labels
)

print(
    "Batch shape :",
    tuple(images.shape)
)

print(
    "Label shape :",
    tuple(labels.shape)
)

print(
    "Logit shape :",
    tuple(logits.shape)
)

print(
    "Loss        :",
    float(loss.item())
)

print(
    "Train steps :",
    len(train_loader)
)

print(
    "Val batches :",
    len(val_loader)
)

print(
    "Test batches:",
    len(test_loader)
)

assert tuple(images.shape) == (
    64,
    3,
    224,
    224
)

assert tuple(logits.shape) == (
    64,
    3
)

assert torch.isfinite(loss)

# ------------------------------------------------------------
# 10. Cleanup
# ------------------------------------------------------------

del images
del labels
del logits
del loss
del netF
del netB
del netC
del criterion
del train_loader
del val_loader
del test_loader

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\n" + "=" * 80)
print("BLOCK 8 COMPLETE")
print("=" * 80)
print(
    "Source-training engine passed "
    "the one-batch sanity check."
)

BUILDING UPA SOURCE-TRAINING ENGINE

ONE-BATCH SANITY CHECK — SEED 42


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Batch shape : (64, 3, 224, 224)
Label shape : (64,)
Logit shape : (64, 3)
Loss        : 1.2349929809570312
Train steps : 24
Val batches : 3
Test batches: 3

BLOCK 8 COMPLETE
Source-training engine passed the one-batch sanity check.


In [17]:
# ============================================================
# BLOCK 9 — TRAIN OFFICIAL UPA SOURCE MODELS
# ALL THREE SEEDS: 42, 123, 2024
# ============================================================

from pathlib import Path
import json
import pandas as pd
import torch
import time
import gc

print("=" * 80)
print("OFFICIAL UPA SOURCE TRAINING — ALL THREE SEEDS")
print("=" * 80)

SOURCE_RUN_SUMMARY = (
    METRIC_OUTPUT /
    "upa_source_training_all_seeds.csv"
)

all_source_results = []

for seed in SEEDS:

    print("\n" + "=" * 80)
    print(f"SEED {seed}")
    print("=" * 80)

    seed_output = (
        SOURCE_OUTPUT /
        f"seed_{seed}"
    )

    seed_output.mkdir(
        parents=True,
        exist_ok=True
    )

    summary_file = (
        seed_output /
        "source_training_summary.json"
    )

    required_ckpts = [
        seed_output / "source_F.pt",
        seed_output / "source_B.pt",
        seed_output / "source_C.pt"
    ]

    # --------------------------------------------------------
    # Resume-safe:
    # skip only if summary AND all three checkpoints exist
    # --------------------------------------------------------

    if (
        summary_file.exists()
        and all(p.exists() for p in required_ckpts)
    ):

        with open(
            summary_file,
            "r"
        ) as f:
            summary = json.load(f)

        print("Completed checkpoint already exists.")
        print(
            f" best_epoch={summary['best_epoch']} | "
            f"val_acc={summary['best_val_accuracy']:.6f} | "
            f"source_test_acc={summary['source_test_accuracy']:.6f}"
        )

        all_source_results.append(
            summary
        )

        continue

    # --------------------------------------------------------
    # Clean incomplete checkpoint state for this seed only
    # --------------------------------------------------------

    incomplete = (
        summary_file.exists()
        or any(p.exists() for p in required_ckpts)
    )

    if incomplete:

        print(
            "Incomplete previous state detected. "
            "Restarting this seed cleanly."
        )

        for p in required_ckpts:
            if p.exists():
                p.unlink()

        if summary_file.exists():
            summary_file.unlink()

        history_path = (
            seed_output /
            "source_training_history.csv"
        )

        best_path = (
            seed_output /
            "best_checkpoint.json"
        )

        if history_path.exists():
            history_path.unlink()

        if best_path.exists():
            best_path.unlink()

    # --------------------------------------------------------
    # Train
    # --------------------------------------------------------

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    gc.collect()

    start_time = time.time()

    try:

        summary = train_upa_source_seed(
            seed=seed,
            max_epoch=UPA_SOURCE_CONFIG["max_epoch"],
            batch_size=UPA_SOURCE_CONFIG["batch_size"],
            num_workers=UPA_SOURCE_CONFIG["num_workers"]
        )

    except RuntimeError as e:

        if "out of memory" in str(e).lower():

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            raise RuntimeError(
                f"CUDA OOM during Seed {seed}. "
                "Stopped without silently changing the official "
                "batch size of 64."
            ) from e

        raise

    elapsed_min = (
        time.time() - start_time
    ) / 60.0

    summary["elapsed_minutes"] = (
        float(elapsed_min)
    )

    # Rewrite summary including runtime
    with open(
        summary_file,
        "w"
    ) as f:

        json.dump(
            summary,
            f,
            indent=2
        )

    all_source_results.append(
        summary
    )

    # --------------------------------------------------------
    # Save aggregate immediately after every seed
    # --------------------------------------------------------

    pd.DataFrame(
        all_source_results
    ).to_csv(
        SOURCE_RUN_SUMMARY,
        index=False
    )

    print("\nSeed completed:")
    print(
        f" Best epoch       : "
        f"{summary['best_epoch']}"
    )

    print(
        f" Best val accuracy: "
        f"{summary['best_val_accuracy']:.6f}"
    )

    print(
        f" Source test acc  : "
        f"{summary['source_test_accuracy']:.6f}"
    )

    print(
        f" Runtime          : "
        f"{elapsed_min:.2f} min"
    )

    # --------------------------------------------------------
    # Release GPU memory before next seed
    # --------------------------------------------------------

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ============================================================
# FINAL CONSISTENCY CHECK
# ============================================================

print("\n" + "=" * 80)
print("FINAL SOURCE CHECKPOINT AUDIT")
print("=" * 80)

final_rows = []

for seed in SEEDS:

    seed_output = (
        SOURCE_OUTPUT /
        f"seed_{seed}"
    )

    required = {
        "F": seed_output / "source_F.pt",
        "B": seed_output / "source_B.pt",
        "C": seed_output / "source_C.pt",
        "summary":
            seed_output /
            "source_training_summary.json",
        "history":
            seed_output /
            "source_training_history.csv"
    }

    for name, path in required.items():

        if not path.exists():
            raise RuntimeError(
                f"Seed {seed}: missing {name}: {path}"
            )

    with open(
        required["summary"],
        "r"
    ) as f:
        s = json.load(f)

    final_rows.append(
        s
    )

    print(
        f"Seed {seed:<4} | "
        f"best_epoch={s['best_epoch']:<2} | "
        f"val={s['best_val_accuracy']:.6f} | "
        f"source_test={s['source_test_accuracy']:.6f}"
    )

final_df = pd.DataFrame(
    final_rows
)

final_df.to_csv(
    SOURCE_RUN_SUMMARY,
    index=False
)

print("\n" + "=" * 80)
print("BLOCK 9 COMPLETE")
print("=" * 80)

print(
    "Source summary:",
    SOURCE_RUN_SUMMARY
)

print(
    "\nAll three official UPA source checkpoints "
    "are saved and ready for source/target evaluation."
)

OFFICIAL UPA SOURCE TRAINING — ALL THREE SEEDS

SEED 42
Seed 42 | Epoch 01/20 | loss=0.514122 | val_acc=0.987616
Seed 42 | Epoch 02/20 | loss=0.379181 | val_acc=0.996904
Seed 42 | Epoch 03/20 | loss=0.356841 | val_acc=0.993808
Seed 42 | Epoch 04/20 | loss=0.349729 | val_acc=1.000000
Seed 42 | Epoch 05/20 | loss=0.343039 | val_acc=1.000000
Seed 42 | Epoch 06/20 | loss=0.339887 | val_acc=1.000000
Seed 42 | Epoch 07/20 | loss=0.332889 | val_acc=1.000000
Seed 42 | Epoch 08/20 | loss=0.331589 | val_acc=1.000000
Seed 42 | Epoch 09/20 | loss=0.327093 | val_acc=1.000000
Seed 42 | Epoch 10/20 | loss=0.328353 | val_acc=1.000000
Seed 42 | Epoch 11/20 | loss=0.326510 | val_acc=1.000000
Seed 42 | Epoch 12/20 | loss=0.323397 | val_acc=1.000000
Seed 42 | Epoch 13/20 | loss=0.325248 | val_acc=1.000000
Seed 42 | Epoch 14/20 | loss=0.322453 | val_acc=1.000000
Seed 42 | Epoch 15/20 | loss=0.323964 | val_acc=0.996904
Seed 42 | Epoch 16/20 | loss=0.322046 | val_acc=1.000000
Seed 42 | Epoch 17/20 | loss=0.3

In [18]:
# ============================================================
# BLOCK 10 — UPA SOURCE-ONLY EVALUATION
# Source test + hard-target test, all three seeds
# Metrics: Accuracy, Macro-F1, ECE, Brier, NLL
# Saves sample-level predictions
# ============================================================

from pathlib import Path
import json
import gc
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix
)

print("=" * 80)
print("UPA SOURCE-ONLY EVALUATION — ALL THREE SEEDS")
print("=" * 80)

# ------------------------------------------------------------
# 1. Metric helpers
# ------------------------------------------------------------

def multiclass_ece(
    probs,
    labels,
    n_bins=15
):
    probs = np.asarray(probs)
    labels = np.asarray(labels)

    confidences = probs.max(axis=1)
    predictions = probs.argmax(axis=1)
    correct = (predictions == labels).astype(float)

    bin_edges = np.linspace(
        0.0,
        1.0,
        n_bins + 1
    )

    ece = 0.0

    for i in range(n_bins):

        lo = bin_edges[i]
        hi = bin_edges[i + 1]

        if i == 0:
            mask = (
                (confidences >= lo)
                & (confidences <= hi)
            )
        else:
            mask = (
                (confidences > lo)
                & (confidences <= hi)
            )

        if not np.any(mask):
            continue

        bin_acc = correct[mask].mean()
        bin_conf = confidences[mask].mean()
        bin_weight = mask.mean()

        ece += (
            bin_weight *
            abs(bin_acc - bin_conf)
        )

    return float(ece)


def multiclass_brier(
    probs,
    labels,
    num_classes=3
):
    probs = np.asarray(
        probs,
        dtype=np.float64
    )

    labels = np.asarray(
        labels,
        dtype=np.int64
    )

    one_hot = np.eye(
        num_classes
    )[labels]

    return float(
        np.mean(
            np.sum(
                (probs - one_hot) ** 2,
                axis=1
            )
        )
    )


def multiclass_nll(
    probs,
    labels
):
    probs = np.asarray(
        probs,
        dtype=np.float64
    )

    labels = np.asarray(
        labels,
        dtype=np.int64
    )

    eps = 1e-12

    chosen = probs[
        np.arange(len(labels)),
        labels
    ]

    chosen = np.clip(
        chosen,
        eps,
        1.0
    )

    return float(
        -np.mean(
            np.log(chosen)
        )
    )


def calculate_metrics(
    labels,
    probs
):
    labels = np.asarray(labels)
    probs = np.asarray(probs)

    preds = probs.argmax(
        axis=1
    )

    return {
        "accuracy":
            float(
                accuracy_score(
                    labels,
                    preds
                )
            ),

        "macro_f1":
            float(
                f1_score(
                    labels,
                    preds,
                    average="macro"
                )
            ),

        "ece":
            multiclass_ece(
                probs,
                labels,
                n_bins=15
            ),

        "brier":
            multiclass_brier(
                probs,
                labels,
                num_classes=3
            ),

        "nll":
            multiclass_nll(
                probs,
                labels
            )
    }


# ------------------------------------------------------------
# 2. Evaluation dataset loader
# ------------------------------------------------------------

def build_eval_loader(
    seed,
    role,
    batch_size=128,
    num_workers=4
):
    list_path = (
        LIST_ROOT /
        f"seed_{seed}" /
        f"{role}.txt"
    )

    lines = read_list_file(
        list_path
    )

    ds = ImageList(
        lines,
        transform=upa_source_eval_transform
    )

    loader = DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        drop_last=False,
        pin_memory=True
    )

    return (
        lines,
        loader
    )


# ------------------------------------------------------------
# 3. Load a trained UPA source checkpoint
# ------------------------------------------------------------

def load_upa_source_checkpoint(
    seed
):
    ckpt_dir = (
        SOURCE_OUTPUT /
        f"seed_{seed}"
    )

    netF, netB, netC = (
        build_upa_source_model()
    )

    netF.load_state_dict(
        torch.load(
            ckpt_dir /
            "source_F.pt",
            map_location=DEVICE,
            weights_only=True
        )
    )

    netB.load_state_dict(
        torch.load(
            ckpt_dir /
            "source_B.pt",
            map_location=DEVICE,
            weights_only=True
        )
    )

    netC.load_state_dict(
        torch.load(
            ckpt_dir /
            "source_C.pt",
            map_location=DEVICE,
            weights_only=True
        )
    )

    netF.eval()
    netB.eval()
    netC.eval()

    return netF, netB, netC


# ------------------------------------------------------------
# 4. Prediction function
# ------------------------------------------------------------

@torch.no_grad()
def predict_upa(
    loader,
    netF,
    netB,
    netC
):
    all_logits = []
    all_probs = []
    all_labels = []

    for images, labels in loader:

        images = images.to(
            DEVICE,
            non_blocking=True
        )

        logits = netC(
            netB(
                netF(images)
            )
        )

        probs = F.softmax(
            logits,
            dim=1
        )

        all_logits.append(
            logits.cpu().numpy()
        )

        all_probs.append(
            probs.cpu().numpy()
        )

        all_labels.append(
            labels.numpy()
        )

    logits = np.concatenate(
        all_logits,
        axis=0
    )

    probs = np.concatenate(
        all_probs,
        axis=0
    )

    labels = np.concatenate(
        all_labels,
        axis=0
    )

    return (
        logits,
        probs,
        labels
    )


# ------------------------------------------------------------
# 5. Restore original paths from alias mapping
# ------------------------------------------------------------

alias_map = pd.read_csv(
    CONFIG_OUTPUT /
    "upa_path_alias_mapping.csv"
)

alias_to_original = (
    alias_map[
        [
            "alias_path",
            "original_path"
        ]
    ]
    .drop_duplicates()
    .set_index("alias_path")[
        "original_path"
    ]
    .to_dict()
)


def parse_upa_line(line):
    path, label = line.rsplit(
        " ",
        1
    )

    return (
        path,
        int(label)
    )


# ------------------------------------------------------------
# 6. Semantic-label consistency audit
# ------------------------------------------------------------

def infer_class_from_path(path):

    s = str(path).lower()

    s_clean = (
        s.replace("-", "")
         .replace("_", "")
         .replace(" ", "")
    )

    if (
        "earlyblight" in s_clean
        or "earlyblt" in s_clean
    ):
        return 0

    if (
        "lateblight" in s_clean
        or "lateblt" in s_clean
    ):
        return 1

    if "healthy" in s_clean:
        return 2

    return None


semantic_checked = 0
semantic_mismatch = []

for _, row in alias_map.iterrows():

    inferred = infer_class_from_path(
        row["original_path"]
    )

    if inferred is None:
        continue

    semantic_checked += 1

    if int(row["label"]) != inferred:

        semantic_mismatch.append({
            "path":
                row["original_path"],
            "stored_label":
                int(row["label"]),
            "inferred_label":
                int(inferred)
        })

if semantic_mismatch:

    mismatch_df = pd.DataFrame(
        semantic_mismatch
    )

    mismatch_path = (
        CONFIG_OUTPUT /
        "semantic_label_mismatches.csv"
    )

    mismatch_df.to_csv(
        mismatch_path,
        index=False
    )

    raise RuntimeError(
        f"Semantic label mapping mismatch detected. "
        f"Stopped before final evaluation. "
        f"Report: {mismatch_path}"
    )

print(
    f"Semantic label audit: "
    f"{semantic_checked:,} identifiable entries checked, "
    f"0 mismatches."
)


# ------------------------------------------------------------
# 7. Evaluate all three seeds
# ------------------------------------------------------------

evaluation_roles = [
    "source_test",
    "target_hard_test"
]

metric_rows = []

for seed in SEEDS:

    print("\n" + "=" * 80)
    print(f"SEED {seed}")
    print("=" * 80)

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    netF, netB, netC = (
        load_upa_source_checkpoint(
            seed
        )
    )

    for role in evaluation_roles:

        lines, loader = (
            build_eval_loader(
                seed=seed,
                role=role,
                batch_size=128,
                num_workers=4
            )
        )

        logits, probs, labels = (
            predict_upa(
                loader,
                netF,
                netB,
                netC
            )
        )

        if len(lines) != len(labels):
            raise RuntimeError(
                f"Prediction length mismatch: "
                f"Seed {seed}, {role}"
            )

        metrics = calculate_metrics(
            labels,
            probs
        )

        preds = probs.argmax(
            axis=1
        )

        cm = confusion_matrix(
            labels,
            preds,
            labels=[0, 1, 2]
        )

        # --------------------------------------------
        # Save sample-level predictions
        # --------------------------------------------

        prediction_records = []

        for i, line in enumerate(lines):

            alias_path, list_label = (
                parse_upa_line(line)
            )

            if (
                int(list_label)
                != int(labels[i])
            ):
                raise RuntimeError(
                    "List-label / DataLoader-label "
                    "misalignment detected."
                )

            original_path = (
                alias_to_original.get(
                    alias_path,
                    alias_path
                )
            )

            prediction_records.append({
                "seed":
                    seed,

                "role":
                    role,

                "sample_index":
                    i,

                "original_path":
                    original_path,

                "alias_path":
                    alias_path,

                "true_label":
                    int(labels[i]),

                "pred_label":
                    int(preds[i]),

                "confidence":
                    float(
                        probs[i].max()
                    ),

                "logit_0":
                    float(logits[i, 0]),

                "logit_1":
                    float(logits[i, 1]),

                "logit_2":
                    float(logits[i, 2]),

                "prob_0":
                    float(probs[i, 0]),

                "prob_1":
                    float(probs[i, 1]),

                "prob_2":
                    float(probs[i, 2])
            })

        pred_path = (
            PRED_OUTPUT /
            f"upa_source_seed_{seed}_{role}_predictions.csv"
        )

        pd.DataFrame(
            prediction_records
        ).to_csv(
            pred_path,
            index=False
        )

        # --------------------------------------------
        # Save confusion matrix
        # --------------------------------------------

        cm_path = (
            METRIC_OUTPUT /
            f"upa_source_seed_{seed}_{role}_confusion.csv"
        )

        pd.DataFrame(
            cm,
            index=[
                "true_0_early",
                "true_1_late",
                "true_2_healthy"
            ],
            columns=[
                "pred_0_early",
                "pred_1_late",
                "pred_2_healthy"
            ]
        ).to_csv(
            cm_path
        )

        metric_row = {
            "method":
                "UPA_source_only",

            "seed":
                seed,

            "dataset":
                role,

            "n":
                int(len(labels)),

            **metrics
        }

        metric_rows.append(
            metric_row
        )

        print(
            f"{role:<18} | "
            f"n={len(labels):<4} | "
            f"Acc={metrics['accuracy']:.6f} | "
            f"Macro-F1={metrics['macro_f1']:.6f} | "
            f"ECE={metrics['ece']:.6f} | "
            f"Brier={metrics['brier']:.6f} | "
            f"NLL={metrics['nll']:.6f}"
        )

        print(
            "Confusion:"
        )
        print(cm)

        del loader
        del logits
        del probs
        del labels
        del preds

    del netF
    del netB
    del netC

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ------------------------------------------------------------
# 8. Save seed-level results
# ------------------------------------------------------------

metrics_df = pd.DataFrame(
    metric_rows
)

seed_metrics_path = (
    METRIC_OUTPUT /
    "upa_source_only_seed_metrics.csv"
)

metrics_df.to_csv(
    seed_metrics_path,
    index=False
)


# ------------------------------------------------------------
# 9. Aggregate mean ± SD
# ------------------------------------------------------------

aggregate_rows = []

for dataset_name in evaluation_roles:

    subset = metrics_df[
        metrics_df["dataset"]
        == dataset_name
    ]

    row = {
        "method":
            "UPA_source_only",

        "dataset":
            dataset_name,

        "num_seeds":
            len(subset)
    }

    for metric in [
        "accuracy",
        "macro_f1",
        "ece",
        "brier",
        "nll"
    ]:

        values = subset[
            metric
        ].to_numpy(
            dtype=float
        )

        row[
            f"{metric}_mean"
        ] = float(
            np.mean(values)
        )

        row[
            f"{metric}_std"
        ] = float(
            np.std(
                values,
                ddof=1
            )
        )

    aggregate_rows.append(
        row
    )


aggregate_df = pd.DataFrame(
    aggregate_rows
)

aggregate_path = (
    METRIC_OUTPUT /
    "upa_source_only_aggregate.csv"
)

aggregate_df.to_csv(
    aggregate_path,
    index=False
)


# ------------------------------------------------------------
# 10. Print final hard-target aggregate
# ------------------------------------------------------------

hard = aggregate_df[
    aggregate_df["dataset"]
    == "target_hard_test"
].iloc[0]

print("\n" + "=" * 80)
print("UPA SOURCE-ONLY — HARD TARGET, THREE-SEED AGGREGATE")
print("=" * 80)

print(
    "Accuracy : "
    f"{hard['accuracy_mean']:.6f} "
    f"± {hard['accuracy_std']:.6f}"
)

print(
    "Macro-F1 : "
    f"{hard['macro_f1_mean']:.6f} "
    f"± {hard['macro_f1_std']:.6f}"
)

print(
    "ECE      : "
    f"{hard['ece_mean']:.6f} "
    f"± {hard['ece_std']:.6f}"
)

print(
    "Brier    : "
    f"{hard['brier_mean']:.6f} "
    f"± {hard['brier_std']:.6f}"
)

print(
    "NLL      : "
    f"{hard['nll_mean']:.6f} "
    f"± {hard['nll_std']:.6f}"
)

print("\n" + "=" * 80)
print("BLOCK 10 COMPLETE")
print("=" * 80)

print(
    "Seed metrics :",
    seed_metrics_path
)

print(
    "Aggregate    :",
    aggregate_path
)

print(
    "Predictions  :",
    PRED_OUTPUT
)

UPA SOURCE-ONLY EVALUATION — ALL THREE SEEDS
Semantic label audit: 33,455 identifiable entries checked, 0 mismatches.

SEED 42


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


source_test        | n=323  | Acc=0.993808 | Macro-F1=0.989571 | ECE=0.096430 | Brier=0.025747 | NLL=0.106607
Confusion:
[[149   1   0]
 [  0 149   1]
 [  0   0  23]]
target_hard_test   | n=1800 | Acc=0.507222 | Macro-F1=0.424484 | ECE=0.130050 | Brier=0.566809 | NLL=0.964245
Confusion:
[[589  10   1]
 [297 290  13]
 [399 167  34]]

SEED 123


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


source_test        | n=323  | Acc=0.990712 | Macro-F1=0.981637 | ECE=0.096112 | Brier=0.032396 | NLL=0.116691
Confusion:
[[150   0   0]
 [  1 147   2]
 [  0   0  23]]
target_hard_test   | n=1800 | Acc=0.697222 | Macro-F1=0.687554 | ECE=0.101017 | Brier=0.434315 | NLL=0.764947
Confusion:
[[590   6   4]
 [297 261  42]
 [153  43 404]]

SEED 2024


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


source_test        | n=323  | Acc=0.990712 | Macro-F1=0.976225 | ECE=0.083954 | Brier=0.028384 | NLL=0.105003
Confusion:
[[150   0   0]
 [  0 147   3]
 [  0   0  23]]
target_hard_test   | n=1800 | Acc=0.585000 | Macro-F1=0.550587 | ECE=0.091968 | Brier=0.475091 | NLL=0.800128
Confusion:
[[594   6   0]
 [291 306   3]
 [319 128 153]]

UPA SOURCE-ONLY — HARD TARGET, THREE-SEED AGGREGATE
Accuracy : 0.596481 ± 0.095519
Macro-F1 : 0.554208 ± 0.131572
ECE      : 0.107678 ± 0.019896
Brier    : 0.492072 ± 0.067859
NLL      : 0.843107 ± 0.106374

BLOCK 10 COMPLETE
Seed metrics : /kaggle/working/official_upa_reproduction/metrics/upa_source_only_seed_metrics.csv
Aggregate    : /kaggle/working/official_upa_reproduction/metrics/upa_source_only_aggregate.csv
Predictions  : /kaggle/working/official_upa_reproduction/predictions


In [19]:
# ============================================================
# BLOCK 11 — LOCK OFFICIAL UPA TARGET-ADAPTATION CONFIG
# No adaptation training yet
# ============================================================

from pathlib import Path
import json
import sys
import torch

print("=" * 80)
print("LOCKING OFFICIAL UPA TARGET-ADAPTATION CONFIGURATION")
print("=" * 80)

# ------------------------------------------------------------
# 1. Repository
# ------------------------------------------------------------

if str(UPA_ROOT) not in sys.path:
    sys.path.insert(0, str(UPA_ROOT))

# ------------------------------------------------------------
# 2. Official published Office-Home UPA configuration
#
# From official repository run_office-home.sh:
#
# --par_noisy_ent 1
# --par_su_cl 1
# --par_noisy_cls 0.3
# --su_cl_t 0.1
# --k_val 4
# --sel_ratio 0.6
# --knn_times 2
#
# Core defaults from upa_v2_amp.py:
#
# max_epoch = 15
# batch_size = 64
# lr = 1e-2
# bottleneck = 256
# classifier = bn
# layer = wn
# ------------------------------------------------------------

UPA_TARGET_CONFIG = {

    # Model
    "net": "resnet50",
    "class_num": 3,
    "bottleneck": 256,
    "classifier": "bn",
    "layer": "wn",

    # Optimization
    "max_epoch": 15,
    "batch_size": 64,
    "base_lr": 1e-2,

    # Official target optimizer grouping
    "netF_lr_multiplier": 0.1,
    "netB_lr_multiplier": 1.0,

    "weight_decay": 1e-3,
    "momentum": 0.9,
    "nesterov": True,

    # UPA loss weights
    "par_noisy_cls": 0.3,
    "par_noisy_ent": 1.0,
    "par_su_cl": 1.0,

    # CACL
    "su_cl_t": 0.1,

    # APS / pseudo-label selection
    "k_val": 4,
    "sel_ratio": 0.6,
    "knn_times": 2,

    # Important protocol restrictions
    "source_free": True,
    "source_images_used_during_adaptation": False,
    "target_train_labels_used_for_learning": False,
    "target_test_labels_used_for_model_selection": False,

    # Evaluation
    "target_test_evaluation":
        "after final epoch only",

    # Dataset adaptation
    "target_dataset":
        "locked hard-target training split",

    "seed_policy":
        [42, 123, 2024],

    # Seed 2024 documented image handling
    "seed_2024_target_train_n": 7199,

    # Benchmark wording
    "baseline_name":
        "UPA (official-code reproduction)"
}

# ------------------------------------------------------------
# 3. Confirm required source checkpoints exist
# ------------------------------------------------------------

print("\nChecking source checkpoints...")

for seed in SEEDS:

    src_dir = (
        SOURCE_OUTPUT /
        f"seed_{seed}"
    )

    required = [
        src_dir / "source_F.pt",
        src_dir / "source_B.pt",
        src_dir / "source_C.pt"
    ]

    for path in required:

        if not path.exists():
            raise RuntimeError(
                f"Missing source checkpoint: {path}"
            )

    print(
        f"Seed {seed:<4} -> source_F/B/C found"
    )

# ------------------------------------------------------------
# 4. Confirm target-training lists
# ------------------------------------------------------------

print("\nChecking target-training lists...")

expected_target_n = {
    42: 7200,
    123: 7200,
    2024: 7199
}

for seed in SEEDS:

    p = (
        LIST_ROOT /
        f"seed_{seed}" /
        "target_hard_train.txt"
    )

    if not p.exists():
        raise RuntimeError(
            f"Missing target list: {p}"
        )

    with open(p, "r") as f:

        lines = [
            x.strip()
            for x in f
            if x.strip()
        ]

    if len(lines) != expected_target_n[seed]:

        raise RuntimeError(
            f"Seed {seed}: expected "
            f"{expected_target_n[seed]} target-training "
            f"samples, found {len(lines)}."
        )

    print(
        f"Seed {seed:<4} -> "
        f"{len(lines)} target-training images"
    )

# ------------------------------------------------------------
# 5. Freeze classifier policy
#
# Official UPA:
# netC remains fixed during target adaptation.
# netF and netB are optimized.
# ------------------------------------------------------------

UPA_TARGET_CONFIG[
    "trainable_modules"
] = [
    "netF",
    "netB"
]

UPA_TARGET_CONFIG[
    "frozen_modules"
] = [
    "netC"
]

# ------------------------------------------------------------
# 6. Learning rates
# ------------------------------------------------------------

UPA_TARGET_CONFIG[
    "initial_learning_rates"
] = {
    "netF":
        UPA_TARGET_CONFIG["base_lr"]
        * UPA_TARGET_CONFIG[
            "netF_lr_multiplier"
        ],

    "netB":
        UPA_TARGET_CONFIG["base_lr"]
        * UPA_TARGET_CONFIG[
            "netB_lr_multiplier"
        ],

    "netC":
        0.0
}

print("\nInitial adaptation learning rates:")
print(
    " netF:",
    UPA_TARGET_CONFIG[
        "initial_learning_rates"
    ]["netF"]
)

print(
    " netB:",
    UPA_TARGET_CONFIG[
        "initial_learning_rates"
    ]["netB"]
)

print(
    " netC:",
    UPA_TARGET_CONFIG[
        "initial_learning_rates"
    ]["netC"],
    "(frozen)"
)

# ------------------------------------------------------------
# 7. Save complete protocol before adaptation
# ------------------------------------------------------------

config_path = (
    CONFIG_OUTPUT /
    "upa_target_adaptation_protocol.json"
)

with open(
    config_path,
    "w"
) as f:

    json.dump(
        UPA_TARGET_CONFIG,
        f,
        indent=2
    )

# ------------------------------------------------------------
# 8. Human-readable locked configuration
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("LOCKED UPA TARGET CONFIGURATION")
print("=" * 80)

print(
    "Epochs                :",
    UPA_TARGET_CONFIG["max_epoch"]
)

print(
    "Batch size            :",
    UPA_TARGET_CONFIG["batch_size"]
)

print(
    "Base LR               :",
    UPA_TARGET_CONFIG["base_lr"]
)

print(
    "Pseudo-label CE weight:",
    UPA_TARGET_CONFIG["par_noisy_cls"]
)

print(
    "Entropy/IM weight     :",
    UPA_TARGET_CONFIG["par_noisy_ent"]
)

print(
    "CACL weight           :",
    UPA_TARGET_CONFIG["par_su_cl"]
)

print(
    "CACL temperature      :",
    UPA_TARGET_CONFIG["su_cl_t"]
)

print(
    "APS k                 :",
    UPA_TARGET_CONFIG["k_val"]
)

print(
    "APS selection ratio   :",
    UPA_TARGET_CONFIG["sel_ratio"]
)

print(
    "APS refinement rounds :",
    UPA_TARGET_CONFIG["knn_times"]
)

print(
    "Classifier            : frozen"
)

print(
    "Source images during "
    "adaptation           : NO"
)

print(
    "Target-test labels for "
    "selection            : NO"
)

print("\n" + "=" * 80)
print("BLOCK 11 COMPLETE")
print("=" * 80)

print(
    "Protocol:",
    config_path
)

LOCKING OFFICIAL UPA TARGET-ADAPTATION CONFIGURATION

Checking source checkpoints...
Seed 42   -> source_F/B/C found
Seed 123  -> source_F/B/C found
Seed 2024 -> source_F/B/C found

Checking target-training lists...
Seed 42   -> 7200 target-training images
Seed 123  -> 7200 target-training images
Seed 2024 -> 7199 target-training images

Initial adaptation learning rates:
 netF: 0.001
 netB: 0.01
 netC: 0.0 (frozen)

LOCKED UPA TARGET CONFIGURATION
Epochs                : 15
Batch size            : 64
Base LR               : 0.01
Pseudo-label CE weight: 0.3
Entropy/IM weight     : 1.0
CACL weight           : 1.0
CACL temperature      : 0.1
APS k                 : 4
APS selection ratio   : 0.6
APS refinement rounds : 2
Classifier            : frozen
Source images during adaptation           : NO
Target-test labels for selection            : NO

BLOCK 11 COMPLETE
Protocol: /kaggle/working/official_upa_reproduction/config/upa_target_adaptation_protocol.json


In [20]:
# ============================================================
# BLOCK 12 — OFFICIAL UPA TARGET ADAPTATION
# ALL THREE SEEDS: 42, 123, 2024
#
# Uses:
#   - official pinned UPA implementation
#   - official Office-Home hyperparameters
#   - exact locked hard-target training images
#   - NO source images during adaptation
#   - NO true target labels available to adaptation code
#
# Compatibility-only patch:
#   iter_train.next() -> next(iter_train)
# ============================================================

from pathlib import Path
from argparse import Namespace
import os
import sys
import json
import time
import shutil
import gc
import random
import importlib

import numpy as np
import pandas as pd
import torch

print("=" * 80)
print("OFFICIAL UPA TARGET ADAPTATION — ALL THREE SEEDS")
print("=" * 80)

# ============================================================
# 1. PATCH PYTHON-3 ITERATOR COMPATIBILITY ONLY
# ============================================================

ENGINE_PATH = (
    UPA_ROOT /
    "trainer" /
    "engine.py"
)

ENGINE_BACKUP = (
    CONFIG_OUTPUT /
    "engine_original_pinned.py"
)

if not ENGINE_BACKUP.exists():
    shutil.copy2(
        ENGINE_PATH,
        ENGINE_BACKUP
    )

engine_text = ENGINE_PATH.read_text()

old_count = engine_text.count(
    "iter_train.next()"
)

if old_count > 0:

    engine_text = engine_text.replace(
        "iter_train.next()",
        "next(iter_train)"
    )

    ENGINE_PATH.write_text(
        engine_text
    )

print(
    "Iterator compatibility replacements:",
    old_count
)

# Verify obsolete syntax is gone
patched_text = ENGINE_PATH.read_text()

if "iter_train.next()" in patched_text:
    raise RuntimeError(
        "Iterator compatibility patch failed."
    )

# Document exact compatibility change
compatibility_info = {
    "official_commit":
        commit_sha,

    "file":
        str(ENGINE_PATH),

    "backup":
        str(ENGINE_BACKUP),

    "change":
        "iter_train.next() -> next(iter_train)",

    "reason":
        (
            "Python/PyTorch iterator compatibility. "
            "No algorithmic computation changed."
        ),

    "algorithm_changed":
        False
}

with open(
    CONFIG_OUTPUT /
    "upa_engine_compatibility_patch.json",
    "w"
) as f:

    json.dump(
        compatibility_info,
        f,
        indent=2
    )


# ============================================================
# 2. CREATE LABEL-BLIND TARGET TRAINING LISTS
# ============================================================

BLIND_LIST_ROOT = (
    OUTPUT_ROOT /
    "upa_blind_target_lists"
)

BLIND_LIST_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

expected_target_n = {
    42: 7200,
    123: 7200,
    2024: 7199
}

blind_registry = []

for seed in SEEDS:

    source_list = (
        LIST_ROOT /
        f"seed_{seed}" /
        "target_hard_train.txt"
    )

    blind_seed_dir = (
        BLIND_LIST_ROOT /
        f"seed_{seed}"
    )

    blind_seed_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    blind_list = (
        blind_seed_dir /
        "target_hard_train_blind.txt"
    )

    with open(
        source_list,
        "r"
    ) as f:

        original_lines = [
            x.strip()
            for x in f
            if x.strip()
        ]

    if (
        len(original_lines)
        != expected_target_n[seed]
    ):
        raise RuntimeError(
            f"Seed {seed}: unexpected target size."
        )

    blind_lines = []

    for line in original_lines:

        # Alias paths are space-free,
        # so exactly two tokens are expected.
        path, _true_label = line.rsplit(
            " ",
            1
        )

        # Replace true target label with dummy label.
        blind_lines.append(
            f"{path} 0"
        )

    with open(
        blind_list,
        "w"
    ) as f:

        f.write(
            "\n".join(blind_lines)
            + "\n"
        )

    # Verify
    with open(
        blind_list,
        "r"
    ) as f:

        verify_lines = [
            x.strip()
            for x in f
            if x.strip()
        ]

    assert (
        len(verify_lines)
        == expected_target_n[seed]
    )

    for x in verify_lines:

        path, label = x.rsplit(
            " ",
            1
        )

        assert label == "0"
        assert Path(path).exists()

    blind_registry.append({
        "seed":
            seed,

        "n":
            len(blind_lines),

        "blind_list":
            str(blind_list),

        "true_labels_available_to_adaptation":
            False
    })


pd.DataFrame(
    blind_registry
).to_csv(
    CONFIG_OUTPUT /
    "upa_blind_target_registry.csv",
    index=False
)

print("\nLabel-blind target lists:")
for row in blind_registry:

    print(
        f" Seed {row['seed']:<4} -> "
        f"{row['n']} images"
    )


# ============================================================
# 3. RELOAD OFFICIAL UPA ENGINE AFTER COMPATIBILITY PATCH
# ============================================================

if str(UPA_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(UPA_ROOT)
    )

# Remove previously imported modules if necessary
for modname in [
    "trainer.engine"
]:
    if modname in sys.modules:
        del sys.modules[modname]

import trainer.engine as upa_engine

importlib.reload(
    upa_engine
)

Upa = upa_engine.Upa


# ============================================================
# 4. OFFICIAL UPA ARGUMENT BUILDER
# ============================================================

def build_upa_args(seed):

    source_ckpt_dir = (
        SOURCE_OUTPUT /
        f"seed_{seed}"
    )

    blind_target_list = (
        BLIND_LIST_ROOT /
        f"seed_{seed}" /
        "target_hard_train_blind.txt"
    )

    seed_output = (
        TARGET_OUTPUT /
        f"seed_{seed}"
    )

    seed_output.mkdir(
        parents=True,
        exist_ok=True
    )

    log_path = (
        seed_output /
        "upa_training_log.txt"
    )

    args = Namespace()

    # --------------------------------------------------------
    # Dataset / paths
    # --------------------------------------------------------

    args.s = 0
    args.t = 1

    # Keep official Office-Home branch behavior.
    # Class count is explicitly overwritten to 3.
    args.dset = "office-home"

    args.da = "uda"

    args.t_dset_path = str(
        blind_target_list
    )

    # Official UPA uses this loader to obtain target
    # features/pseudo-labels each epoch.
    # We intentionally use the SAME label-blind target list.
    args.test_dset_path = str(
        blind_target_list
    )

    args.output_dir_src = str(
        source_ckpt_dir
    )

    args.output_dir = str(
        seed_output
    )

    args.append_root = None

    # --------------------------------------------------------
    # Model
    # --------------------------------------------------------

    args.net = "resnet50"
    args.class_num = 3
    args.bottleneck = 256
    args.layer = "wn"
    args.classifier = "bn"

    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------

    args.max_epoch = 15
    args.batch_size = 64
    args.worker = 4
    args.lr = 1e-2

    args.lr_decay1 = 0.1
    args.lr_decay2 = 1.0

    args.warmup_epochs = 0

    # Official scheduler warm-up
    args.scheduler_warmup_epochs = 1

    # --------------------------------------------------------
    # UPA loss weights
    # --------------------------------------------------------

    args.par_cls = 0.3
    args.par_ent = 1.0

    args.par_noisy_cls = 0.3
    args.par_noisy_ent = 1.0
    args.par_su_cl = 1.0

    # --------------------------------------------------------
    # CACL
    # --------------------------------------------------------

    args.su_cl_t = 0.1

    # --------------------------------------------------------
    # APS
    # --------------------------------------------------------

    args.k_val = 4
    args.sel_ratio = 0.6
    args.knn_times = 2

    args.distance = "cosine"
    args.cos_t = 5

    args.balance_class = True
    args.sel_cls = True

    # --------------------------------------------------------
    # Augmentation
    # --------------------------------------------------------

    args.aug = "mocov2"

    # --------------------------------------------------------
    # Numerical
    # --------------------------------------------------------

    args.epsilon = 1e-5

    # --------------------------------------------------------
    # Saving / logging
    # --------------------------------------------------------

    args.issave = True

    args.name = (
        f"PotatoHard_seed{seed}"
    )

    args.savename = (
        f"official_UPA_seed_{seed}"
    )

    args.seed = seed

    args.out_file = open(
        log_path,
        "w"
    )

    return args, seed_output, log_path


# ============================================================
# 5. STANDARDIZED OUTPUT CHECK
# ============================================================

def final_target_paths(seed):

    seed_dir = (
        TARGET_OUTPUT /
        f"seed_{seed}"
    )

    return {
        "F":
            seed_dir /
            "target_F.pt",

        "B":
            seed_dir /
            "target_B.pt",

        "C":
            seed_dir /
            "target_C.pt",

        "summary":
            seed_dir /
            "adaptation_summary.json"
    }


# ============================================================
# 6. RUN ALL THREE SEEDS
# ============================================================

adaptation_summaries = []

for seed in SEEDS:

    print("\n" + "=" * 80)
    print(f"UPA ADAPTATION — SEED {seed}")
    print("=" * 80)

    standardized = (
        final_target_paths(seed)
    )

    # --------------------------------------------------------
    # Resume-safe completed-seed skip
    # --------------------------------------------------------

    if all(
        p.exists()
        for p in standardized.values()
    ):

        with open(
            standardized["summary"],
            "r"
        ) as f:

            existing_summary = (
                json.load(f)
            )

        adaptation_summaries.append(
            existing_summary
        )

        print(
            "Completed adaptation checkpoint already exists."
        )

        continue

    # --------------------------------------------------------
    # Clear only incomplete adaptation products
    # --------------------------------------------------------

    seed_output = (
        TARGET_OUTPUT /
        f"seed_{seed}"
    )

    for key in [
        "F",
        "B",
        "C",
        "summary"
    ]:

        p = standardized[key]

        if p.exists():
            p.unlink()

    # Remove old official acc_* save folders
    for p in seed_output.glob(
        "acc_*"
    ):

        if p.is_dir():
            shutil.rmtree(
                p
            )

    # --------------------------------------------------------
    # Seed exactly as official implementation
    # --------------------------------------------------------

    torch.manual_seed(
        seed
    )

    torch.cuda.manual_seed_all(
        seed
    )

    np.random.seed(
        seed
    )

    random.seed(
        seed
    )

    # Official script enables cudnn benchmark.
    torch.backends.cudnn.benchmark = True

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    args, seed_output, log_path = (
        build_upa_args(
            seed
        )
    )

    start_time = time.time()

    try:

        print(
            "Building official UPA object..."
        )

        upa = Upa(
            args
        )

        # ----------------------------------------------------
        # Critical pre-training assertions
        # ----------------------------------------------------

        # Classifier must be source classifier before adaptation.
        source_C_path = (
            SOURCE_OUTPUT /
            f"seed_{seed}" /
            "source_C.pt"
        )

        if not source_C_path.exists():
            raise RuntimeError(
                f"Missing source classifier: "
                f"{source_C_path}"
            )

        print(
            "Target images:",
            expected_target_n[seed]
        )

        print(
            "True target labels available "
            "to adaptation: NO"
        )

        print(
            "Epochs: 15 | "
            "Batch: 64 | "
            "k=4 | "
            "ratio=0.6 | "
            "CACL T=0.1"
        )

        # ----------------------------------------------------
        # OFFICIAL UPA TRAINING
        # ----------------------------------------------------

        _official_return = (
            upa.start_train()
        )

    except RuntimeError as e:

        args.out_file.close()

        if (
            "out of memory"
            in str(e).lower()
        ):

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            raise RuntimeError(
                f"CUDA OOM during official UPA "
                f"adaptation Seed {seed}. "
                f"Stopped without changing "
                f"batch size 64."
            ) from e

        raise

    finally:

        try:
            args.out_file.flush()
            args.out_file.close()
        except Exception:
            pass

    elapsed_minutes = (
        time.time()
        - start_time
    ) / 60.0

    # --------------------------------------------------------
    # Locate the exact official saved checkpoint directory
    # --------------------------------------------------------

    official_save_dirs = sorted(
        [
            p
            for p in seed_output.glob(
                "acc_*"
            )
            if p.is_dir()
        ],
        key=lambda p:
            p.stat().st_mtime
    )

    if not official_save_dirs:

        raise RuntimeError(
            f"Seed {seed}: official UPA finished "
            f"but no saved target checkpoint "
            f"directory was found."
        )

    official_save_dir = (
        official_save_dirs[-1]
    )

    official_F = (
        official_save_dir /
        "target_F.pt"
    )

    official_B = (
        official_save_dir /
        "target_B.pt"
    )

    official_C = (
        official_save_dir /
        "target_C.pt"
    )

    for p in [
        official_F,
        official_B,
        official_C
    ]:

        if not p.exists():

            raise RuntimeError(
                f"Missing official saved "
                f"checkpoint: {p}"
            )

    # --------------------------------------------------------
    # Copy to stable standardized paths
    # --------------------------------------------------------

    shutil.copy2(
        official_F,
        standardized["F"]
    )

    shutil.copy2(
        official_B,
        standardized["B"]
    )

    shutil.copy2(
        official_C,
        standardized["C"]
    )

    # --------------------------------------------------------
    # Verify classifier stayed frozen
    # --------------------------------------------------------

    src_state = torch.load(
        SOURCE_OUTPUT /
        f"seed_{seed}" /
        "source_C.pt",
        map_location="cpu",
        weights_only=True
    )

    tgt_state = torch.load(
        standardized["C"],
        map_location="cpu",
        weights_only=True
    )

    if (
        set(src_state.keys())
        != set(tgt_state.keys())
    ):

        raise RuntimeError(
            f"Seed {seed}: classifier state "
            f"keys changed unexpectedly."
        )

    classifier_identical = all(
        torch.equal(
            src_state[k],
            tgt_state[k]
        )
        for k in src_state.keys()
    )

    if not classifier_identical:

        raise RuntimeError(
            f"Seed {seed}: frozen classifier "
            f"changed during adaptation."
        )

    # --------------------------------------------------------
    # Save seed summary
    # --------------------------------------------------------

    summary = {
        "method":
            "UPA (official-code reproduction)",

        "seed":
            seed,

        "target_train_n":
            expected_target_n[seed],

        "target_true_labels_available":
            False,

        "source_images_used":
            False,

        "epochs":
            15,

        "batch_size":
            64,

        "base_lr":
            1e-2,

        "k_val":
            4,

        "sel_ratio":
            0.6,

        "knn_times":
            2,

        "su_cl_t":
            0.1,

        "par_noisy_cls":
            0.3,

        "par_noisy_ent":
            1.0,

        "par_su_cl":
            1.0,

        "classifier_frozen_verified":
            True,

        "elapsed_minutes":
            float(
                elapsed_minutes
            ),

        "official_save_dir":
            str(
                official_save_dir
            ),

        "target_F":
            str(
                standardized["F"]
            ),

        "target_B":
            str(
                standardized["B"]
            ),

        "target_C":
            str(
                standardized["C"]
            )
    }

    with open(
        standardized["summary"],
        "w"
    ) as f:

        json.dump(
            summary,
            f,
            indent=2
        )

    adaptation_summaries.append(
        summary
    )

    pd.DataFrame(
        adaptation_summaries
    ).to_csv(
        METRIC_OUTPUT /
        "upa_adaptation_run_summary.csv",
        index=False
    )

    print("\nSeed adaptation completed:")
    print(
        f" Runtime: "
        f"{elapsed_minutes:.2f} min"
    )

    print(
        " Frozen classifier verified: YES"
    )

    print(
        " Saved:",
        standardized["F"]
    )

    # --------------------------------------------------------
    # Cleanup
    # --------------------------------------------------------

    del upa
    del src_state
    del tgt_state

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ============================================================
# 7. FINAL THREE-SEED AUDIT
# ============================================================

print("\n" + "=" * 80)
print("FINAL UPA ADAPTATION CHECKPOINT AUDIT")
print("=" * 80)

final_rows = []

for seed in SEEDS:

    paths = final_target_paths(
        seed
    )

    for name, path in paths.items():

        if not path.exists():

            raise RuntimeError(
                f"Seed {seed}: "
                f"missing {name}: "
                f"{path}"
            )

    with open(
        paths["summary"],
        "r"
    ) as f:

        s = json.load(
            f
        )

    final_rows.append(
        s
    )

    print(
        f"Seed {seed:<4} | "
        f"target_n={s['target_train_n']:<4} | "
        f"epochs={s['epochs']} | "
        f"labels_blinded=YES | "
        f"classifier_frozen=YES | "
        f"time={s['elapsed_minutes']:.2f} min"
    )


final_adapt_df = pd.DataFrame(
    final_rows
)

final_adapt_path = (
    METRIC_OUTPUT /
    "upa_adaptation_run_summary.csv"
)

final_adapt_df.to_csv(
    final_adapt_path,
    index=False
)

print("\n" + "=" * 80)
print("BLOCK 12 COMPLETE")
print("=" * 80)

print(
    "Adaptation summary:",
    final_adapt_path
)

print(
    "\nAll three final official-UPA adapted "
    "checkpoints are ready for evaluation."
)

OFFICIAL UPA TARGET ADAPTATION — ALL THREE SEEDS
Iterator compatibility replacements: 2

Label-blind target lists:
 Seed 42   -> 7200 images
 Seed 123  -> 7200 images
 Seed 2024 -> 7199 images

UPA ADAPTATION — SEED 42
Building official UPA object...
Target images: 7200
True target labels available to adaptation: NO
Epochs: 15 | Batch: 64 | k=4 | ratio=0.6 | CACL T=0.1
Task:PotatoHard_seed42   Accuracy = 70.53% -> 34.96%

starting the 1st knn....
starting the 2st knn....
selection samples accuracy:34.96%
Task: PotatoHard_seed42, epoch:1/15; Accuracy = 33.89 ;Loss = 0.21 

Task:PotatoHard_seed42   Accuracy = 33.89% -> 33.72%

starting the 1st knn....
starting the 2st knn....
selection samples accuracy:33.72%
Task: PotatoHard_seed42, epoch:2/15; Accuracy = 33.26 ;Loss = 1.23 

Task:PotatoHard_seed42   Accuracy = 33.26% -> 33.39%

starting the 1st knn....
starting the 2st knn....
selection samples accuracy:33.40%
Task: PotatoHard_seed42, epoch:3/15; Accuracy = 33.22 ;Loss = 1.38 

Task:Po

In [21]:
# ============================================================
# BLOCK 13 — FINAL OFFICIAL UPA CLOSED-SET EVALUATION
# Untouched hard-target test only
# All three seeds
#
# Metrics:
# Accuracy, Macro-F1, ECE, Brier, NLL
# + confusion matrices
# + sample-level predictions
# + source-only -> adapted deltas
# ============================================================

from pathlib import Path
import gc
import json

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix
)

print("=" * 80)
print("FINAL OFFICIAL UPA EVALUATION — HARD TARGET TEST")
print("=" * 80)


# ------------------------------------------------------------
# 1. Required-function audit
# ------------------------------------------------------------

required_names = [
    "build_upa_source_model",
    "build_eval_loader",
    "calculate_metrics",
    "parse_upa_line",
    "alias_to_original"
]

for name in required_names:
    if name not in globals():
        raise RuntimeError(
            f"Required object '{name}' is missing. "
            f"Block 10 must remain executed in this kernel."
        )


# ------------------------------------------------------------
# 2. Load final adapted checkpoint
# ------------------------------------------------------------

def load_final_upa_target(seed):

    target_dir = (
        TARGET_OUTPUT /
        f"seed_{seed}"
    )

    paths = {
        "F": target_dir / "target_F.pt",
        "B": target_dir / "target_B.pt",
        "C": target_dir / "target_C.pt"
    }

    for path in paths.values():
        if not path.exists():
            raise RuntimeError(
                f"Missing adapted checkpoint: {path}"
            )

    netF, netB, netC = (
        build_upa_source_model()
    )

    netF.load_state_dict(
        torch.load(
            paths["F"],
            map_location=DEVICE,
            weights_only=True
        )
    )

    netB.load_state_dict(
        torch.load(
            paths["B"],
            map_location=DEVICE,
            weights_only=True
        )
    )

    netC.load_state_dict(
        torch.load(
            paths["C"],
            map_location=DEVICE,
            weights_only=True
        )
    )

    netF.eval()
    netB.eval()
    netC.eval()

    return netF, netB, netC


# ------------------------------------------------------------
# 3. Prediction function
# ------------------------------------------------------------

@torch.no_grad()
def predict_final_upa(
    loader,
    netF,
    netB,
    netC
):

    all_logits = []
    all_probs = []
    all_labels = []

    for images, labels in loader:

        images = images.to(
            DEVICE,
            non_blocking=True
        )

        logits = netC(
            netB(
                netF(images)
            )
        )

        probs = F.softmax(
            logits,
            dim=1
        )

        all_logits.append(
            logits.detach().cpu().numpy()
        )

        all_probs.append(
            probs.detach().cpu().numpy()
        )

        all_labels.append(
            labels.numpy()
        )

    return (
        np.concatenate(all_logits),
        np.concatenate(all_probs),
        np.concatenate(all_labels)
    )


# ------------------------------------------------------------
# 4. Evaluate each seed
# ------------------------------------------------------------

rows = []

for seed in SEEDS:

    print("\n" + "=" * 80)
    print(f"SEED {seed}")
    print("=" * 80)

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    netF, netB, netC = (
        load_final_upa_target(seed)
    )

    lines, loader = build_eval_loader(
        seed=seed,
        role="target_hard_test",
        batch_size=128,
        num_workers=4
    )

    logits, probs, labels = (
        predict_final_upa(
            loader,
            netF,
            netB,
            netC
        )
    )

    if len(labels) != 1800:
        raise RuntimeError(
            f"Seed {seed}: expected 1800 hard-target "
            f"test samples, got {len(labels)}."
        )

    if len(lines) != len(labels):
        raise RuntimeError(
            f"Seed {seed}: list/prediction length mismatch."
        )

    preds = probs.argmax(
        axis=1
    )

    metrics = calculate_metrics(
        labels,
        probs
    )

    cm = confusion_matrix(
        labels,
        preds,
        labels=[0, 1, 2]
    )

    # --------------------------------------------------------
    # Sample-level predictions
    # --------------------------------------------------------

    sample_rows = []

    for i, line in enumerate(lines):

        alias_path, stored_label = (
            parse_upa_line(line)
        )

        if int(stored_label) != int(labels[i]):
            raise RuntimeError(
                f"Seed {seed}: true-label alignment "
                f"failure at sample {i}."
            )

        original_path = (
            alias_to_original.get(
                alias_path,
                alias_path
            )
        )

        sample_rows.append({
            "method":
                "UPA_official_adapted",

            "seed":
                seed,

            "sample_index":
                i,

            "original_path":
                original_path,

            "true_label":
                int(labels[i]),

            "pred_label":
                int(preds[i]),

            "correct":
                int(
                    preds[i] == labels[i]
                ),

            "confidence":
                float(
                    probs[i].max()
                ),

            "logit_0":
                float(logits[i, 0]),

            "logit_1":
                float(logits[i, 1]),

            "logit_2":
                float(logits[i, 2]),

            "prob_0":
                float(probs[i, 0]),

            "prob_1":
                float(probs[i, 1]),

            "prob_2":
                float(probs[i, 2])
        })

    pred_path = (
        PRED_OUTPUT /
        f"upa_adapted_seed_{seed}_target_hard_test_predictions.csv"
    )

    pd.DataFrame(
        sample_rows
    ).to_csv(
        pred_path,
        index=False
    )

    # --------------------------------------------------------
    # Confusion matrix
    # --------------------------------------------------------

    cm_path = (
        METRIC_OUTPUT /
        f"upa_adapted_seed_{seed}_target_hard_confusion.csv"
    )

    pd.DataFrame(
        cm,
        index=[
            "true_early",
            "true_late",
            "true_healthy"
        ],
        columns=[
            "pred_early",
            "pred_late",
            "pred_healthy"
        ]
    ).to_csv(
        cm_path
    )

    rows.append({
        "method":
            "UPA_official_adapted",

        "seed":
            seed,

        "dataset":
            "target_hard_test",

        "n":
            int(len(labels)),

        **metrics
    })

    print(
        f"Accuracy : {metrics['accuracy']:.6f}"
    )

    print(
        f"Macro-F1 : {metrics['macro_f1']:.6f}"
    )

    print(
        f"ECE      : {metrics['ece']:.6f}"
    )

    print(
        f"Brier    : {metrics['brier']:.6f}"
    )

    print(
        f"NLL      : {metrics['nll']:.6f}"
    )

    print("\nConfusion matrix:")
    print(cm)

    del netF
    del netB
    del netC
    del loader
    del logits
    del probs
    del labels
    del preds

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ------------------------------------------------------------
# 5. Save adapted seed metrics
# ------------------------------------------------------------

adapted_df = pd.DataFrame(
    rows
)

adapted_seed_path = (
    METRIC_OUTPUT /
    "upa_adapted_hard_target_seed_metrics.csv"
)

adapted_df.to_csv(
    adapted_seed_path,
    index=False
)


# ------------------------------------------------------------
# 6. Three-seed aggregate
# ------------------------------------------------------------

metrics_to_report = [
    "accuracy",
    "macro_f1",
    "ece",
    "brier",
    "nll"
]

aggregate = {
    "method":
        "UPA_official_adapted",

    "dataset":
        "target_hard_test",

    "num_seeds":
        len(adapted_df)
}

for metric in metrics_to_report:

    values = adapted_df[
        metric
    ].to_numpy(
        dtype=float
    )

    aggregate[
        f"{metric}_mean"
    ] = float(
        values.mean()
    )

    aggregate[
        f"{metric}_std"
    ] = float(
        values.std(
            ddof=1
        )
    )

aggregate_df = pd.DataFrame(
    [aggregate]
)

aggregate_path = (
    METRIC_OUTPUT /
    "upa_adapted_hard_target_aggregate.csv"
)

aggregate_df.to_csv(
    aggregate_path,
    index=False
)


# ------------------------------------------------------------
# 7. Source-only -> adapted paired comparison
# ------------------------------------------------------------

source_path = (
    METRIC_OUTPUT /
    "upa_source_only_seed_metrics.csv"
)

if not source_path.exists():
    raise RuntimeError(
        f"Missing Block 10 metrics: {source_path}"
    )

source_df = pd.read_csv(
    source_path
)

source_hard = source_df[
    source_df["dataset"]
    == "target_hard_test"
].copy()

comparison = source_hard[
    [
        "seed",
        "accuracy",
        "macro_f1",
        "ece",
        "brier",
        "nll"
    ]
].merge(
    adapted_df[
        [
            "seed",
            "accuracy",
            "macro_f1",
            "ece",
            "brier",
            "nll"
        ]
    ],
    on="seed",
    suffixes=(
        "_source",
        "_adapted"
    )
)

comparison[
    "accuracy_delta"
] = (
    comparison["accuracy_adapted"]
    - comparison["accuracy_source"]
)

comparison[
    "macro_f1_delta"
] = (
    comparison["macro_f1_adapted"]
    - comparison["macro_f1_source"]
)

# Lower is better for reliability metrics
comparison[
    "ece_reduction"
] = (
    comparison["ece_source"]
    - comparison["ece_adapted"]
)

comparison[
    "brier_reduction"
] = (
    comparison["brier_source"]
    - comparison["brier_adapted"]
)

comparison[
    "nll_reduction"
] = (
    comparison["nll_source"]
    - comparison["nll_adapted"]
)

comparison_path = (
    METRIC_OUTPUT /
    "upa_source_vs_adapted_paired.csv"
)

comparison.to_csv(
    comparison_path,
    index=False
)


# ------------------------------------------------------------
# 8. Print final result
# ------------------------------------------------------------

a = aggregate_df.iloc[0]

print("\n" + "=" * 80)
print("OFFICIAL UPA — HARD TARGET, THREE-SEED AGGREGATE")
print("=" * 80)

print(
    f"Accuracy : "
    f"{a['accuracy_mean']:.6f} "
    f"± {a['accuracy_std']:.6f}"
)

print(
    f"Macro-F1 : "
    f"{a['macro_f1_mean']:.6f} "
    f"± {a['macro_f1_std']:.6f}"
)

print(
    f"ECE      : "
    f"{a['ece_mean']:.6f} "
    f"± {a['ece_std']:.6f}"
)

print(
    f"Brier    : "
    f"{a['brier_mean']:.6f} "
    f"± {a['brier_std']:.6f}"
)

print(
    f"NLL      : "
    f"{a['nll_mean']:.6f} "
    f"± {a['nll_std']:.6f}"
)


print("\n" + "=" * 80)
print("SOURCE-ONLY -> UPA ADAPTATION, PER SEED")
print("=" * 80)

for _, r in comparison.iterrows():

    print(
        f"Seed {int(r['seed']):<4} | "
        f"Acc "
        f"{r['accuracy_source']:.4f}"
        f" -> "
        f"{r['accuracy_adapted']:.4f} "
        f"({r['accuracy_delta']:+.4f}) | "
        f"Macro-F1 "
        f"{r['macro_f1_source']:.4f}"
        f" -> "
        f"{r['macro_f1_adapted']:.4f} "
        f"({r['macro_f1_delta']:+.4f})"
    )


print("\n" + "=" * 80)
print("BLOCK 13 COMPLETE")
print("=" * 80)

print(
    "Seed metrics :",
    adapted_seed_path
)

print(
    "Aggregate    :",
    aggregate_path
)

print(
    "Comparison   :",
    comparison_path
)

FINAL OFFICIAL UPA EVALUATION — HARD TARGET TEST

SEED 42
Accuracy : 0.988889
Macro-F1 : 0.988872
ECE      : 0.005794
Brier    : 0.019421
NLL      : 0.055615

Confusion matrix:
[[588   7   5]
 [  7 592   1]
 [  0   0 600]]

SEED 123
Accuracy : 0.981111
Macro-F1 : 0.981030
ECE      : 0.016138
Brier    : 0.035785
NLL      : 0.122094

Confusion matrix:
[[570  21   9]
 [  2 597   1]
 [  0   1 599]]

SEED 2024
Accuracy : 0.987778
Macro-F1 : 0.987755
ECE      : 0.004137
Brier    : 0.018048
NLL      : 0.039451

Confusion matrix:
[[594   5   1]
 [  8 585   7]
 [  0   1 599]]

OFFICIAL UPA — HARD TARGET, THREE-SEED AGGREGATE
Accuracy : 0.985926 ± 0.004207
Macro-F1 : 0.985885 ± 0.004242
ECE      : 0.008690 ± 0.006503
Brier    : 0.024418 ± 0.009868
NLL      : 0.072386 ± 0.043800

SOURCE-ONLY -> UPA ADAPTATION, PER SEED
Seed 42   | Acc 0.5072 -> 0.9889 (+0.4817) | Macro-F1 0.4245 -> 0.9889 (+0.5644)
Seed 123  | Acc 0.6972 -> 0.9811 (+0.2839) | Macro-F1 0.6876 -> 0.9810 (+0.2935)
Seed 2024 | Acc 0.

In [22]:
# ============================================================
# BLOCK 14 — FIXED THREE-WAY GATE ON OFFICIAL UPA
# Hard-target test, all three seeds
#
# SAME gate used for our method:
#   Accept : confidence >= 0.9999
#   Defer  : 0.6667 <= confidence < 0.9999
#   Reject : confidence < 0.6667
#
# No threshold tuning is performed here.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

print("=" * 80)
print("OFFICIAL UPA — FIXED THREE-WAY GATE")
print("=" * 80)

ALPHA = 0.9999
BETA = 0.6667

print(f"Accept threshold α = {ALPHA}")
print(f"Reject threshold β = {BETA}")

rows = []

for seed in SEEDS:

    pred_path = (
        PRED_OUTPUT /
        f"upa_adapted_seed_{seed}_target_hard_test_predictions.csv"
    )

    if not pred_path.exists():
        raise RuntimeError(
            f"Missing Block 13 predictions: {pred_path}"
        )

    df = pd.read_csv(pred_path)

    required_cols = [
        "true_label",
        "pred_label",
        "confidence"
    ]

    missing = [
        c for c in required_cols
        if c not in df.columns
    ]

    if missing:
        raise RuntimeError(
            f"Seed {seed}: missing columns {missing}"
        )

    if len(df) != 1800:
        raise RuntimeError(
            f"Seed {seed}: expected 1800 rows, "
            f"found {len(df)}"
        )

    conf = df["confidence"].to_numpy(
        dtype=float
    )

    y_true = df["true_label"].to_numpy(
        dtype=int
    )

    y_pred = df["pred_label"].to_numpy(
        dtype=int
    )

    # --------------------------------------------------------
    # Fixed three-way partition
    # --------------------------------------------------------

    region = np.full(
        len(df),
        "Defer",
        dtype=object
    )

    region[
        conf >= ALPHA
    ] = "Accept"

    region[
        conf < BETA
    ] = "Reject"

    df["three_way_region"] = region

    region_pred_path = (
        PRED_OUTPUT /
        f"upa_seed_{seed}_fixed_gate_regions.csv"
    )

    df.to_csv(
        region_pred_path,
        index=False
    )

    print("\n" + "-" * 80)
    print(f"SEED {seed}")
    print("-" * 80)

    for region_name in [
        "Accept",
        "Defer",
        "Reject"
    ]:

        mask = (
            region == region_name
        )

        n = int(
            mask.sum()
        )

        coverage = (
            n / len(df)
        )

        if n > 0:

            acc = float(
                np.mean(
                    y_pred[mask]
                    == y_true[mask]
                )
            )

        else:

            acc = np.nan

        rows.append({
            "method":
                "UPA_official_adapted",

            "seed":
                seed,

            "region":
                region_name,

            "n":
                n,

            "coverage":
                coverage,

            "accuracy":
                acc,

            "alpha":
                ALPHA,

            "beta":
                BETA
        })

        print(
            f"{region_name:<7} | "
            f"n={n:<4} | "
            f"coverage={coverage:.6f} | "
            f"accuracy="
            f"{acc:.6f}"
            if not np.isnan(acc)
            else
            f"{region_name:<7} | "
            f"n={n:<4} | "
            f"coverage={coverage:.6f} | "
            f"accuracy=NA"
        )


# ============================================================
# Aggregate across seeds
# ============================================================

region_df = pd.DataFrame(
    rows
)

seed_path = (
    METRIC_OUTPUT /
    "upa_fixed_gate_region_seed_metrics.csv"
)

region_df.to_csv(
    seed_path,
    index=False
)

aggregate_rows = []

for region_name in [
    "Accept",
    "Defer",
    "Reject"
]:

    sub = region_df[
        region_df["region"]
        == region_name
    ]

    aggregate_rows.append({

        "method":
            "UPA_official_adapted",

        "region":
            region_name,

        "num_seeds":
            len(sub),

        "count_mean":
            float(
                sub["n"].mean()
            ),

        "count_std":
            float(
                sub["n"].std(
                    ddof=1
                )
            ),

        "coverage_mean":
            float(
                sub["coverage"].mean()
            ),

        "coverage_std":
            float(
                sub["coverage"].std(
                    ddof=1
                )
            ),

        "accuracy_mean":
            float(
                sub["accuracy"].mean()
            ),

        "accuracy_std":
            float(
                sub["accuracy"].std(
                    ddof=1
                )
            )
    })


aggregate_df = pd.DataFrame(
    aggregate_rows
)

aggregate_path = (
    METRIC_OUTPUT /
    "upa_fixed_gate_region_aggregate.csv"
)

aggregate_df.to_csv(
    aggregate_path,
    index=False
)


# ============================================================
# High-confidence wrong Accepts
# ============================================================

accept_error_rows = []

for seed in SEEDS:

    pred_path = (
        PRED_OUTPUT /
        f"upa_seed_{seed}_fixed_gate_regions.csv"
    )

    df = pd.read_csv(
        pred_path
    )

    errors = df[
        (df["three_way_region"] == "Accept")
        &
        (df["true_label"] != df["pred_label"])
    ].copy()

    errors["seed"] = seed

    accept_error_rows.append(
        errors
    )

if accept_error_rows:

    all_accept_errors = pd.concat(
        accept_error_rows,
        ignore_index=True
    )

else:

    all_accept_errors = pd.DataFrame()

accept_error_path = (
    METRIC_OUTPUT /
    "upa_fixed_gate_false_accept_known_errors.csv"
)

all_accept_errors.to_csv(
    accept_error_path,
    index=False
)


# ============================================================
# Print three-seed aggregate
# ============================================================

print("\n" + "=" * 80)
print("OFFICIAL UPA — FIXED GATE THREE-SEED AGGREGATE")
print("=" * 80)

for _, r in aggregate_df.iterrows():

    print(
        f"{r['region']:<7} | "
        f"count="
        f"{r['count_mean']:.2f} "
        f"± {r['count_std']:.2f} | "
        f"coverage="
        f"{100*r['coverage_mean']:.2f}% "
        f"± {100*r['coverage_std']:.2f}% | "
        f"accuracy="
        f"{100*r['accuracy_mean']:.2f}% "
        f"± {100*r['accuracy_std']:.2f}%"
    )


print("\n" + "=" * 80)
print("BLOCK 14 COMPLETE")
print("=" * 80)

print(
    "Seed metrics :",
    seed_path
)

print(
    "Aggregate    :",
    aggregate_path
)

print(
    "Accept errors:",
    accept_error_path
)

OFFICIAL UPA — FIXED THREE-WAY GATE
Accept threshold α = 0.9999
Reject threshold β = 0.6667

--------------------------------------------------------------------------------
SEED 42
--------------------------------------------------------------------------------
Accept  | n=38   | coverage=0.021111 | accuracy=1.000000
Defer   | n=1756 | coverage=0.975556 | accuracy=0.990319
Reject  | n=6    | coverage=0.003333 | accuracy=0.500000

--------------------------------------------------------------------------------
SEED 123
--------------------------------------------------------------------------------
Accept  | n=79   | coverage=0.043889 | accuracy=0.987342
Defer   | n=1716 | coverage=0.953333 | accuracy=0.982517
Reject  | n=5    | coverage=0.002778 | accuracy=0.400000

--------------------------------------------------------------------------------
SEED 2024
--------------------------------------------------------------------------------
Accept  | n=74   | coverage=0.041111 | accuracy=1.

In [23]:
# ============================================================
# BLOCK 15 — OFFICIAL UPA OOD FALSE-ACCEPT EVALUATION
# Same 2,000 OOD images
# Same fixed gate:
#   Accept : confidence >= 0.9999
#   Defer  : 0.6667 <= confidence < 0.9999
#   Reject : confidence < 0.6667
#
# All three seeds.
# No threshold tuning.
# ============================================================

from pathlib import Path
import gc
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import Dataset, DataLoader

print("=" * 80)
print("OFFICIAL UPA — OOD FALSE-ACCEPT EVALUATION")
print("=" * 80)

ALPHA = 0.9999
BETA = 0.6667

# ------------------------------------------------------------
# 1. Locate the SAME OOD manifest used in the final package
# ------------------------------------------------------------

registry_path = (
    CONFIG_OUTPUT /
    "locked_split_registry.csv"
)

registry = pd.read_csv(
    registry_path
)

# First try registry role.
ood_candidates = registry[
    registry["role"].astype(str).str.lower().str.contains(
        "ood",
        na=False
    )
].copy()

# If OOD was not part of the locked split registry,
# inspect the already-scanned final-package inventory.
if len(ood_candidates) == 0:

    scan_path = (
        CONFIG_OUTPUT /
        "split_manifest_scan.csv"
    )

    if not scan_path.exists():
        raise RuntimeError(
            "Cannot locate OOD manifest registry."
        )

    scan_df = pd.read_csv(
        scan_path
    )

    path_col = None

    for c in scan_df.columns:
        if "path" in c.lower():
            path_col = c
            break

    if path_col is None:
        raise RuntimeError(
            "No path column found in split_manifest_scan.csv"
        )

    tmp = scan_df[
        scan_df[path_col]
        .astype(str)
        .str.lower()
        .str.contains("ood", na=False)
    ]

    if len(tmp) == 0:
        raise RuntimeError(
            "OOD manifest was not found automatically. "
            "Stop here rather than using a different OOD set."
        )

    print("\nOOD-related package entries:")
    print(
        tmp[[path_col]]
        .drop_duplicates()
        .to_string(index=False)
    )


# ------------------------------------------------------------
# 2. Use existing original experiment OOD predictions/manifests
#    to recover the exact 2,000 image paths.
#
# Search only inside the fixed final package.
# ------------------------------------------------------------

ood_csv_files = [
    p for p in FINAL_PACKAGE.rglob("*.csv")
    if "ood" in p.name.lower()
]

print("\nOOD CSV candidates found:", len(ood_csv_files))

for p in ood_csv_files:
    print(" ", p.relative_to(FINAL_PACKAGE))


def inspect_csv_for_paths(path):

    try:
        df = pd.read_csv(path)
    except Exception:
        return None

    path_columns = [
        c for c in df.columns
        if (
            "path" in c.lower()
            or "file" in c.lower()
            or "image" in c.lower()
        )
    ]

    if not path_columns:
        return None

    return df, path_columns


valid_ood_tables = []

for p in ood_csv_files:

    result = inspect_csv_for_paths(p)

    if result is None:
        continue

    df, path_cols = result

    for col in path_cols:

        values = (
            df[col]
            .dropna()
            .astype(str)
        )

        if len(values) == 2000:

            valid_ood_tables.append(
                (p, df, col)
            )


if not valid_ood_tables:

    raise RuntimeError(
        "Could not identify an exact 2,000-row OOD "
        "image manifest from the final package. "
        "Do not substitute another OOD set."
    )


# Prefer a unique 2,000-row manifest.
# If several files contain the same paths, verify equivalence.
path_sets = []

for p, df, col in valid_ood_tables:

    paths = tuple(
        df[col].astype(str).tolist()
    )

    path_sets.append(
        (p, col, paths)
    )


# ------------------------------------------------------------
# 3. Resolve paths against the existing alias mapping
# ------------------------------------------------------------

alias_map_path = (
    CONFIG_OUTPUT /
    "upa_path_alias_mapping.csv"
)

alias_df = pd.read_csv(
    alias_map_path
)

original_to_alias = dict(
    zip(
        alias_df["original_path"].astype(str),
        alias_df["alias_path"].astype(str)
    )
)


# Existing alias map was created from locked train/test images.
# OOD images may not yet have aliases, so create deterministic
# OOD aliases if needed.
import hashlib
import os

OOD_ALIAS_ROOT = (
    OUTPUT_ROOT /
    "upa_ood_path_aliases"
)

OOD_ALIAS_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


def resolve_existing_path(raw_path):

    p = Path(raw_path)

    if p.exists():
        return str(p)

    # Search by basename only within Kaggle input.
    matches = list(
        KAGGLE_INPUT.rglob(
            p.name
        )
    )

    # Exact suffix matching when possible.
    if len(matches) > 1:

        raw_parts = p.parts

        scored = []

        for m in matches:

            m_parts = m.parts

            score = 0

            for a, b in zip(
                reversed(raw_parts),
                reversed(m_parts)
            ):
                if a.lower() == b.lower():
                    score += 1
                else:
                    break

            scored.append(
                (score, m)
            )

        scored.sort(
            key=lambda x: x[0],
            reverse=True
        )

        if (
            len(scored) >= 2
            and scored[0][0] == scored[1][0]
        ):
            raise RuntimeError(
                f"Ambiguous OOD path: {raw_path}"
            )

        return str(
            scored[0][1]
        )

    if len(matches) == 1:
        return str(matches[0])

    raise RuntimeError(
        f"Could not resolve OOD image: {raw_path}"
    )


# Use first candidate initially.
candidate_path, candidate_df, candidate_col = (
    valid_ood_tables[0]
)

raw_ood_paths = (
    candidate_df[candidate_col]
    .astype(str)
    .tolist()
)

# Verify any additional 2000-row candidates represent
# the same image population when their basenames agree.
print(
    "\nSelected OOD source:",
    candidate_path
)

print(
    "Path column:",
    candidate_col
)

print(
    "Rows:",
    len(raw_ood_paths)
)


resolved_ood_paths = []

for i, raw_path in enumerate(raw_ood_paths):

    if (
        raw_path in original_to_alias
        and Path(
            original_to_alias[raw_path]
        ).exists()
    ):

        resolved = raw_path

    else:

        resolved = resolve_existing_path(
            raw_path
        )

    resolved_ood_paths.append(
        resolved
    )


if len(resolved_ood_paths) != 2000:
    raise RuntimeError(
        "OOD population is not exactly 2,000."
    )

if len(set(resolved_ood_paths)) != 2000:
    raise RuntimeError(
        "Duplicate OOD image paths detected."
    )


# ------------------------------------------------------------
# 4. Verify all OOD images are readable
# ------------------------------------------------------------

bad_ood = []

for path in resolved_ood_paths:

    try:
        with Image.open(path) as img:
            img.verify()
    except Exception as e:

        bad_ood.append(
            (path, repr(e))
        )

if bad_ood:

    print("\nUnreadable OOD images:")
    for x in bad_ood[:20]:
        print(x)

    raise RuntimeError(
        f"{len(bad_ood)} OOD images are unreadable. "
        "Stopping rather than changing the OOD population."
    )

print(
    "\nExact OOD population verified:",
    len(resolved_ood_paths)
)


# ------------------------------------------------------------
# 5. OOD dataset using official UPA evaluation transform
# ------------------------------------------------------------

class OODImageDataset(Dataset):

    def __init__(
        self,
        paths,
        transform
    ):

        self.paths = list(paths)
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(
        self,
        idx
    ):

        path = self.paths[idx]

        with Image.open(path) as img:

            img = img.convert("RGB")

            image = self.transform(
                img
            )

        return image, idx


ood_dataset = OODImageDataset(
    resolved_ood_paths,
    upa_source_eval_transform
)

ood_loader = DataLoader(
    ood_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)


# ------------------------------------------------------------
# 6. Evaluate each adapted UPA model
# ------------------------------------------------------------

seed_rows = []

for seed in SEEDS:

    print("\n" + "-" * 80)
    print(f"SEED {seed}")
    print("-" * 80)

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    netF, netB, netC = (
        load_final_upa_target(
            seed
        )
    )

    all_probs = []
    all_logits = []
    all_indices = []

    with torch.no_grad():

        for images, indices in ood_loader:

            images = images.to(
                DEVICE,
                non_blocking=True
            )

            logits = netC(
                netB(
                    netF(images)
                )
            )

            probs = F.softmax(
                logits,
                dim=1
            )

            all_logits.append(
                logits.cpu().numpy()
            )

            all_probs.append(
                probs.cpu().numpy()
            )

            all_indices.append(
                indices.numpy()
            )

    logits = np.concatenate(
        all_logits
    )

    probs = np.concatenate(
        all_probs
    )

    indices = np.concatenate(
        all_indices
    )

    if len(probs) != 2000:
        raise RuntimeError(
            f"Seed {seed}: expected 2000 OOD "
            f"predictions, got {len(probs)}."
        )

    confidence = probs.max(
        axis=1
    )

    pred = probs.argmax(
        axis=1
    )

    region = np.full(
        2000,
        "Defer",
        dtype=object
    )

    region[
        confidence >= ALPHA
    ] = "Accept"

    region[
        confidence < BETA
    ] = "Reject"

    accept_n = int(
        np.sum(
            region == "Accept"
        )
    )

    defer_n = int(
        np.sum(
            region == "Defer"
        )
    )

    reject_n = int(
        np.sum(
            region == "Reject"
        )
    )

    false_accept_rate = (
        accept_n / 2000.0
    )

    seed_rows.append({
        "method":
            "UPA_official_adapted",

        "seed":
            seed,

        "ood_n":
            2000,

        "accept_n":
            accept_n,

        "defer_n":
            defer_n,

        "reject_n":
            reject_n,

        "false_accept_rate":
            false_accept_rate,

        "alpha":
            ALPHA,

        "beta":
            BETA
    })

    # Sample-level OOD predictions
    ood_pred_df = pd.DataFrame({
        "method":
            "UPA_official_adapted",

        "seed":
            seed,

        "sample_index":
            indices,

        "original_path":
            [
                resolved_ood_paths[i]
                for i in indices
            ],

        "pred_label":
            pred,

        "confidence":
            confidence,

        "region":
            region,

        "logit_0":
            logits[:, 0],

        "logit_1":
            logits[:, 1],

        "logit_2":
            logits[:, 2],

        "prob_0":
            probs[:, 0],

        "prob_1":
            probs[:, 1],

        "prob_2":
            probs[:, 2]
    })

    ood_pred_df.to_csv(
        PRED_OUTPUT /
        f"upa_adapted_seed_{seed}_ood_predictions.csv",
        index=False
    )

    print(
        f"Accept : {accept_n:<4} "
        f"({100*accept_n/2000:.2f}%)"
    )

    print(
        f"Defer  : {defer_n:<4} "
        f"({100*defer_n/2000:.2f}%)"
    )

    print(
        f"Reject : {reject_n:<4} "
        f"({100*reject_n/2000:.2f}%)"
    )

    print(
        f"OOD false-Accept rate: "
        f"{100*false_accept_rate:.2f}%"
    )

    del netF
    del netB
    del netC
    del logits
    del probs

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ------------------------------------------------------------
# 7. Aggregate
# ------------------------------------------------------------

seed_df = pd.DataFrame(
    seed_rows
)

seed_path = (
    METRIC_OUTPUT /
    "upa_ood_fixed_gate_seed_metrics.csv"
)

seed_df.to_csv(
    seed_path,
    index=False
)

fa = seed_df[
    "false_accept_rate"
].to_numpy(
    dtype=float
)

aggregate = {
    "method":
        "UPA_official_adapted",

    "num_seeds":
        3,

    "ood_n_per_seed":
        2000,

    "false_accept_mean":
        float(
            fa.mean()
        ),

    "false_accept_std":
        float(
            fa.std(
                ddof=1
            )
        ),

    "accept_count_mean":
        float(
            seed_df[
                "accept_n"
            ].mean()
        ),

    "accept_count_std":
        float(
            seed_df[
                "accept_n"
            ].std(
                ddof=1
            )
        )
}

aggregate_df = pd.DataFrame(
    [aggregate]
)

aggregate_path = (
    METRIC_OUTPUT /
    "upa_ood_fixed_gate_aggregate.csv"
)

aggregate_df.to_csv(
    aggregate_path,
    index=False
)


# ------------------------------------------------------------
# 8. Print final OOD result
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("OFFICIAL UPA — OOD FIXED-GATE THREE-SEED AGGREGATE")
print("=" * 80)

print(
    f"False-Accept rate : "
    f"{100*aggregate['false_accept_mean']:.2f}% "
    f"± "
    f"{100*aggregate['false_accept_std']:.2f}%"
)

print(
    f"Accept count      : "
    f"{aggregate['accept_count_mean']:.2f} "
    f"± "
    f"{aggregate['accept_count_std']:.2f} "
    f"of 2000"
)

print("\n" + "=" * 80)
print("BLOCK 15 COMPLETE")
print("=" * 80)

print(
    "Seed metrics:",
    seed_path
)

print(
    "Aggregate   :",
    aggregate_path
)

OFFICIAL UPA — OOD FALSE-ACCEPT EVALUATION

OOD-related package entries:
                                                                                                                                                                                                                                       path
                                                                                                      /kaggle/input/datasets/dmmehedihasanabid/twd-kaggle/TWD_paper_final_package_FIXED_20260905_234741/raw_result_files/ood_extra_test.csv
                                                                                              /kaggle/input/datasets/dmmehedihasanabid/twd-kaggle/TWD_paper_final_package_FIXED_20260905_234741/raw_result_files/ood_pld_extra_manifest.csv
                               /kaggle/input/datasets/dmmehedihasanabid/twd-kaggle/TWD_paper_final_package_FIXED_20260905_234741/raw_result_files/results_multiseed_post_twd/post_twd_20260905_231915/all_seed_ood_class_re

In [25]:
# ============================================================
# BLOCK 16 — COVERAGE-MATCHED OOD COMPARISON
# FIXED numeric parsing for region counts
# ============================================================

from pathlib import Path
import re
import numpy as np
import pandas as pd

print("=" * 80)
print("COVERAGE-MATCHED OOD COMPARISON")
print("=" * 80)


# ------------------------------------------------------------
# Helper: robust integer conversion
# Handles 13, 13.0, "13", "13.0"
# ------------------------------------------------------------

def safe_int(value):
    return int(round(float(value)))


# ------------------------------------------------------------
# 1. Recover our proposed method's exact Accept counts
# ------------------------------------------------------------

candidate_csvs = [
    p for p in FINAL_PACKAGE.rglob("*.csv")
    if any(
        key in str(p).lower()
        for key in [
            "post_twd",
            "threeway",
            "three_way",
            "class_aware",
            "classaware",
            "region"
        ]
    )
]

print(
    "Candidate proposed-method CSV files:",
    len(candidate_csvs)
)


def normalize_columns(df):
    return {
        c.lower().strip(): c
        for c in df.columns
    }


def extract_seed_from_text(text):

    s = str(text).lower()

    for seed in SEEDS:

        patterns = [
            rf"seed[_\-\s]*{seed}",
            rf"\b{seed}\b"
        ]

        for pat in patterns:
            if re.search(pat, s):
                return seed

    return None


proposed_accept_counts = {}


# ------------------------------------------------------------
# 2. Search likely candidate files first
# ------------------------------------------------------------

for path in candidate_csvs:

    try:
        df = pd.read_csv(path)
    except Exception:
        continue

    if len(df) == 0:
        continue

    cols = normalize_columns(df)

    region_col = None

    for name in [
        "region",
        "three_way_region",
        "decision_region",
        "twd_region"
    ]:
        if name in cols:
            region_col = cols[name]
            break

    if region_col is None:
        continue

    accept_mask = (
        df[region_col]
        .astype(str)
        .str.lower()
        .str.strip()
        == "accept"
    )

    if not accept_mask.any():
        continue


    # --------------------------------------------------------
    # A. Sample-level file
    # --------------------------------------------------------

    if len(df) >= 1000:

        seed_col = cols.get("seed")

        if seed_col is not None:

            for seed in SEEDS:

                sub = df[
                    df[seed_col]
                    .astype(str)
                    == str(seed)
                ]

                if len(sub) == 1800:

                    count = int(
                        (
                            sub[region_col]
                            .astype(str)
                            .str.lower()
                            .str.strip()
                            == "accept"
                        ).sum()
                    )

                    proposed_accept_counts[
                        seed
                    ] = count

        else:

            inferred_seed = extract_seed_from_text(
                path
            )

            if (
                inferred_seed in SEEDS
                and len(df) == 1800
            ):

                proposed_accept_counts[
                    inferred_seed
                ] = int(
                    accept_mask.sum()
                )


    # --------------------------------------------------------
    # B. Summary file
    # --------------------------------------------------------

    else:

        seed_col = cols.get("seed")

        count_col = None

        for candidate in [
            "n",
            "count",
            "region_count",
            "num_samples"
        ]:

            if candidate in cols:
                count_col = cols[candidate]
                break

        if count_col is None:
            continue

        if seed_col is not None:

            for seed in SEEDS:

                temp = df[
                    (
                        df[seed_col]
                        .astype(str)
                        == str(seed)
                    )
                    &
                    accept_mask
                ]

                if len(temp) == 1:

                    proposed_accept_counts[
                        seed
                    ] = safe_int(
                        temp.iloc[0][count_col]
                    )

        else:

            inferred_seed = extract_seed_from_text(
                path
            )

            temp = df[
                accept_mask
            ]

            if (
                inferred_seed in SEEDS
                and len(temp) == 1
            ):

                proposed_accept_counts[
                    inferred_seed
                ] = safe_int(
                    temp.iloc[0][count_col]
                )


# ------------------------------------------------------------
# 3. Fallback search across all package CSVs
# ------------------------------------------------------------

if set(
    proposed_accept_counts.keys()
) != set(SEEDS):

    for path in FINAL_PACKAGE.rglob("*.csv"):

        if set(
            proposed_accept_counts.keys()
        ) == set(SEEDS):
            break

        try:
            df = pd.read_csv(path)
        except Exception:
            continue

        if len(df) == 0:
            continue

        cols = normalize_columns(df)

        region_col = None

        for name in [
            "region",
            "three_way_region",
            "decision_region",
            "twd_region"
        ]:
            if name in cols:
                region_col = cols[name]
                break

        if region_col is None:
            continue

        accept_mask = (
            df[region_col]
            .astype(str)
            .str.lower()
            .str.strip()
            == "accept"
        )

        if not accept_mask.any():
            continue

        seed_col = cols.get("seed")

        # sample-level fallback
        if len(df) >= 1000:

            if seed_col is not None:

                for seed in SEEDS:

                    if seed in proposed_accept_counts:
                        continue

                    sub = df[
                        df[seed_col]
                        .astype(str)
                        == str(seed)
                    ]

                    if len(sub) == 1800:

                        proposed_accept_counts[
                            seed
                        ] = int(
                            (
                                sub[region_col]
                                .astype(str)
                                .str.lower()
                                .str.strip()
                                == "accept"
                            ).sum()
                        )

            else:

                seed = extract_seed_from_text(
                    path
                )

                if (
                    seed in SEEDS
                    and seed not in proposed_accept_counts
                    and len(df) == 1800
                ):

                    proposed_accept_counts[
                        seed
                    ] = int(
                        accept_mask.sum()
                    )

        # summary-level fallback
        else:

            count_col = None

            for candidate in [
                "n",
                "count",
                "region_count",
                "num_samples"
            ]:
                if candidate in cols:
                    count_col = cols[candidate]
                    break

            if count_col is None:
                continue

            if seed_col is not None:

                for seed in SEEDS:

                    if seed in proposed_accept_counts:
                        continue

                    temp = df[
                        (
                            df[seed_col]
                            .astype(str)
                            == str(seed)
                        )
                        &
                        accept_mask
                    ]

                    if len(temp) == 1:

                        proposed_accept_counts[
                            seed
                        ] = safe_int(
                            temp.iloc[0][count_col]
                        )

            else:

                seed = extract_seed_from_text(
                    path
                )

                temp = df[
                    accept_mask
                ]

                if (
                    seed in SEEDS
                    and seed not in proposed_accept_counts
                    and len(temp) == 1
                ):

                    proposed_accept_counts[
                        seed
                    ] = safe_int(
                        temp.iloc[0][count_col]
                    )


# ------------------------------------------------------------
# 4. Validate proposed Accept counts
# ------------------------------------------------------------

if set(
    proposed_accept_counts.keys()
) != set(SEEDS):

    raise RuntimeError(
        "Could not recover exact per-seed proposed-method "
        "Accept counts from the final package."
    )

print(
    "\nProposed-method known-target Accept counts:"
)

for seed in SEEDS:

    n = proposed_accept_counts[seed]

    print(
        f" Seed {seed:<4}: "
        f"{n}/1800 "
        f"({100*n/1800:.2f}%)"
    )


accept_counts_arr = np.array(
    [
        proposed_accept_counts[s]
        for s in SEEDS
    ],
    dtype=float
)

mean_accept = accept_counts_arr.mean()

print(
    f"\nRecovered mean Accept count: "
    f"{mean_accept:.2f}"
)

print(
    f"Recovered mean coverage    : "
    f"{100*mean_accept/1800:.2f}%"
)

if not (
    250 <= mean_accept <= 370
):

    raise RuntimeError(
        "Recovered Accept counts are inconsistent "
        "with our completed proposed-method results."
    )


# ============================================================
# 5. Coverage-match UPA on known target
# ============================================================

upa_thresholds = {}
known_rows = []

for seed in SEEDS:

    pred_path = (
        PRED_OUTPUT /
        f"upa_adapted_seed_{seed}_target_hard_test_predictions.csv"
    )

    df = pd.read_csv(pred_path)

    if len(df) != 1800:
        raise RuntimeError(
            f"Seed {seed}: expected 1800 known target rows."
        )

    conf = df[
        "confidence"
    ].to_numpy(
        dtype=float
    )

    true = df[
        "true_label"
    ].to_numpy(
        dtype=int
    )

    pred = df[
        "pred_label"
    ].to_numpy(
        dtype=int
    )

    k = proposed_accept_counts[
        seed
    ]

    order = np.argsort(
        -conf,
        kind="mergesort"
    )

    selected_idx = order[:k]

    kth_conf = float(
        conf[
            selected_idx[-1]
        ]
    )

    higher_n = int(
        np.sum(
            conf > kth_conf
        )
    )

    equal_n = int(
        np.sum(
            conf == kth_conf
        )
    )

    boundary_tie = (
        higher_n < k
        and higher_n + equal_n > k
    )

    known_accept_acc = float(
        np.mean(
            pred[selected_idx]
            == true[selected_idx]
        )
    )

    upa_thresholds[
        seed
    ] = kth_conf

    known_rows.append({
        "seed":
            seed,

        "matched_accept_n":
            k,

        "matched_coverage":
            k / 1800.0,

        "upa_threshold":
            kth_conf,

        "known_accept_accuracy":
            known_accept_acc,

        "boundary_tie":
            bool(boundary_tie),

        "boundary_equal_count":
            equal_n
    })

    print(
        f"\nSeed {seed}"
    )

    print(
        f" Matched known Accept : "
        f"{k}/1800 "
        f"({100*k/1800:.2f}%)"
    )

    print(
        f" UPA threshold        : "
        f"{kth_conf:.10f}"
    )

    print(
        f" Known Accept accuracy: "
        f"{100*known_accept_acc:.2f}%"
    )

    print(
        f" Boundary tie         : "
        f"{boundary_tie}"
    )


known_df = pd.DataFrame(
    known_rows
)

known_path = (
    METRIC_OUTPUT /
    "upa_coverage_matched_known_target.csv"
)

known_df.to_csv(
    known_path,
    index=False
)


# ============================================================
# 6. Apply matched thresholds to OOD
# ============================================================

ood_rows = []

for seed in SEEDS:

    ood_path = (
        PRED_OUTPUT /
        f"upa_adapted_seed_{seed}_ood_predictions.csv"
    )

    if not ood_path.exists():

        raise RuntimeError(
            f"Missing Block 15 predictions: {ood_path}"
        )

    ood_df = pd.read_csv(
        ood_path
    )

    if len(ood_df) != 2000:

        raise RuntimeError(
            f"Seed {seed}: expected 2000 OOD rows."
        )

    threshold = upa_thresholds[
        seed
    ]

    ood_conf = ood_df[
        "confidence"
    ].to_numpy(
        dtype=float
    )

    ood_accept = (
        ood_conf >= threshold
    )

    accept_n = int(
        ood_accept.sum()
    )

    false_accept_rate = (
        accept_n / 2000.0
    )

    ood_rows.append({
        "seed":
            seed,

        "matched_known_accept_n":
            proposed_accept_counts[seed],

        "matched_known_coverage":
            proposed_accept_counts[seed] / 1800.0,

        "upa_threshold":
            threshold,

        "ood_accept_n":
            accept_n,

        "ood_false_accept_rate":
            false_accept_rate
    })

    print(
        f"\nSeed {seed} OOD:"
    )

    print(
        f" Accept: {accept_n}/2000 "
        f"({100*false_accept_rate:.2f}%)"
    )


ood_match_df = pd.DataFrame(
    ood_rows
)

ood_match_path = (
    METRIC_OUTPUT /
    "upa_coverage_matched_ood_seed_metrics.csv"
)

ood_match_df.to_csv(
    ood_match_path,
    index=False
)


# ============================================================
# 7. Aggregate
# ============================================================

fa = ood_match_df[
    "ood_false_accept_rate"
].to_numpy(
    dtype=float
)

known_acc = known_df[
    "known_accept_accuracy"
].to_numpy(
    dtype=float
)

aggregate = {
    "method":
        "UPA_official_adapted",

    "matching_reference":
        "ThreeWayClassAware_accept_count_per_seed",

    "num_seeds":
        3,

    "matched_known_coverage_mean":
        float(
            ood_match_df[
                "matched_known_coverage"
            ].mean()
        ),

    "matched_known_coverage_std":
        float(
            ood_match_df[
                "matched_known_coverage"
            ].std(
                ddof=1
            )
        ),

    "known_accept_accuracy_mean":
        float(
            known_acc.mean()
        ),

    "known_accept_accuracy_std":
        float(
            known_acc.std(
                ddof=1
            )
        ),

    "ood_false_accept_mean":
        float(
            fa.mean()
        ),

    "ood_false_accept_std":
        float(
            fa.std(
                ddof=1
            )
        )
}

aggregate_df = pd.DataFrame(
    [aggregate]
)

aggregate_path = (
    METRIC_OUTPUT /
    "upa_coverage_matched_ood_aggregate.csv"
)

aggregate_df.to_csv(
    aggregate_path,
    index=False
)


# ============================================================
# 8. Final print
# ============================================================

print("\n" + "=" * 80)
print("UPA — COVERAGE-MATCHED THREE-SEED RESULT")
print("=" * 80)

print(
    f"Matched known coverage : "
    f"{100*aggregate['matched_known_coverage_mean']:.2f}% "
    f"± "
    f"{100*aggregate['matched_known_coverage_std']:.2f}%"
)

print(
    f"Known Accept accuracy  : "
    f"{100*aggregate['known_accept_accuracy_mean']:.2f}% "
    f"± "
    f"{100*aggregate['known_accept_accuracy_std']:.2f}%"
)

print(
    f"OOD false-Accept rate : "
    f"{100*aggregate['ood_false_accept_mean']:.2f}% "
    f"± "
    f"{100*aggregate['ood_false_accept_std']:.2f}%"
)

print("\nPer-seed:")

for _, r in ood_match_df.iterrows():

    print(
        f"Seed {int(r['seed']):<4} | "
        f"coverage="
        f"{100*r['matched_known_coverage']:.2f}% | "
        f"threshold="
        f"{r['upa_threshold']:.6f} | "
        f"OOD FA="
        f"{100*r['ood_false_accept_rate']:.2f}%"
    )

print("\n" + "=" * 80)
print("BLOCK 16 COMPLETE")
print("=" * 80)

print(
    "Known-target match:",
    known_path
)

print(
    "OOD seed metrics :",
    ood_match_path
)

print(
    "Aggregate        :",
    aggregate_path
)

COVERAGE-MATCHED OOD COMPARISON
Candidate proposed-method CSV files: 70

Proposed-method known-target Accept counts:
 Seed 42  : 307/1800 (17.06%)
 Seed 123 : 4/1800 (0.22%)
 Seed 2024: 13/1800 (0.72%)

Recovered mean Accept count: 108.00
Recovered mean coverage    : 6.00%


RuntimeError: Recovered Accept counts are inconsistent with our completed proposed-method results.

In [26]:
# ============================================================
# BLOCK 16A — RECOVER THE EXACT PROPOSED-METHOD ACCEPT COUNTS
# Strict audit: no heuristic overwrite from unrelated CSVs
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import re

print("=" * 80)
print("RECOVERING EXACT THREEWAY-CLASSAWARE ACCEPT COUNTS")
print("=" * 80)

EXPECTED_MEAN = 307.33
EXPECTED_STD  = 57.07

def norm_cols(df):
    return {str(c).strip().lower(): c for c in df.columns}

def to_int(x):
    return int(round(float(x)))

candidates = []

for path in FINAL_PACKAGE.rglob("*.csv"):

    try:
        df = pd.read_csv(path)
    except Exception:
        continue

    if df.empty:
        continue

    cols = norm_cols(df)

    # Require seed + region information.
    seed_col = cols.get("seed")

    region_col = None
    for key in [
        "region",
        "three_way_region",
        "decision_region",
        "twd_region"
    ]:
        if key in cols:
            region_col = cols[key]
            break

    if seed_col is None or region_col is None:
        continue

    # --------------------------------------------------------
    # Case 1: summary table with an explicit count column
    # --------------------------------------------------------

    count_col = None

    for key in [
        "n",
        "count",
        "region_count",
        "num_samples",
        "samples"
    ]:
        if key in cols:
            count_col = cols[key]
            break

    if count_col is not None:

        counts = {}

        for seed in SEEDS:

            sub = df[
                (
                    pd.to_numeric(
                        df[seed_col],
                        errors="coerce"
                    ) == seed
                )
                &
                (
                    df[region_col]
                    .astype(str)
                    .str.strip()
                    .str.lower()
                    == "accept"
                )
            ]

            if len(sub) == 1:

                try:
                    counts[seed] = to_int(
                        sub.iloc[0][count_col]
                    )
                except Exception:
                    pass

        if set(counts) == set(SEEDS):

            vals = np.array(
                [counts[s] for s in SEEDS],
                dtype=float
            )

            candidates.append({
                "path": str(path),
                "type": "summary",
                "counts": counts,
                "mean": vals.mean(),
                "std": vals.std(ddof=1)
            })


    # --------------------------------------------------------
    # Case 2: sample-level table containing all three seeds
    # --------------------------------------------------------

    counts = {}

    for seed in SEEDS:

        sub = df[
            pd.to_numeric(
                df[seed_col],
                errors="coerce"
            ) == seed
        ]

        # Exact hard-target test should contain 1800 per seed.
        if len(sub) == 1800:

            counts[seed] = int(
                (
                    sub[region_col]
                    .astype(str)
                    .str.strip()
                    .str.lower()
                    == "accept"
                ).sum()
            )

    if set(counts) == set(SEEDS):

        vals = np.array(
            [counts[s] for s in SEEDS],
            dtype=float
        )

        candidates.append({
            "path": str(path),
            "type": "sample_level",
            "counts": counts,
            "mean": vals.mean(),
            "std": vals.std(ddof=1)
        })


# ------------------------------------------------------------
# Rank candidates against the VERIFIED previous result
# ------------------------------------------------------------

if not candidates:
    raise RuntimeError(
        "No three-seed Accept-region table found in final package."
    )

for c in candidates:

    c["score"] = (
        abs(c["mean"] - EXPECTED_MEAN)
        +
        abs(c["std"] - EXPECTED_STD)
    )

candidates = sorted(
    candidates,
    key=lambda x: x["score"]
)

print(f"\nThree-seed candidate tables found: {len(candidates)}")

print("\nBest candidates:")
for c in candidates[:10]:

    print("\nFILE :", c["path"])
    print("TYPE :", c["type"])
    print("COUNT:", c["counts"])
    print(
        f"MEAN : {c['mean']:.4f} | "
        f"STD: {c['std']:.4f} | "
        f"score={c['score']:.4f}"
    )


best = candidates[0]

# Very strict check against our completed result.
if (
    abs(best["mean"] - EXPECTED_MEAN) > 0.10
    or
    abs(best["std"] - EXPECTED_STD) > 0.15
):
    raise RuntimeError(
        "No saved table reproduces the verified "
        "307.33 ± 57.07 Accept result closely enough. "
        "Stopping instead of guessing."
    )

proposed_accept_counts = {
    int(k): int(v)
    for k, v in best["counts"].items()
}

print("\n" + "=" * 80)
print("LOCKED PROPOSED-METHOD ACCEPT COUNTS")
print("=" * 80)

for seed in SEEDS:
    n = proposed_accept_counts[seed]

    print(
        f"Seed {seed:<4} : "
        f"{n}/1800 "
        f"({100*n/1800:.2f}%)"
    )

vals = np.array(
    [proposed_accept_counts[s] for s in SEEDS],
    dtype=float
)

print(
    f"\nMean count    : {vals.mean():.2f} "
    f"± {vals.std(ddof=1):.2f}"
)

print(
    f"Mean coverage : "
    f"{100*vals.mean()/1800:.2f}%"
)

print("\nSOURCE FILE:")
print(best["path"])

print("\nBLOCK 16A COMPLETE")

RECOVERING EXACT THREEWAY-CLASSAWARE ACCEPT COUNTS

Three-seed candidate tables found: 4

Best candidates:

FILE : /kaggle/input/datasets/dmmehedihasanabid/twd-kaggle/TWD_paper_final_package_FIXED_20260905_234741/tables/three_seed_known_target_regions.csv
TYPE : summary
COUNT: {42: 286, 123: 264, 2024: 372}
MEAN : 307.3333 | STD: 57.0731 | score=0.0064

FILE : /kaggle/input/datasets/dmmehedihasanabid/twd-kaggle/TWD_paper_final_package_FIXED_20260905_234741/raw_result_files/results_multiseed_post_twd/post_twd_with_seed42_20260905_232652/all_3_seed_known_target_region_summary.csv
TYPE : summary
COUNT: {42: 286, 123: 264, 2024: 372}
MEAN : 307.3333 | STD: 57.0731 | score=0.0064

FILE : /kaggle/input/datasets/dmmehedihasanabid/twd-kaggle/TWD_paper_final_package_FIXED_20260905_234741/tables/three_seed_ood_regions.csv
TYPE : summary
COUNT: {42: 22, 123: 4, 2024: 13}
MEAN : 13.0000 | STD: 9.0000 | score=342.4000

FILE : /kaggle/input/datasets/dmmehedihasanabid/twd-kaggle/TWD_paper_final_packa

In [27]:
# ============================================================
# BLOCK 16B — FINAL COVERAGE-MATCHED OOD COMPARISON
#
# Uses VERIFIED proposed-method Accept counts:
#   Seed 42   = 286
#   Seed 123  = 264
#   Seed 2024 = 372
#
# For each seed:
#   1. rank UPA known-target samples by confidence
#   2. choose threshold giving same known-target Accept count
#   3. apply that threshold to UPA OOD predictions
#   4. compare against proposed-method OOD false Accept
#
# No OOD data is used to choose UPA threshold.
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

print("=" * 80)
print("FINAL COVERAGE-MATCHED OOD COMPARISON")
print("=" * 80)


# ------------------------------------------------------------
# 1. Verify locked proposed counts from Block 16A
# ------------------------------------------------------------

EXPECTED_COUNTS = {
    42: 286,
    123: 264,
    2024: 372
}

if "proposed_accept_counts" not in globals():
    raise RuntimeError(
        "proposed_accept_counts is missing. "
        "Run Block 16A first."
    )

for seed in SEEDS:

    actual = int(
        proposed_accept_counts[seed]
    )

    expected = EXPECTED_COUNTS[seed]

    if actual != expected:

        raise RuntimeError(
            f"Seed {seed}: expected proposed Accept count "
            f"{expected}, found {actual}."
        )

print("\nVerified proposed known-target Accept counts:")

for seed in SEEDS:

    n = EXPECTED_COUNTS[seed]

    print(
        f"Seed {seed:<4}: "
        f"{n}/1800 "
        f"({100*n/1800:.2f}%)"
    )


# ------------------------------------------------------------
# 2. Coverage-match UPA on known hard-target test
# ------------------------------------------------------------

known_rows = []
upa_thresholds = {}

for seed in SEEDS:

    pred_path = (
        PRED_OUTPUT /
        f"upa_adapted_seed_{seed}_target_hard_test_predictions.csv"
    )

    if not pred_path.exists():

        raise RuntimeError(
            f"Missing UPA prediction file: {pred_path}"
        )

    df = pd.read_csv(
        pred_path
    )

    if len(df) != 1800:

        raise RuntimeError(
            f"Seed {seed}: expected 1800 known-target rows, "
            f"found {len(df)}."
        )

    conf = df[
        "confidence"
    ].to_numpy(
        dtype=float
    )

    y_true = df[
        "true_label"
    ].to_numpy(
        dtype=int
    )

    y_pred = df[
        "pred_label"
    ].to_numpy(
        dtype=int
    )

    k = EXPECTED_COUNTS[
        seed
    ]

    # Highest-confidence samples first
    order = np.argsort(
        -conf,
        kind="mergesort"
    )

    selected_idx = order[:k]

    threshold = float(
        conf[
            selected_idx[-1]
        ]
    )

    # --------------------------------------------------------
    # Audit threshold boundary
    # --------------------------------------------------------

    higher_n = int(
        np.sum(
            conf > threshold
        )
    )

    equal_n = int(
        np.sum(
            conf == threshold
        )
    )

    threshold_accept_n = int(
        np.sum(
            conf >= threshold
        )
    )

    boundary_tie = (
        threshold_accept_n != k
    )

    if boundary_tie:

        raise RuntimeError(
            f"Seed {seed}: confidence tie at matched "
            f"coverage boundary. Exact target count={k}, "
            f"but threshold >= {threshold:.12f} accepts "
            f"{threshold_accept_n}. "
            f"Stopping rather than using an ambiguous threshold."
        )

    known_accept_accuracy = float(
        np.mean(
            y_pred[selected_idx]
            == y_true[selected_idx]
        )
    )

    upa_thresholds[
        seed
    ] = threshold

    known_rows.append({
        "seed":
            seed,

        "matched_accept_n":
            k,

        "matched_coverage":
            k / 1800.0,

        "upa_threshold":
            threshold,

        "known_accept_accuracy":
            known_accept_accuracy,

        "higher_than_threshold_n":
            higher_n,

        "equal_threshold_n":
            equal_n
    })

    print("\n" + "-" * 80)
    print(f"SEED {seed}")
    print("-" * 80)

    print(
        f"Matched UPA Accept : "
        f"{k}/1800 "
        f"({100*k/1800:.2f}%)"
    )

    print(
        f"UPA threshold      : "
        f"{threshold:.10f}"
    )

    print(
        f"UPA Accept accuracy: "
        f"{100*known_accept_accuracy:.2f}%"
    )


known_df = pd.DataFrame(
    known_rows
)

known_path = (
    METRIC_OUTPUT /
    "upa_coverage_matched_known_target.csv"
)

known_df.to_csv(
    known_path,
    index=False
)


# ------------------------------------------------------------
# 3. Apply matched threshold to UPA OOD predictions
# ------------------------------------------------------------

upa_ood_rows = []

for seed in SEEDS:

    ood_path = (
        PRED_OUTPUT /
        f"upa_adapted_seed_{seed}_ood_predictions.csv"
    )

    if not ood_path.exists():

        raise RuntimeError(
            f"Missing UPA OOD predictions: {ood_path}"
        )

    df = pd.read_csv(
        ood_path
    )

    if len(df) != 2000:

        raise RuntimeError(
            f"Seed {seed}: expected 2000 OOD rows, "
            f"found {len(df)}."
        )

    threshold = upa_thresholds[
        seed
    ]

    conf = df[
        "confidence"
    ].to_numpy(
        dtype=float
    )

    accept_mask = (
        conf >= threshold
    )

    accept_n = int(
        accept_mask.sum()
    )

    fa_rate = (
        accept_n / 2000.0
    )

    upa_ood_rows.append({
        "seed":
            seed,

        "matched_known_accept_n":
            EXPECTED_COUNTS[seed],

        "matched_known_coverage":
            EXPECTED_COUNTS[seed] / 1800.0,

        "upa_threshold":
            threshold,

        "upa_ood_accept_n":
            accept_n,

        "upa_ood_false_accept_rate":
            fa_rate
    })

    print(
        f"\nSeed {seed} UPA OOD "
        f"Accept: {accept_n}/2000 "
        f"({100*fa_rate:.2f}%)"
    )


upa_ood_df = pd.DataFrame(
    upa_ood_rows
)


# ------------------------------------------------------------
# 4. Load VERIFIED proposed-method OOD region counts
# ------------------------------------------------------------

proposed_ood_path = (
    FINAL_PACKAGE /
    "tables" /
    "three_seed_ood_regions.csv"
)

if not proposed_ood_path.exists():

    raise RuntimeError(
        f"Missing verified proposed OOD table: "
        f"{proposed_ood_path}"
    )

proposal_ood_df = pd.read_csv(
    proposed_ood_path
)

cols = {
    str(c).strip().lower(): c
    for c in proposal_ood_df.columns
}

if "seed" not in cols or "region" not in cols:

    raise RuntimeError(
        "Proposed OOD table does not contain "
        "seed and region columns."
    )

seed_col = cols["seed"]
region_col = cols["region"]

count_col = None

for key in [
    "n",
    "count",
    "region_count",
    "num_samples",
    "samples"
]:

    if key in cols:
        count_col = cols[key]
        break

if count_col is None:

    raise RuntimeError(
        "Could not identify count column in "
        "proposed OOD table."
    )


proposed_ood_counts = {}

for seed in SEEDS:

    sub = proposal_ood_df[
        (
            pd.to_numeric(
                proposal_ood_df[seed_col],
                errors="coerce"
            ) == seed
        )
        &
        (
            proposal_ood_df[region_col]
            .astype(str)
            .str.strip()
            .str.lower()
            == "accept"
        )
    ]

    if len(sub) != 1:

        raise RuntimeError(
            f"Seed {seed}: could not uniquely recover "
            f"proposed OOD Accept count."
        )

    proposed_ood_counts[
        seed
    ] = int(
        round(
            float(
                sub.iloc[0][
                    count_col
                ]
            )
        )
    )


print("\nVerified proposed-method OOD Accept counts:")

for seed in SEEDS:

    n = proposed_ood_counts[
        seed
    ]

    print(
        f"Seed {seed:<4}: "
        f"{n}/2000 "
        f"({100*n/2000:.2f}%)"
    )


# ------------------------------------------------------------
# 5. Final per-seed comparison
# ------------------------------------------------------------

comparison_rows = []

for seed in SEEDS:

    upa_row = upa_ood_df[
        upa_ood_df["seed"]
        == seed
    ].iloc[0]

    proposed_n = proposed_ood_counts[
        seed
    ]

    proposed_rate = (
        proposed_n / 2000.0
    )

    upa_rate = float(
        upa_row[
            "upa_ood_false_accept_rate"
        ]
    )

    comparison_rows.append({
        "seed":
            seed,

        "known_accept_n":
            EXPECTED_COUNTS[seed],

        "known_coverage":
            EXPECTED_COUNTS[seed] / 1800.0,

        "upa_threshold":
            float(
                upa_row[
                    "upa_threshold"
                ]
            ),

        "upa_known_accept_accuracy":
            float(
                known_df.loc[
                    known_df["seed"] == seed,
                    "known_accept_accuracy"
                ].iloc[0]
            ),

        "upa_ood_accept_n":
            int(
                upa_row[
                    "upa_ood_accept_n"
                ]
            ),

        "upa_ood_false_accept_rate":
            upa_rate,

        "proposed_ood_accept_n":
            proposed_n,

        "proposed_ood_false_accept_rate":
            proposed_rate,

        "upa_minus_proposed_ood_fa":
            upa_rate - proposed_rate
    })


comparison_df = pd.DataFrame(
    comparison_rows
)

comparison_path = (
    METRIC_OUTPUT /
    "coverage_matched_upa_vs_proposed_ood.csv"
)

comparison_df.to_csv(
    comparison_path,
    index=False
)


# ------------------------------------------------------------
# 6. Aggregate
# ------------------------------------------------------------

upa_fa = comparison_df[
    "upa_ood_false_accept_rate"
].to_numpy(
    dtype=float
)

proposed_fa = comparison_df[
    "proposed_ood_false_accept_rate"
].to_numpy(
    dtype=float
)

known_cov = comparison_df[
    "known_coverage"
].to_numpy(
    dtype=float
)

upa_accept_acc = comparison_df[
    "upa_known_accept_accuracy"
].to_numpy(
    dtype=float
)

difference = (
    upa_fa - proposed_fa
)

aggregate = {
    "known_coverage_mean":
        float(
            known_cov.mean()
        ),

    "known_coverage_std":
        float(
            known_cov.std(
                ddof=1
            )
        ),

    "upa_known_accept_accuracy_mean":
        float(
            upa_accept_acc.mean()
        ),

    "upa_known_accept_accuracy_std":
        float(
            upa_accept_acc.std(
                ddof=1
            )
        ),

    "upa_ood_false_accept_mean":
        float(
            upa_fa.mean()
        ),

    "upa_ood_false_accept_std":
        float(
            upa_fa.std(
                ddof=1
            )
        ),

    "proposed_ood_false_accept_mean":
        float(
            proposed_fa.mean()
        ),

    "proposed_ood_false_accept_std":
        float(
            proposed_fa.std(
                ddof=1
            )
        ),

    "upa_minus_proposed_mean":
        float(
            difference.mean()
        ),

    "upa_minus_proposed_std":
        float(
            difference.std(
                ddof=1
            )
        )
}

aggregate_df = pd.DataFrame(
    [aggregate]
)

aggregate_path = (
    METRIC_OUTPUT /
    "coverage_matched_upa_vs_proposed_ood_aggregate.csv"
)

aggregate_df.to_csv(
    aggregate_path,
    index=False
)


# ------------------------------------------------------------
# 7. Final output
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("COVERAGE-MATCHED UPA VS PROPOSED — THREE-SEED RESULT")
print("=" * 80)

print(
    f"Known-target coverage : "
    f"{100*aggregate['known_coverage_mean']:.2f}% "
    f"± "
    f"{100*aggregate['known_coverage_std']:.2f}%"
)

print(
    f"UPA Accept accuracy   : "
    f"{100*aggregate['upa_known_accept_accuracy_mean']:.2f}% "
    f"± "
    f"{100*aggregate['upa_known_accept_accuracy_std']:.2f}%"
)

print(
    f"UPA OOD false-Accept  : "
    f"{100*aggregate['upa_ood_false_accept_mean']:.2f}% "
    f"± "
    f"{100*aggregate['upa_ood_false_accept_std']:.2f}%"
)

print(
    f"Our OOD false-Accept  : "
    f"{100*aggregate['proposed_ood_false_accept_mean']:.2f}% "
    f"± "
    f"{100*aggregate['proposed_ood_false_accept_std']:.2f}%"
)

print(
    f"UPA - Ours difference : "
    f"{100*aggregate['upa_minus_proposed_mean']:+.2f} "
    f"± "
    f"{100*aggregate['upa_minus_proposed_std']:.2f} pp"
)

print("\nPer seed:")

for _, r in comparison_df.iterrows():

    print(
        f"Seed {int(r['seed']):<4} | "
        f"coverage="
        f"{100*r['known_coverage']:.2f}% | "
        f"UPA threshold="
        f"{r['upa_threshold']:.6f} | "
        f"UPA OOD FA="
        f"{100*r['upa_ood_false_accept_rate']:.2f}% | "
        f"Ours="
        f"{100*r['proposed_ood_false_accept_rate']:.2f}%"
    )


print("\n" + "=" * 80)
print("BLOCK 16B COMPLETE")
print("=" * 80)

print(
    "Per-seed comparison:",
    comparison_path
)

print(
    "Aggregate          :",
    aggregate_path
)

FINAL COVERAGE-MATCHED OOD COMPARISON

Verified proposed known-target Accept counts:
Seed 42  : 286/1800 (15.89%)
Seed 123 : 264/1800 (14.67%)
Seed 2024: 372/1800 (20.67%)

--------------------------------------------------------------------------------
SEED 42
--------------------------------------------------------------------------------
Matched UPA Accept : 286/1800 (15.89%)
UPA threshold      : 0.9997116923
UPA Accept accuracy: 100.00%

--------------------------------------------------------------------------------
SEED 123
--------------------------------------------------------------------------------
Matched UPA Accept : 264/1800 (14.67%)
UPA threshold      : 0.9997239709
UPA Accept accuracy: 98.86%

--------------------------------------------------------------------------------
SEED 2024
--------------------------------------------------------------------------------
Matched UPA Accept : 372/1800 (20.67%)
UPA threshold      : 0.9995096922
UPA Accept accuracy: 100.00%

Seed 4

In [28]:
# ============================================================
# FINAL SAVE — PACKAGE COMPLETE OFFICIAL UPA EXPERIMENT
# Run BEFORE ending the Kaggle session
# ============================================================

from pathlib import Path
import shutil
import hashlib
import json
import os
from datetime import datetime

ROOT = Path("/kaggle/working/official_upa_reproduction")

if not ROOT.exists():
    raise RuntimeError(f"Experiment directory missing: {ROOT}")

# ------------------------------------------------------------
# 1. Critical-file audit
# ------------------------------------------------------------

critical_files = [
    ROOT / "metrics" / "upa_source_training_all_seeds.csv",
    ROOT / "metrics" / "upa_source_only_seed_metrics.csv",
    ROOT / "metrics" / "upa_adapted_hard_target_seed_metrics.csv",
    ROOT / "metrics" / "upa_adapted_hard_target_aggregate.csv",
    ROOT / "metrics" / "upa_fixed_gate_region_seed_metrics.csv",
    ROOT / "metrics" / "upa_ood_fixed_gate_seed_metrics.csv",
    ROOT / "metrics" / "coverage_matched_upa_vs_proposed_ood.csv",
    ROOT / "metrics" / "coverage_matched_upa_vs_proposed_ood_aggregate.csv",
]

missing = [str(p) for p in critical_files if not p.exists()]

if missing:
    print("MISSING CRITICAL FILES:")
    for p in missing:
        print(" -", p)
    raise RuntimeError(
        "Critical result files are missing. "
        "Do NOT leave Kaggle yet."
    )

# Check all three adapted checkpoints
for seed in [42, 123, 2024]:
    for name in ["target_F.pt", "target_B.pt", "target_C.pt"]:
        p = ROOT / "target_models" / f"seed_{seed}" / name

        # tolerate different TARGET_OUTPUT folder name by searching
        if not p.exists():
            matches = list(ROOT.rglob(f"seed_{seed}/{name}"))
            if not matches:
                raise RuntimeError(
                    f"Missing adapted checkpoint: seed={seed}, {name}"
                )

print("Critical result audit: PASSED")


# ------------------------------------------------------------
# 2. Write final experiment README
# ------------------------------------------------------------

readme = ROOT / "FINAL_EXPERIMENT_SUMMARY.txt"

readme.write_text(
"""OFFICIAL UPA REPRODUCTION — FINAL SAVED EXPERIMENT
====================================================

Seeds:
42, 123, 2024

Official UPA repository commit:
3827ef989fa88e91430160b9c8f3187e1a702b65

Setting:
Official-code UPA reproduction adapted to the exact locked
potato source/target splits.

Target adaptation:
15 epochs
Batch size 64
Base LR 0.01
Pseudo-label CE weight 0.3
IM weight 1.0
CACL weight 1.0
CACL temperature 0.1
APS k 4
Selection ratio 0.6
KNN refinement rounds 2
Classifier frozen
Source-free target adaptation

Important protocol:
Target-training labels were replaced with dummy labels during
UPA adaptation. Therefore internal UPA training 'accuracy'
messages are diagnostic artifacts and are NOT evaluation results.

Hard-target test labels were used only after adaptation for
final evaluation.

Seed 2024 target training:
7199 images because the previously documented unreadable
earlyblt838.jpg was excluded without replacement.
Target test remained unchanged at 1800 images.

FINAL UPA CLOSED-SET RESULT
Accuracy : 0.985926 ± 0.004207
Macro-F1 : 0.985885 ± 0.004242
ECE      : 0.008690 ± 0.006503
Brier    : 0.024418 ± 0.009868
NLL      : 0.072386 ± 0.043800

FIXED GATE alpha=0.9999
Known-target Accept coverage:
3.54% ± 1.24%

Known-target Accept accuracy:
99.58% ± 0.73%

OOD false-Accept:
0.93% ± 0.77%

COVERAGE-MATCHED COMPARISON
Matched known-target coverage:
17.07% ± 3.17%

UPA known-target Accept accuracy:
99.62% ± 0.66%

UPA OOD false-Accept:
4.43% ± 3.64%

ThreeWay-ClassAware OOD false-Accept:
0.65% ± 0.45%

UPA minus proposed:
+3.78 ± 3.91 percentage points

Verified proposed known-target Accept counts:
Seed 42   286 / 1800
Seed 123  264 / 1800
Seed 2024 372 / 1800

Verified proposed OOD Accept counts:
Seed 42   22 / 2000
Seed 123  4 / 2000
Seed 2024 13 / 2000

Interpretation:
UPA achieves substantially stronger closed-set adaptation.
At matched known-target Accept coverage, the proposed
ThreeWay-ClassAware method produces a lower mean OOD
false-Accept rate. With only three seeds, this comparison
must not be described as statistically significant superiority.
""",
    encoding="utf-8"
)


# ------------------------------------------------------------
# 3. Create file inventory with SHA256
# ------------------------------------------------------------

inventory = []

for p in sorted(ROOT.rglob("*")):
    if not p.is_file():
        continue

    # Don't hash archive itself if rerunning.
    if p.suffix.lower() == ".zip":
        continue

    h = hashlib.sha256()

    with open(p, "rb") as f:
        while True:
            chunk = f.read(1024 * 1024)
            if not chunk:
                break
            h.update(chunk)

    inventory.append({
        "relative_path": str(p.relative_to(ROOT)),
        "size_bytes": p.stat().st_size,
        "sha256": h.hexdigest()
    })

import pandas as pd

inventory_path = ROOT / "FINAL_FILE_INVENTORY_SHA256.csv"

pd.DataFrame(inventory).to_csv(
    inventory_path,
    index=False
)

print("Files inventoried:", len(inventory))


# ------------------------------------------------------------
# 4. Save final metadata
# ------------------------------------------------------------

metadata = {
    "saved_at": datetime.now().isoformat(),
    "official_upa_commit":
        "3827ef989fa88e91430160b9c8f3187e1a702b65",
    "seeds": [42, 123, 2024],
    "status": "COMPLETE",
    "block_13_closed_set_evaluation": "COMPLETE",
    "block_14_fixed_gate_known_target": "COMPLETE",
    "block_15_fixed_gate_ood": "COMPLETE",
    "block_16_coverage_matched_ood": "COMPLETE",
    "hard_target_test_n_per_seed": 1800,
    "ood_n": 2000,
    "alpha_fixed": 0.9999,
    "beta_fixed": 0.6667,
    "proposed_accept_counts": {
        "42": 286,
        "123": 264,
        "2024": 372
    },
    "proposed_ood_accept_counts": {
        "42": 22,
        "123": 4,
        "2024": 13
    }
}

with open(
    ROOT / "FINAL_RUN_METADATA.json",
    "w"
) as f:
    json.dump(
        metadata,
        f,
        indent=2
    )


# ------------------------------------------------------------
# 5. Create ZIP outside experiment directory
# ------------------------------------------------------------

archive_base = Path(
    "/kaggle/working/"
    "OFFICIAL_UPA_COMPLETE_3SEED_FINAL_20260914"
)

archive_file = Path(str(archive_base) + ".zip")

if archive_file.exists():
    archive_file.unlink()

shutil.make_archive(
    str(archive_base),
    "zip",
    root_dir=ROOT.parent,
    base_dir=ROOT.name
)

if not archive_file.exists():
    raise RuntimeError("ZIP creation failed.")

size_mb = archive_file.stat().st_size / (1024 ** 2)


# ------------------------------------------------------------
# 6. Final verification
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("FINAL EXPERIMENT SAVE COMPLETE")
print("=" * 80)

print("Experiment directory:")
print(ROOT)

print("\nArchive:")
print(archive_file)

print(f"\nArchive size: {size_mb:.2f} MB")

print("\nREADME:")
print(readme)

print("\nSHA256 inventory:")
print(inventory_path)

print("\nSTATUS: SAFE TO SAVE/COMMIT KAGGLE NOTEBOOK")
print("=" * 80)

Critical result audit: PASSED
Files inventoried: 25330

FINAL EXPERIMENT SAVE COMPLETE
Experiment directory:
/kaggle/working/official_upa_reproduction

Archive:
/kaggle/working/OFFICIAL_UPA_COMPLETE_3SEED_FINAL_20260914.zip

Archive size: 3475.92 MB

README:
/kaggle/working/official_upa_reproduction/FINAL_EXPERIMENT_SUMMARY.txt

SHA256 inventory:
/kaggle/working/official_upa_reproduction/FINAL_FILE_INVENTORY_SHA256.csv

STATUS: SAFE TO SAVE/COMMIT KAGGLE NOTEBOOK


In [29]:
# ============================================================
# FINAL MASTER ZIP
# Includes:
#   1. Entire official_upa_reproduction directory
#   2. ALL .keras files under /kaggle/working
#   3. ALL related CSV / JSON / TXT / PT / PY / IPYNB files
#      that are already inside the official experiment folder
#
# Run this BEFORE leaving Kaggle.
# ============================================================

from pathlib import Path
import shutil
import os
import hashlib
import pandas as pd
from datetime import datetime

WORKING = Path("/kaggle/working")

UPA_ROOT = WORKING / "official_upa_reproduction"

if not UPA_ROOT.exists():
    raise RuntimeError(
        f"Missing official UPA directory: {UPA_ROOT}"
    )

# ------------------------------------------------------------
# 1. Create temporary master package directory
# ------------------------------------------------------------

MASTER_DIR = WORKING / "FINAL_TWD_UPA_MASTER_PACKAGE"

if MASTER_DIR.exists():
    shutil.rmtree(MASTER_DIR)

MASTER_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Master package:")
print(MASTER_DIR)


# ------------------------------------------------------------
# 2. Copy entire official UPA experiment
# ------------------------------------------------------------

UPA_DEST = MASTER_DIR / "official_upa_reproduction"

shutil.copytree(
    UPA_ROOT,
    UPA_DEST
)

print("\nCopied official UPA experiment.")


# ------------------------------------------------------------
# 3. Find and copy ALL Keras models in /kaggle/working
# ------------------------------------------------------------

KERAS_DEST = MASTER_DIR / "keras_models"

KERAS_DEST.mkdir(
    parents=True,
    exist_ok=True
)

keras_files = sorted(
    WORKING.rglob("*.keras")
)

# Avoid recursively copying files already inside MASTER_DIR
keras_files = [
    p for p in keras_files
    if MASTER_DIR not in p.parents
]

print(
    f"\nKeras files found: {len(keras_files)}"
)

keras_registry = []

for i, src in enumerate(keras_files):

    relative = src.relative_to(
        WORKING
    )

    # Preserve directory structure to avoid filename collisions
    dst = (
        KERAS_DEST /
        relative
    )

    dst.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    shutil.copy2(
        src,
        dst
    )

    keras_registry.append({
        "source_path":
            str(src),

        "saved_relative_path":
            str(
                dst.relative_to(
                    MASTER_DIR
                )
            ),

        "size_mb":
            src.stat().st_size /
            (1024 ** 2)
    })

    print(
        f"[{i+1}/{len(keras_files)}] "
        f"{relative}"
    )


pd.DataFrame(
    keras_registry
).to_csv(
    MASTER_DIR /
    "KERAS_MODEL_REGISTRY.csv",
    index=False
)


# ------------------------------------------------------------
# 4. Also preserve important standalone result directories
#    from older experiments if they exist
# ------------------------------------------------------------

important_dirs = [

    "external_sfuda_baseline",

    "results_multiseed_summary",

    "results_multiseed_post_twd",

    "results_ood_source_vs_proposed",

    "results_threeway_class_aware",

    "logit_model_ResNet50",

    "results_threeway_softpl",

    "results_threeway_source_anchor",

    "results_accept_only"
]

EXTRA_DEST = (
    MASTER_DIR /
    "additional_working_results"
)

EXTRA_DEST.mkdir(
    parents=True,
    exist_ok=True
)

copied_extra = []

for dirname in important_dirs:

    src = (
        WORKING /
        dirname
    )

    if not src.exists():
        continue

    # Don't duplicate official directory
    if src == UPA_ROOT:
        continue

    dst = (
        EXTRA_DEST /
        dirname
    )

    print(
        f"\nCopying additional directory: {src}"
    )

    shutil.copytree(
        src,
        dst,
        dirs_exist_ok=True
    )

    copied_extra.append(
        str(src)
    )


# ------------------------------------------------------------
# 5. Copy standalone useful files from /kaggle/working
#
# Includes notebooks/configs/results that may not be in dirs.
# Excludes giant archives and our package itself.
# ------------------------------------------------------------

STANDALONE_DEST = (
    MASTER_DIR /
    "standalone_working_files"
)

STANDALONE_DEST.mkdir(
    parents=True,
    exist_ok=True
)

extensions_to_preserve = {
    ".csv",
    ".json",
    ".txt",
    ".log",
    ".py",
    ".ipynb",
    ".npy",
    ".npz",
    ".pkl",
    ".yaml",
    ".yml"
}

standalone_registry = []

for src in WORKING.iterdir():

    if not src.is_file():
        continue

    if src.suffix.lower() not in extensions_to_preserve:
        continue

    if (
        "FINAL_TWD_UPA_MASTER_PACKAGE"
        in src.name
    ):
        continue

    dst = (
        STANDALONE_DEST /
        src.name
    )

    shutil.copy2(
        src,
        dst
    )

    standalone_registry.append({
        "source_path":
            str(src),

        "saved_relative_path":
            str(
                dst.relative_to(
                    MASTER_DIR
                )
            ),

        "size_mb":
            src.stat().st_size /
            (1024 ** 2)
    })


pd.DataFrame(
    standalone_registry
).to_csv(
    MASTER_DIR /
    "STANDALONE_FILE_REGISTRY.csv",
    index=False
)


# ------------------------------------------------------------
# 6. Create complete file inventory + SHA256
# ------------------------------------------------------------

inventory = []

all_files = [
    p for p in MASTER_DIR.rglob("*")
    if p.is_file()
]

print(
    f"\nBuilding SHA256 inventory for "
    f"{len(all_files)} files..."
)

for idx, p in enumerate(
    sorted(all_files)
):

    h = hashlib.sha256()

    with open(
        p,
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                1024 * 1024
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    inventory.append({

        "relative_path":
            str(
                p.relative_to(
                    MASTER_DIR
                )
            ),

        "size_bytes":
            p.stat().st_size,

        "size_mb":
            p.stat().st_size /
            (1024 ** 2),

        "sha256":
            h.hexdigest()
    })


inventory_df = pd.DataFrame(
    inventory
)

inventory_df.to_csv(
    MASTER_DIR /
    "MASTER_FILE_INVENTORY_SHA256.csv",
    index=False
)


# ------------------------------------------------------------
# 7. Write package README
# ------------------------------------------------------------

readme = MASTER_DIR / "README_MASTER_PACKAGE.txt"

readme.write_text(
f"""
FINAL TWD + OFFICIAL UPA MASTER PACKAGE
=======================================

Created:
{datetime.now().isoformat()}

This package preserves:

1. Complete official UPA reproduction
   - three seeds: 42, 123, 2024
   - source checkpoints
   - adapted checkpoints
   - predictions
   - metrics
   - configs
   - split manifests
   - logs
   - coverage-matched OOD results

2. All .keras model files found under /kaggle/working

3. Important earlier result directories when available

4. Standalone CSV / JSON / TXT / LOG / PY / IPYNB /
   NPY / NPZ / PKL / YAML files found at /kaggle/working

Official UPA pinned commit:
3827ef989fa88e91430160b9c8f3187e1a702b65

Important final UPA result:
Accuracy = 0.985926 ± 0.004207
Macro-F1 = 0.985885 ± 0.004242
ECE = 0.008690 ± 0.006503
Brier = 0.024418 ± 0.009868
NLL = 0.072386 ± 0.043800

Coverage-matched OOD comparison:
Known-target coverage = 17.07% ± 3.17%
UPA OOD false-Accept = 4.43% ± 3.64%
ThreeWay-ClassAware OOD false-Accept = 0.65% ± 0.45%

Verified proposed Accept counts:
Seed 42   = 286
Seed 123  = 264
Seed 2024 = 372

Verified proposed OOD Accept counts:
Seed 42   = 22
Seed 123  = 4
Seed 2024 = 13
""".strip(),
    encoding="utf-8"
)


# ------------------------------------------------------------
# 8. Create final ZIP
# ------------------------------------------------------------

ZIP_BASE = (
    WORKING /
    "FINAL_TWD_UPA_MASTER_PACKAGE_20260914"
)

ZIP_PATH = Path(
    str(ZIP_BASE) + ".zip"
)

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

print("\nCreating ZIP...")

shutil.make_archive(
    str(ZIP_BASE),
    "zip",
    root_dir=MASTER_DIR.parent,
    base_dir=MASTER_DIR.name
)


# ------------------------------------------------------------
# 9. Verify ZIP
# ------------------------------------------------------------

if not ZIP_PATH.exists():
    raise RuntimeError(
        "ZIP creation failed."
    )

zip_size_gb = (
    ZIP_PATH.stat().st_size /
    (1024 ** 3)
)

zip_size_mb = (
    ZIP_PATH.stat().st_size /
    (1024 ** 2)
)

print("\n" + "=" * 80)
print("MASTER ZIP COMPLETE")
print("=" * 80)

print(
    "ZIP:",
    ZIP_PATH
)

print(
    f"ZIP size: "
    f"{zip_size_mb:.2f} MB "
    f"({zip_size_gb:.2f} GB)"
)

print(
    "Keras models copied:",
    len(keras_files)
)

print(
    "Additional result dirs copied:",
    len(copied_extra)
)

print(
    "Files in package:",
    len(inventory_df)
)

print(
    "\nSTATUS: MASTER PACKAGE READY"
)
print("=" * 80)

Master package:
/kaggle/working/FINAL_TWD_UPA_MASTER_PACKAGE

Copied official UPA experiment.

Keras files found: 0

Building SHA256 inventory for 25334 files...

Creating ZIP...

MASTER ZIP COMPLETE
ZIP: /kaggle/working/FINAL_TWD_UPA_MASTER_PACKAGE_20260914.zip
ZIP size: 3478.95 MB (3.40 GB)
Keras models copied: 0
Additional result dirs copied: 0
Files in package: 25334

STATUS: MASTER PACKAGE READY
